# RQ 12.1 — From seasonal to extreme flooding in South Sudan, and early warning from upstream

**This one notebook contains the complete analysis.** Every line of code is in the cells below; no other
code files are needed.

**Sub-questions answered**
- **SQ1.** Which hydro-climatic factors are associated with the shift from normal seasonal flooding to
  extreme or unusual floods, and can upstream water-level anomalies give an early indication of these changes?
- **SQ3.** How can flood hazard, exposure and vulnerability be combined at ADM3 (payam) level to identify areas
  to prioritise for anticipatory humanitarian action?

**How to run**
1. Put this notebook inside the course `raw_data` folder, in a sub-folder (e.g. `raw_data/extreme_flood_early_warning/`).
2. Install the packages once (cell below, set `INSTALL = True`), then restart the kernel.
3. `Kernel → Restart Kernel and Run All Cells`.

Every step writes its results to `data/` and `outputs/` next to this notebook and is **skipped next time if its
results already exist** (set `FORCE_RECOMPUTE = True` to redo everything). A run from scratch takes about
**2 hours**, mostly the GRU model (step 3) and the first remote read of the Landsat maps (step 6).

| Step | What it does |
|---|---|
| 1 | Download the open external data (lake levels, Indian Ocean index, flood-event records, poster font) |
| 2 | Build the daily/weekly dataset from the raw data |
| 2b | **Data cleaning and EDA**: inventory, cleaning decisions with evidence, coverage, flood cycle, threshold, correlations, data quality |
| 3 | GRU model and benchmarks (leave-one-year-out and forward tests) |
| 4 | Physically constrained model: is the limit the model or the data? |
| 5 | Robustness: confidence intervals, surrogate test, spatial control, target sensitivity |
| 6 | Longer record: Landsat 1984–2021, lakes 1992–2026, Indian Ocean → lakes → Sudd |
| 7 | Documented flood events vs lake levels |
| 8 | One model per question: odds ratios (Q1), early-warning skill (Q2) |
| 9 | Animation and SQ1 poster figure |
| 10 | SQ3: ADM3 priority index and map |
| 11–12 | Answers (numbers read from the results) and a check that every output exists |

## 0a · Install packages (once)

In [ ]:
INSTALL = False   # set True the first time, run this cell, then restart the kernel
# Tested with Python 3.12. PyTorch 2.2.2 is the last release with Intel-Mac wheels and needs numpy < 2,
# which is why numpy and pyarrow are pinned.
if INSTALL:
    %pip install "numpy==1.26.4" "pandas==2.3.3" "xarray==2026.9.0" "netCDF4==1.7.5.1.post1" "dask==2026.8.0" "rasterio==1.4.4" "geopandas==1.1.4" "shapely==2.2.0" "pyarrow==18.1.0" "scipy==1.17.1" "scikit-learn==1.9.1" "torch==2.2.2" "matplotlib==3.11.2" "joblib==1.6.0" "openpyxl==3.1.5" "jupyterlab==4.6.4" "nbclient==0.11.0" "nbconvert==7.17.2" "ipykernel==7.4.0"

## 0b · Settings, input checks and helpers

In [ ]:
import os
os.environ.setdefault("OMP_NUM_THREADS", "1")   # torch and sklearn ship different OpenMP runtimes: avoid a crash

FORCE_RECOMPUTE = False   # True: recompute every step even if its outputs exist
MAKE_VIDEO = True         # the animation needs ffmpeg; set False to skip it

import shutil, sys, time, types
from pathlib import Path
import pandas as pd
from IPython.core.magic import register_cell_magic
from IPython.display import Image, Markdown, display

# the folder this notebook is in (remembered, because this cell changes the working folder below);
# data/ and outputs/ are written here
PROJ = Path(os.environ.setdefault("RQ121_NOTEBOOK_DIR", str(Path.cwd().resolve())))
RAW = PROJ.parent                    # the course raw_data folder
TAB, FIG, POSTER = PROJ / "outputs" / "tables", PROJ / "outputs" / "figures", PROJ / "outputs" / "poster"
os.chdir(RAW)                        # the code reads the raw data with paths relative to raw_data
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)
print("notebook folder:", PROJ); print("raw data folder:", RAW); print("python:", sys.executable)

needed = ["flood_masks/compact_recurring", "flood_masks/compact_unusual", "rainfall and runoff/ERA5_2020.nc",
          "evapotranspiration/ET_2020", "Water levels lakes/water_level_altimetry_Albert.nc",
          "Darthmouth Flood Observatory/100205_discharge.csv", "Administrative boundaries/ssd_admin0.geojson",
          "Administrative boundaries/sdn_admin0.geojson", "farmland/geonode__cattle_gha.tif",
          # made earlier by the team (payam geometries/population and county IPC table), used in SQ3:
          "flood_driver_analysis/data/payam_flood_predictions.geojson",
          "flood_driver_analysis/data/payam_flood_predictions_annual_summary.csv",
          "QA_results/county_ipc_flood_analysis.csv"]
missing = [p for p in needed if not (RAW / p).exists()]
assert not missing, f"missing input data in {RAW}: {missing}"
HAS_FFMPEG = shutil.which("ffmpeg") is not None
print("all raw inputs found ✓ | ffmpeg:", "found" if HAS_FFMPEG else "NOT found (the video will be skipped)")


@register_cell_magic
def module(line, cell):
    """Run a cell as its own module (e.g. `%%module train_models`), so later cells can `import train_models`
    and names such as main() or REGIONS in different steps do not overwrite each other."""
    name = line.strip()
    mod = types.ModuleType(name)
    mod.__file__ = str(PROJ / f"{name}.py")    # its outputs go to PROJ/data and PROJ/outputs
    sys.modules[name] = mod
    exec(compile(cell, f"<{name}>", "exec"), mod.__dict__)
    get_ipython().user_ns[name] = mod


def run_step(func, outputs):
    """Run one step unless all its outputs already exist (FORCE_RECOMPUTE = True to redo)."""
    outs = [PROJ / o for o in outputs]
    if not FORCE_RECOMPUTE and outs and all(o.exists() for o in outs):
        print(f"skip {func.__module__}.{func.__name__}: outputs already exist")
        return
    t0 = time.time()
    func()
    missing = [str(o) for o in outs if not o.exists()]
    assert not missing, f"step finished but did not write: {missing}"
    print(f"done in {(time.time() - t0) / 60:.1f} min")


def show(*names, width=900):
    for n in names:
        display(Image(filename=str(n), width=width))

## 1 · Open external data

Lake Victoria & Kyoga levels (NASA G-REALM), Indian Ocean Dipole and ENSO (NOAA), documented flood events (GLIDE, Dartmouth Flood Observatory via HDX) and the poster font. No registration needed.

<details><summary>About this code</summary>

```
Download the open external data used by the extended analysis (no registration needed).
Already-downloaded files are skipped. Everything goes to raw_data/external/.

  grealm/   Lake Victoria & Kyoga levels 1992-today, NASA GSFC G-REALM 10-day altimetry (~325 MB bundle;
            only the two lake files are kept)
  noaa/     Indian Ocean Dipole (DMI, HadISST) and Nino 3.4 monthly indices, NOAA PSL
  events/   GLIDE disaster records (South Sudan, Sudan) and the Dartmouth Flood Observatory master list (via HDX)
  fonts/    Source Sans 3 (Adobe, SIL Open Font License), the poster font used by the poster figures

The JRC Global Surface Water maps are not downloaded here: external_data.py reads only the South Sudan window
directly from the JRC server.
```
</details>

In [ ]:
%%module download_external
import shutil
import tarfile
import time
import urllib.request
import zipfile
from pathlib import Path

BASE = Path(__file__).resolve().parent.parent
EXT = BASE / "external"

FILES = {
    "noaa/dmi.had.long.data": "https://psl.noaa.gov/gcos_wgsp/Timeseries/Data/dmi.had.long.data",
    "noaa/nina34.anom.data": "https://psl.noaa.gov/data/correlation/nina34.anom.data",
    "events/ssd_glide_events.csv": "https://data.humdata.org/dataset/35c1f93c-64f2-4533-8efa-d28ed9c7753d/resource/"
                                   "74c4e896-3ddb-40c5-81b1-f1ed721fb8cc/download/ssd_glide_events.csv",
    "events/sdn_glide_events.csv": "https://data.humdata.org/dataset/4277194b-d1a7-4ab2-8fa6-ae31699cf507/resource/"
                                   "8e6bfc42-4530-46e0-8fb7-08e3362bdbeb/download/sdn_glide_events.csv",
    "events/dfo_masterlist_20190418.zip": "https://data.humdata.org/dataset/1fd855de-57c6-42b3-83e1-9cf989b0f70d/"
                                          "resource/984cc240-b2b7-4266-9f61-5715a9e10ff5/download/"
                                          "wlf_nhr_fl_dfomasterlist_20190418.zip",
}
GREALM_URL = "https://earth.gsfc.nasa.gov/gwm/zip/lake.10.tar.gz"
GREALM_KEEP = ["lake000314.10d.2.txt", "lake000398.10d.2.txt"]   # Victoria, Kyoga
FONT_URL = "https://github.com/adobe-fonts/source-sans/releases/download/3.052R/OTF-source-sans-3.052R.zip"


def fetch(url, dest, tries=3):
    dest.parent.mkdir(parents=True, exist_ok=True)
    tmp = dest.with_suffix(dest.suffix + ".part")
    for i in range(tries):
        try:
            req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0 (research data download)"})
            with urllib.request.urlopen(req, timeout=600) as r, open(tmp, "wb") as f:
                shutil.copyfileobj(r, f, length=1 << 20)
            tmp.rename(dest)
            return
        except Exception as e:  # network hiccups: retry
            print(f"  retry {i + 1}/{tries} for {url}: {e}")
            time.sleep(5)
    raise RuntimeError(f"could not download {url}")


def main():
    for rel, url in FILES.items():
        dest = EXT / rel
        if dest.exists():
            print("ok      ", rel)
            continue
        print("download", rel)
        fetch(url, dest)
    dfo = EXT / "events" / "dfo"
    if not any(dfo.glob("*.shp")):
        with zipfile.ZipFile(EXT / "events" / "dfo_masterlist_20190418.zip") as z:
            z.extractall(dfo)

    gdir = EXT / "grealm"
    if all((gdir / f).exists() for f in GREALM_KEEP):
        print("ok       grealm lake files")
    else:
        tar = gdir / "lake.10.tar.gz"
        if not tar.exists():
            print("download grealm bundle (~325 MB)")
            fetch(GREALM_URL, tar)
        with tarfile.open(tar) as t:
            for m in t.getmembers():
                if Path(m.name).name in GREALM_KEEP:
                    m.name = Path(m.name).name
                    t.extract(m, gdir)
        tar.unlink()                           # keep only the two lake files

    fdir = EXT / "fonts" / "ss3" / "OTF"
    if any(fdir.glob("SourceSans3-*.otf")):
        print("ok       fonts")
    else:
        z = EXT / "fonts" / "ss3.zip"
        print("download fonts")
        fetch(FONT_URL, z)
        with zipfile.ZipFile(z) as zz:
            zz.extractall(EXT / "fonts" / "ss3")
        z.unlink()
    print("external data ready in", EXT)

In [ ]:
run_step(download_external.main, ["../external/grealm/lake000314.10d.2.txt", "../external/noaa/dmi.had.long.data",
                                 "../external/events/ssd_glide_events.csv", "../external/events/dfo"])

## 2 · Build the dataset (~10 min)

Flood masks are reduced to daily recurring/unusual flood area in South Sudan and the Sudd; hourly ERA5 rain/runoff and daily AgERA5 evaporation to regional means; lake levels and discharge are added.

<details><summary>About this code</summary>

```
Research question 12.1 — Step 1: build the hydro-climatic dataset.

Extracts daily, region-aggregated time series from the raw data folder and saves them to
extreme_flood_early_warning/data/daily_series.parquet:

  Flood regime (MODIS/VIIRS MCDWD 3-day composites, South Sudan only)
    - recurring_px_{region} / unusual_px_{region}: flood pixels detected per day
  Upstream water levels (satellite altimetry)
    - lake_victoria, lake_kyoga, lake_albert: water level [m], on observation days only
  River discharge (Dartmouth Flood Observatory, satellite-derived)
    - q_malakal (station 100205, 9.6N 31.6E, White Nile at the Sobat confluence)
    - q_ethiopia (station 1505, 4.9N 36.0E, south-west Ethiopian highlands)
  ERA5 precipitation / runoff [mm/day], area means over hydrologically meaningful regions
  AgERA5 reference evapotranspiration [mm/day], area means
```
</details>

In [ ]:
%%module build_dataset
import glob
import io
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd
import xarray as xr
from shapely import contains_xy

BASE = Path(__file__).resolve().parent.parent
OUT = Path(__file__).resolve().parent / "data"
OUT.mkdir(exist_ok=True)
YEARS = range(2000, 2026)

# Boxes as (lat_min, lat_max, lon_min, lon_max)
REGIONS = {
    # Equatorial lakes plateau: Lake Victoria/Kyoga/Albert catchment, source of the White Nile
    "lakes": (-3.0, 3.5, 29.0, 35.5),
    # Sudd wetland and the Bahr el Jebel / White Nile floodplain (Jonglei, Unity, Lakes, Upper Nile)
    "sudd": (6.0, 10.0, 29.5, 32.5),
    # Baro-Akobo-Sobat headwaters in the south-west Ethiopian highlands
    "sobat": (5.5, 9.5, 33.5, 36.5),
    # Bahr el Ghazal rain-fed tributaries (western South Sudan)
    "ghazal": (5.0, 10.0, 24.0, 29.5),
}


def south_sudan_geom():
    return gpd.read_file(BASE / "Administrative boundaries" / "ssd_admin0.geojson").geometry.union_all()


def grid_masks(lat, lon, ssd):
    lon2, lat2 = np.meshgrid(lon, lat)
    masks = {"ssd": contains_xy(ssd, lon2, lat2)}
    for name, (a, b, c, d) in REGIONS.items():
        masks[name] = (lat2 >= a) & (lat2 <= b) & (lon2 >= c) & (lon2 <= d)
    return masks


# --------------------------------------------------------------------------------------
# Flood masks
# --------------------------------------------------------------------------------------
def flood_series(ssd):
    sudd = REGIONS["sudd"]
    rows = []
    for typ in ["recurring", "unusual"]:
        for year in YEARS:
            parts = [
                pd.read_parquet(BASE / "flood_masks" / f"compact_{typ}" / f"flood_events_{t}_{year}.parquet",
                                columns=["date", "lat", "lon"])
                for t in ["h20v08", "h21v08"]
            ]
            d = pd.concat(parts, ignore_index=True)
            d = d[contains_xy(ssd, d["lon"].values, d["lat"].values)]
            in_sudd = (d["lat"].between(sudd[0], sudd[1]) & d["lon"].between(sudd[2], sudd[3]))
            g = pd.DataFrame({
                f"{typ}_px_ssd": d.groupby("date").size(),
                f"{typ}_px_sudd": d[in_sudd].groupby("date").size(),
            })
            rows.append(g)
            print(f"  flood masks {typ} {year}: {len(d):,} pixels in South Sudan")
    out = pd.concat(rows).groupby(level=0).sum()
    out.index = pd.to_datetime(out.index)
    return out


# --------------------------------------------------------------------------------------
# Lakes and discharge
# --------------------------------------------------------------------------------------
def lake_series():
    missions = {'TOPEX', 'JASON-1', 'JASON-2', 'JASON-3', 'J1', 'J2', 'J3',
                'S3A', 'S3B', 'S6A', 'ENVISAT', 'ERS-1', 'ERS-2'}
    cols = ['mission', 'cycle', 'date', 'hour', 'minute', 'height_wrt_ref', 'height_err', 'backscatter_ku',
            'wet_tropo_corr', 'iono_corr', 'dry_tropo_corr', 'mode1', 'mode2', 'ice_flag',
            'height_egm2008', 'data_source_flag']
    out = {}
    for name in ["victoria", "Kyoga"]:
        with open(BASE / "Water levels lakes" / f"water_level_{name}.txt", encoding="latin-1") as f:
            lines = [ln.strip() for ln in f if ln.strip() and ln.split()[0] in missions]
        df = pd.read_csv(io.StringIO("\n".join(lines)), sep=r"\s+", names=cols,
                         na_values=['999.99', '99.999', '9999.99'])
        df = df.dropna(subset=["height_wrt_ref"])
        df["date"] = pd.to_datetime(df["date"].astype(str), format="%Y%m%d")
        out[f"lake_{name.lower()}"] = df.groupby("date")["height_wrt_ref"].median()
    ds = xr.open_dataset(BASE / "Water levels lakes" / "water_level_altimetry_Albert.nc")
    alb = pd.Series(ds["water_level"].values, index=pd.to_datetime(ds["datetime"].values))
    out["lake_albert"] = alb.groupby(alb.index.normalize()).median()
    return pd.DataFrame(out)


def discharge_series():
    out = {}
    for col, sid in [("q_malakal", 100205), ("q_ethiopia", 1505)]:
        df = pd.read_csv(BASE / "Darthmouth Flood Observatory" / f"{sid}_discharge.csv", parse_dates=["Date"])
        out[col] = df.set_index("Date")["Discharge (m3/s)"]
    return pd.DataFrame(out)


# --------------------------------------------------------------------------------------
# ERA5 precipitation & runoff (hourly -> daily, area mean)
# --------------------------------------------------------------------------------------
def era5_year(year):
    ssd = south_sudan_geom()
    ds = xr.open_dataset(BASE / "rainfall and runoff" / f"ERA5_{year}.nc")
    masks = grid_masks(ds.latitude.values, ds.longitude.values, ssd)
    times = pd.to_datetime(ds.valid_time.values)
    res = {}
    for var in ["tp", "ro"]:
        arr = ds[var].values  # (time, lat, lon), metres per hour
        arr = np.nan_to_num(arr, nan=0.0)
        for reg, m in masks.items():
            s = pd.Series(arr[:, m].mean(axis=1) * 1000.0, index=times)  # mm
            res[f"{var}_{reg}"] = s.resample("1D").sum()
    ds.close()
    print(f"  ERA5 {year} done")
    return pd.DataFrame(res)


# --------------------------------------------------------------------------------------
# AgERA5 reference evapotranspiration (one file per day)
# --------------------------------------------------------------------------------------
_ET_MASKS = None


def et_file(path):
    global _ET_MASKS
    with xr.open_dataset(path) as ds:
        if _ET_MASKS is None:
            _ET_MASKS = grid_masks(ds.lat.values, ds.lon.values, south_sudan_geom())
        a = ds["ReferenceET_PenmanMonteith_FAO56"].values[0]
        t = pd.Timestamp(ds["time"].values[0])
    return t, {f"et_{k}": float(np.nanmean(a[m])) for k, m in _ET_MASKS.items() if k in ("ssd", "sudd", "lakes")}


def et_series():
    files = sorted(glob.glob(str(BASE / "evapotranspiration" / "ET_*" / "*.nc")))
    res = []                      # one file per day; a plain loop works inside a notebook on every OS
    for i, f in enumerate(files):
        res.append(et_file(f))
        if i % 1000 == 0:
            print(f"  evapotranspiration {i}/{len(files)} files", flush=True)
    return pd.DataFrame({t: v for t, v in res}).T.sort_index()


def main():
    ssd = south_sudan_geom()
    cache = {}
    for name, fn in [("flood", lambda: flood_series(ssd)), ("lakes", lake_series),
                     ("discharge", discharge_series), ("et", et_series)]:
        p = OUT / f"_{name}.parquet"
        if not p.exists():
            print(f"Extracting {name} ...")
            fn().to_parquet(p)
        cache[name] = pd.read_parquet(p)

    p = OUT / "_era5.parquet"
    if not p.exists():
        print("Extracting ERA5 ...")
        pd.concat([era5_year(y) for y in YEARS]).sort_index().to_parquet(p)
    cache["era5"] = pd.read_parquet(p)

    idx = pd.date_range("2000-01-01", "2025-12-31", freq="D")
    daily = pd.concat([v.reindex(idx) for v in cache.values()], axis=1)
    # Days without any detection in a year that has data are true zeros for the flood counts
    for c in [c for c in daily if c.endswith("_px_ssd") or c.endswith("_px_sudd")]:
        daily[c] = daily[c].fillna(0)
    daily.index.name = "date"
    daily.to_parquet(OUT / "daily_series.parquet")
    print(daily.describe().T.to_string())

In [ ]:
run_step(build_dataset.main, ["data/daily_series.parquet"])
d = pd.read_parquet(PROJ / "data" / "daily_series.parquet")
print(d.shape, d.index.min(), "→", d.index.max())
d[["unusual_px_ssd", "recurring_px_ssd", "lake_albert", "tp_ssd", "tp_lakes"]].describe().round(2)

## 2a · Shared setup: features, target and models (definitions only)

<details><summary>About this code</summary>

```
Research question 12.1 — Step 2: drivers of extreme/unusual flooding and early warning from upstream anomalies.

Inputs : extreme_flood_early_warning/data/daily_series.parquet   (from build_dataset.py)
Outputs: extreme_flood_early_warning/outputs/{tables,figures}

Pipeline
  1. Weekly hydro-climatic features (all strictly causal: only information available at week t).
  2. Target: an "extreme / unusual flood week" = unusual-flood extent in South Sudan above the 90th percentile
     of the 2003-2019 reference period (the "normal seasonal flooding" regime).
  3. Association analysis: lagged correlations and onset composites of each driver.
  4. Prediction: multi-horizon GRU (2-26 weeks ahead) with feature-group ablations, benchmarked against
     climatology, persistence, logistic regression and gradient boosting.
     Validation: leave-one-year-out with a 13-week buffer, plus a forward test (train <=2018, test 2019-2025)
     that asks whether the 2020 regime shift could have been anticipated.
  5. Permutation importance of feature groups for the GRU.
```
</details>

This cell only **defines** the features, the extreme-week target and the models (used by the EDA and every later step). Training happens in step 3.

In [ ]:
%%module train_models
import json
import os
import warnings
from pathlib import Path

# torch and sklearn ship different OpenMP runtimes; multi-threaded use of both segfaults on macOS/conda
os.environ.setdefault("OMP_NUM_THREADS", "1")

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, brier_score_loss, roc_auc_score
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn

matplotlib.use("Agg")
warnings.filterwarnings("ignore", category=RuntimeWarning)
torch.set_num_threads(1)

HERE = Path(__file__).resolve().parent
DATA = HERE / "data" / "daily_series.parquet"
OUT_T = HERE / "outputs" / "tables"
OUT_F = HERE / "outputs" / "figures"
OUT_T.mkdir(parents=True, exist_ok=True)
OUT_F.mkdir(parents=True, exist_ok=True)

PIXEL_KM2 = 0.0534          # MCDWD pixel (0.0020833 deg) at ~7 N
REF = (2003, 2019)          # reference "normal regime" period for anomalies and the event threshold
START, END = "2003-01-01", "2025-12-31"
HORIZONS = [2, 4, 8, 13, 26]  # weeks ahead
LOOKBACK = 52
SEEDS = [0, 1, 2]
BUFFER_W = 13

# Palette (validated reference palette, light mode)
C = dict(blue="#2a78d6", orange="#eb6834", aqua="#1baf7a", yellow="#eda100", violet="#4a3aa7",
         red="#e34948", ink="#0b0b0b", ink2="#52514e", muted="#898781", grid="#e1e0d9", surface="#fcfcfb")
plt.rcParams.update({
    "figure.facecolor": C["surface"], "axes.facecolor": C["surface"], "axes.edgecolor": "#c3c2b7",
    "axes.labelcolor": C["ink2"], "xtick.color": C["muted"], "ytick.color": C["muted"], "text.color": C["ink"],
    "axes.grid": True, "grid.color": C["grid"], "grid.linewidth": 0.6, "axes.spines.top": False,
    "axes.spines.right": False, "font.size": 9.5, "axes.titlesize": 11, "axes.titleweight": "bold",
    "axes.titlelocation": "left", "lines.linewidth": 2, "legend.frameon": False,
})

# --------------------------------------------------------------------------------------
# 1. Weekly features
# --------------------------------------------------------------------------------------
FEATURE_GROUPS = {
    "Lake Albert level (upstream)": ["albert_anom", "albert_d13", "albert_d26"],
    "Equatorial-lakes rain & runoff (upstream)": ["tp_lakes_a4", "tp_lakes_a13", "tp_lakes_a26", "tp_lakes_a52",
                                                  "ro_lakes_a13", "ro_lakes_a52"],
    "Ethiopian highlands / Sobat (upstream)": ["tp_sobat_a4", "tp_sobat_a13", "ro_sobat_a13"],
    # DFO 1505 drains to Lake Turkana, not the Nile: a regional-wetness proxy, deliberately not "upstream"
    "SW Ethiopia discharge (regional)": ["q_ethiopia_anom"],
    "Local rain, runoff & ET (South Sudan)": ["tp_ssd_a1", "tp_ssd_a4", "tp_ssd_a13", "tp_ssd_a26", "ro_ssd_a4",
                                              "ro_ssd_a13", "tp_sudd_a13", "tp_ghazal_a13", "et_ssd_a4",
                                              "wb_ssd_a13", "wb_ssd_a26"],
    "White Nile discharge at Malakal (local)": ["q_malakal_anom"],
    "Current flood state": ["log_unusual", "log_recurring", "unusual_share"],
    "Seasonality": ["woy_sin", "woy_cos"],
}
UPSTREAM = [g for g in FEATURE_GROUPS if "(upstream)" in g]
VARIANTS = {
    "GRU – all features": list(FEATURE_GROUPS),
    "GRU – hydro-climate only (no flood state)": [g for g in FEATURE_GROUPS if g != "Current flood state"],
    "GRU – without upstream": [g for g in FEATURE_GROUPS if g not in UPSTREAM],
    "GRU – upstream + season only": UPSTREAM + ["Seasonality"],
}
ENSEMBLE = "Ensemble (GRU + logistic + boosting)"


def woy_anomaly(s, ref_mask):
    """Standardised anomaly relative to the week-of-year climatology of the reference period."""
    woy = s.index.isocalendar().week.clip(upper=52).values
    ref = pd.DataFrame({"v": s.values, "w": woy})[ref_mask]
    clim = ref.groupby("w")["v"].agg(["mean", "std"])
    # smooth the climatology over +-2 weeks (circular) to reduce sampling noise
    clim = pd.concat([clim.iloc[-2:], clim, clim.iloc[:2]]).rolling(5, center=True).mean().iloc[2:-2]
    mu, sd = clim["mean"].reindex(woy).values, clim["std"].reindex(woy).values
    return pd.Series((s.values - mu) / np.where(sd > 0, sd, np.nan), index=s.index)


def build_weekly():
    d = pd.read_parquet(DATA)
    d.index = pd.to_datetime(d.index)
    flood_cols = [c for c in d if "_px_" in c]
    flux_cols = [c for c in d if c.startswith(("tp_", "ro_", "et_"))]
    w = pd.concat([
        d[flood_cols].resample("W-SUN").max() * PIXEL_KM2,          # robust to cloud-gapped days
        d[flux_cols].resample("W-SUN").sum(),                        # weekly totals [mm]
        d[["q_malakal", "q_ethiopia"]].resample("W-SUN").mean(),
        d[["lake_albert"]].resample("W-SUN").median(),
    ], axis=1)
    w.columns = [c.replace("_px_", "_km2_") for c in w.columns]
    w["wb_ssd"] = w["tp_ssd"] - w["et_ssd"]
    ref = (w.index.year >= REF[0]) & (w.index.year <= REF[1])

    f = pd.DataFrame(index=w.index)
    # Upstream lake: last-observation-carried-forward (causal; satellite revisit is ~10-35 days)
    alb = w["lake_albert"].ffill(limit=26)
    f["albert_anom"] = alb - alb[ref].mean()
    f["albert_d13"] = alb - alb.shift(13)
    f["albert_d26"] = alb - alb.shift(26)
    # Fluxes: standardised weekly anomalies, then accumulated over k weeks (SPI-like, scaled by sqrt(k))
    for v in ["tp_lakes", "ro_lakes", "tp_sobat", "ro_sobat", "tp_ssd", "ro_ssd", "tp_sudd", "tp_ghazal",
              "et_ssd", "wb_ssd"]:
        a = woy_anomaly(w[v], ref)
        for k in [1, 4, 13, 26, 52]:
            f[f"{v}_a{k}"] = a.rolling(k, min_periods=k).sum() / np.sqrt(k)
    for q in ["q_malakal", "q_ethiopia"]:
        f[f"{q}_anom"] = woy_anomaly(w[q], ref)
    f["log_unusual"] = np.log1p(w["unusual_km2_ssd"])
    f["log_recurring"] = np.log1p(w["recurring_km2_ssd"])
    f["unusual_share"] = w["unusual_km2_ssd"] / (w["unusual_km2_ssd"] + w["recurring_km2_ssd"] + 1e-9)
    woy = w.index.isocalendar().week.values.astype(float)
    f["woy_sin"], f["woy_cos"] = np.sin(2 * np.pi * woy / 52.18), np.cos(2 * np.pi * woy / 52.18)

    thr = w.loc[ref, "unusual_km2_ssd"].quantile(0.90)
    w["extreme"] = (w["unusual_km2_ssd"] > thr).astype(int)
    return w, f, thr


# --------------------------------------------------------------------------------------
# 2. Association analysis
# --------------------------------------------------------------------------------------
DRIVERS = {
    "Lake Albert level anomaly": "albert_anom",
    "Lake Albert 26-wk rise": "albert_d26",
    "Equatorial-lakes rain, 52-wk": "tp_lakes_a52",
    "Equatorial-lakes runoff, 52-wk": "ro_lakes_a52",
    "Sobat/Ethiopia rain, 13-wk": "tp_sobat_a13",
    "SW Ethiopia discharge (regional)": "q_ethiopia_anom",
    "South Sudan rain, 4-wk": "tp_ssd_a4",
    "South Sudan rain, 26-wk": "tp_ssd_a26",
    "South Sudan runoff, 13-wk": "ro_ssd_a13",
    "South Sudan P–ET, 26-wk": "wb_ssd_a26",
    "Sudd rain, 13-wk": "tp_sudd_a13",
    "Malakal discharge": "q_malakal_anom",
}


def association_analysis(w, f):
    target = woy_anomaly(np.log1p(w["unusual_km2_ssd"]), np.ones(len(w), bool))  # de-seasonalised log extent
    lags = list(range(0, 53, 2))
    rows = []
    for period, (a, b) in {"2003–2025": (2003, 2025), "2003–2019 (pre-shift)": (2003, 2019)}.items():
        m = (w.index.year >= a) & (w.index.year <= b)
        for name, col in DRIVERS.items():
            for lag in lags:
                x = f[col].shift(lag)[m]
                ok = x.notna() & target[m].notna()
                r = spearmanr(x[ok], target[m][ok]).statistic if ok.sum() > 30 else np.nan
                rows.append(dict(period=period, driver=name, lag_weeks=lag, spearman_r=r))
    corr = pd.DataFrame(rows)
    corr.to_csv(OUT_T / "lagged_correlations.csv", index=False)

    # Onset composites: first extreme week after >= 8 non-extreme weeks
    ext = w["extreme"].values
    onsets = [i for i in range(8, len(ext)) if ext[i] == 1 and ext[i - 8:i].sum() == 0
              and START <= str(w.index[i].date()) <= END]
    comp = {}
    for name, col in DRIVERS.items():
        mat = np.array([f[col].values[i - 26:i + 5] for i in onsets if i >= 26 and i + 5 <= len(f)])
        comp[name] = np.nanmean(mat, axis=0)
    comp = pd.DataFrame(comp, index=range(-26, 5))
    comp.to_csv(OUT_T / "onset_composites.csv")

    # Standardised driver contrast: extreme vs normal weeks, same flood season (Jul-Dec)
    m = (w.index.month >= 7) & (w.index >= START) & (w.index <= END)
    cont = []
    for name, col in DRIVERS.items():
        x = f.loc[m, col]
        e, n = x[w.loc[m, "extreme"] == 1], x[w.loc[m, "extreme"] == 0]
        sd = np.sqrt((e.var() + n.var()) / 2)
        cont.append(dict(driver=name, mean_extreme=e.mean(), mean_normal=n.mean(), cohens_d=(e.mean() - n.mean()) / sd))
    cont = pd.DataFrame(cont).sort_values("cohens_d", ascending=False)
    cont.to_csv(OUT_T / "driver_contrast_extreme_vs_normal.csv", index=False)
    return corr, comp, onsets, cont


def indicator_skill(w, f):
    """Early-warning value of each raw driver, no model fitting: ROC-AUC of the driver value at week t
    for discriminating extreme vs. normal weeks at t+h. AUC < 0.5 means the driver is inversely related."""
    rows = []
    for period, (a, b) in {"2003–2025": (2003, 2025), "2003–2019 (pre-shift)": (2003, 2019)}.items():
        m = (w.index.year >= a) & (w.index.year <= b)
        for name, col in DRIVERS.items():
            for h in [0] + HORIZONS:
                yy = w["extreme"].shift(-h)[m]
                x = f[col][m]
                ok = yy.notna() & x.notna()
                auc = roc_auc_score(yy[ok], x[ok]) if yy[ok].nunique() == 2 else np.nan
                rows.append(dict(period=period, driver=name, horizon=h, auc=auc))
    ind = pd.DataFrame(rows)
    ind.to_csv(OUT_T / "indicator_skill_univariate_auc.csv", index=False)

    # Lead of upstream threshold crossings over the onset of the sustained extreme regime of the 2020s
    ext = w["extreme"]
    share_ahead = ext[::-1].rolling(26, min_periods=26).mean()[::-1]   # share of extreme weeks in [t, t+25]
    regime_start = share_ahead[(share_ahead >= 0.5) & (share_ahead.index >= "2019-01-01")].index[0]

    def first_sustained(s, thr, n=4):
        hit = (s > thr).astype(int).rolling(n).sum() == n
        hit = hit[hit.index >= "2019-01-01"]
        return hit[hit].index[0] - pd.Timedelta(weeks=n - 1) if hit.any() else None

    leads = []
    for label, col, thr in [("Lake Albert anomaly > +0.5 m", "albert_anom", 0.5),
                            ("Lake Albert anomaly > +1.0 m", "albert_anom", 1.0),
                            ("Equatorial-lakes 52-wk rain > +2σ", "tp_lakes_a52", 2.0),
                            ("Equatorial-lakes 52-wk runoff > +2σ", "ro_lakes_a52", 2.0)]:
        d = first_sustained(f[col], thr)
        # how often did the same signal fire in 2003-2018 (false-alarm check)?
        pre = f[col][(f.index >= START) & (f.index < "2019-01-01")]
        leads.append(dict(signal=label, first_crossing=None if d is None else str(d.date()),
                          sustained_regime_onset=str(regime_start.date()),
                          lead_weeks=None if d is None else int((regime_start - d).days // 7),
                          weeks_above_threshold_2003_2018=int((pre > thr).sum())))
    leads = pd.DataFrame(leads)
    leads.to_csv(OUT_T / "regime_shift_early_warning_leads.csv", index=False)
    return ind, leads, regime_start


def fig_indicator(ind):
    show = [("Lake Albert level anomaly", C["violet"]), ("Equatorial-lakes runoff, 52-wk", C["blue"]),
            ("Sobat/Ethiopia rain, 13-wk", C["yellow"]), ("South Sudan rain, 26-wk", C["aqua"]),
            ("South Sudan rain, 4-wk", C["orange"])]
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.4), sharey=True)
    for ax, period in zip(axes, ind["period"].unique()):
        for name, c in show:
            s = ind[(ind.period == period) & (ind.driver == name)]
            ax.plot(s["horizon"], s["auc"], color=c, marker="o", ms=4.5, label=name)
        ax.axhline(0.5, color="#c3c2b7", lw=1)
        ax.set_xticks([0] + HORIZONS)
        ax.set_xlabel("Lead time (weeks)")
        ax.set_title(period)
    axes[0].set_ylabel("ROC-AUC of the raw driver value")
    axes[1].legend(loc="center left", bbox_to_anchor=(1.01, 0.5), fontsize=8.5)
    fig.suptitle("Early-warning value of single drivers (no model): does the anomaly at week t flag an extreme at t + lead?",
                 x=0.01, ha="left", fontweight="bold")
    fig.tight_layout()
    fig.savefig(OUT_F / "08_single_driver_early_warning_skill.png", dpi=170, bbox_inches="tight")
    plt.close(fig)


# --------------------------------------------------------------------------------------
# 3. Models
# --------------------------------------------------------------------------------------
class GRUNet(nn.Module):
    def __init__(self, n_in, n_out, hidden=32):
        super().__init__()
        self.gru = nn.GRU(n_in, hidden, batch_first=True)
        self.drop = nn.Dropout(0.25)
        self.head = nn.Linear(hidden, n_out)

    def forward(self, x):
        _, h = self.gru(x)
        return self.head(self.drop(h[-1]))


def make_targets(w):
    y = np.stack([w["extreme"].shift(-h).values for h in HORIZONS], axis=1)
    return y  # NaN where t+h is beyond the record


def sequences(X, idx):
    return np.stack([X[i - LOOKBACK + 1:i + 1] for i in idx])


def fit_gru(Xtr, ytr, seed, epochs=60):
    torch.manual_seed(seed)
    np.random.seed(seed)
    net = GRUNet(Xtr.shape[2], ytr.shape[1])
    opt = torch.optim.AdamW(net.parameters(), lr=3e-3, weight_decay=1e-2)
    pos = np.nanmean(ytr, axis=0)
    pw = torch.tensor((1 - pos) / np.clip(pos, 0.05, None), dtype=torch.float32).clamp(max=10)
    Xt, yt = torch.tensor(Xtr, dtype=torch.float32), torch.tensor(ytr, dtype=torch.float32)
    mask = ~torch.isnan(yt)
    yt = torch.nan_to_num(yt)
    lossf = nn.BCEWithLogitsLoss(pos_weight=pw, reduction="none")
    n = len(Xt)
    for _ in range(epochs):
        net.train()
        perm = torch.randperm(n)
        for b in range(0, n, 64):
            j = perm[b:b + 64]
            xb = Xt[j] + 0.05 * torch.randn_like(Xt[j])  # input noise: regularises on a small sample
            loss = (lossf(net(xb), yt[j]) * mask[j]).sum() / mask[j].sum().clamp(min=1)
            opt.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(net.parameters(), 1.0)
            opt.step()
    net.eval()
    return net


def predict_gru(nets, X):
    with torch.no_grad():
        Xt = torch.tensor(X, dtype=torch.float32)
        return np.mean([torch.sigmoid(n(Xt)).numpy() for n in nets], axis=0)


def cols_for(groups):
    return [c for g in groups for c in FEATURE_GROUPS[g]]


def make_folds(dates, valid):
    """Leave-one-year-out with a buffer, plus the forward 'regime shift' test."""
    years = sorted(set(dates[valid].year))
    folds = []
    for y in years:
        te = valid & (dates.year == y)
        lo, hi = pd.Timestamp(f"{y}-01-01") - pd.Timedelta(weeks=BUFFER_W), pd.Timestamp(f"{y}-12-31") + pd.Timedelta(weeks=BUFFER_W)
        tr = valid & ~((dates >= lo) & (dates <= hi))
        folds.append((f"LOYO {y}", tr, te))
    return folds


def run_models(w, f, train_mask, test_mask, y, tag, permute=False, fitted=None):
    """Fit every model on train_mask, predict test_mask. Returns {model: probs[n_test, H]} and permutation results.
    If `fitted` is a dict, the fitted scalers/models are stored in it (used for the final, all-data models)."""
    preds, perm_out = {}, {}
    idx_tr, idx_te = np.where(train_mask)[0], np.where(test_mask)[0]
    ytr = y[idx_tr]

    # Climatology: P(extreme at t+h | week of year of t+h) from training targets
    woy = w.index.isocalendar().week.clip(upper=52).values
    clim = np.zeros((len(idx_te), len(HORIZONS)))
    for k, h in enumerate(HORIZONS):
        wt = np.roll(woy, -h)
        tab = pd.Series(ytr[:, k]).groupby(wt[idx_tr]).mean()
        base = np.nanmean(ytr[:, k])
        clim[:, k] = pd.Series(wt[idx_te]).map(tab).fillna(base).values
    preds["Climatology"] = clim

    # Tabular models on the current-week feature vector
    def tab_model(cols, make, name=None):
        out = np.zeros((len(idx_te), len(HORIZONS)))
        Xa = f[cols].fillna(0).values
        sc = StandardScaler().fit(Xa[idx_tr])
        mdls = []
        for k in range(len(HORIZONS)):
            ok = ~np.isnan(ytr[:, k])
            if len(np.unique(ytr[ok, k])) < 2:
                out[:, k] = np.nanmean(ytr[:, k])
                mdls.append(None)
                continue
            mdl = make().fit(sc.transform(Xa[idx_tr][ok]), ytr[ok, k])
            out[:, k] = mdl.predict_proba(sc.transform(Xa[idx_te]))[:, 1]
            mdls.append(mdl)
        if fitted is not None and name:
            fitted[name] = dict(columns=cols, scaler=sc, models=mdls)
        return out

    preds["Persistence (logistic, flood state + season)"] = tab_model(
        cols_for(["Current flood state", "Seasonality"]), lambda: LogisticRegression(C=1.0, max_iter=2000))
    all_cols = cols_for(FEATURE_GROUPS)
    preds["Logistic regression – all features"] = tab_model(
        all_cols, lambda: LogisticRegression(C=0.1, max_iter=5000), "logistic")
    preds["Gradient boosting – all features"] = tab_model(
        all_cols, lambda: HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=200,
                                                         l2_regularization=1.0, random_state=0), "gbm")

    # GRU variants
    for vname, groups in VARIANTS.items():
        cols = cols_for(groups)
        Xa = f[cols].values.astype(float)
        sc = StandardScaler().fit(np.nan_to_num(Xa[idx_tr]))
        Xs = np.nan_to_num(sc.transform(np.nan_to_num(Xa)))
        nets = [fit_gru(sequences(Xs, idx_tr), ytr, s) for s in SEEDS]
        Xte = sequences(Xs, idx_te)
        preds[vname] = predict_gru(nets, Xte)
        if fitted is not None and vname == "GRU – all features":
            fitted["gru"] = dict(columns=cols, scaler=sc, state_dicts=[n.state_dict() for n in nets],
                                 n_in=len(cols), lookback=LOOKBACK, horizons=HORIZONS)

        if permute and vname == "GRU – all features":
            rng = np.random.default_rng(0)
            for g in FEATURE_GROUPS:
                ci = [cols.index(c) for c in FEATURE_GROUPS[g]]
                reps = []
                for _ in range(5):
                    Xp = Xte.copy()
                    p = rng.permutation(len(Xp))
                    Xp[:, :, ci] = Xte[p][:, :, ci]   # swap whole input histories of this group between samples
                    reps.append(predict_gru(nets, Xp))
                perm_out[g] = np.mean(reps, axis=0)

    preds[ENSEMBLE] = np.mean([preds["GRU – all features"], preds["Logistic regression – all features"],
                               preds["Gradient boosting – all features"]], axis=0)
    print(f"    {tag}: done")
    return preds, perm_out


def score(y, p):
    ok = ~np.isnan(y) & ~np.isnan(p)
    y, p = y[ok], p[ok]
    if len(np.unique(y)) < 2:
        return dict(auc=np.nan, ap=np.nan, brier=brier_score_loss(y, p), n=len(y), pos=y.mean())
    return dict(auc=roc_auc_score(y, p), ap=average_precision_score(y, p), brier=brier_score_loss(y, p),
                n=len(y), pos=y.mean())


# --------------------------------------------------------------------------------------
# 4. Figures
# --------------------------------------------------------------------------------------
def fig_timeline(w, f, thr):
    fig, axes = plt.subplots(3, 1, figsize=(11, 7.5), sharex=True, gridspec_kw=dict(height_ratios=[2, 1.2, 1.2]))
    m = (w.index >= START) & (w.index <= END)
    ww, ff = w[m], f[m]
    ax = axes[0]
    for i, (s, e) in enumerate(_runs(ww["extreme"].values)):
        ax.axvspan(ww.index[s], ww.index[e], color=C["red"], alpha=0.08, lw=0, label="Extreme week" if i == 0 else None)
    ax.plot(ww.index, ww["recurring_km2_ssd"], color=C["blue"], lw=1.2, label="Recurring (normal seasonal) flood")
    ax.plot(ww.index, ww["unusual_km2_ssd"], color=C["orange"], lw=1.2, label="Unusual flood")
    ax.axhline(thr, color=C["ink2"], lw=1, ls="--")
    ax.text(ww.index[5], thr * 1.25, f"Extreme threshold: P90 of 2003–2019 = {thr:,.0f} km²", color=C["ink2"], fontsize=8.5)
    ax.set_yscale("log")
    ax.set_ylabel("Detected flood extent, km² (log)")
    ax.set_title("Flood regime in South Sudan: normal seasonal vs. unusual flooding (weekly max of 3-day composites)")
    ax.legend(loc="upper left", ncol=3, fontsize=8.5)
    ax = axes[1]
    ax.plot(ff.index, ff["albert_anom"], color=C["violet"], lw=1.6)
    ax.axhline(0, color="#c3c2b7", lw=1)
    ax.set_ylabel("m vs. 2003–19 mean")
    ax.set_title("Upstream: Lake Albert water-level anomaly (satellite altimetry)")
    ax = axes[2]
    ax.fill_between(ff.index, 0, ff["tp_lakes_a52"], where=ff["tp_lakes_a52"] > 0, color=C["blue"], alpha=0.6, lw=0)
    ax.fill_between(ff.index, 0, ff["tp_lakes_a52"], where=ff["tp_lakes_a52"] <= 0, color=C["red"], alpha=0.45, lw=0)
    ax.axhline(0, color="#c3c2b7", lw=1)
    ax.set_ylabel("Standardised anomaly")
    ax.set_title("Upstream: Equatorial-lakes basin rainfall, 52-week accumulated anomaly (ERA5)")
    fig.tight_layout()
    fig.savefig(OUT_F / "01_flood_regime_and_upstream_timeline.png", dpi=170)
    plt.close(fig)


def _runs(a):
    runs, s = [], None
    for i, v in enumerate(a):
        if v and s is None:
            s = i
        if not v and s is not None:
            runs.append((s, i - 1)); s = None
    if s is not None:
        runs.append((s, len(a) - 1))
    return runs


def fig_lagcorr(corr):
    from matplotlib.colors import LinearSegmentedColormap
    cmap = LinearSegmentedColormap.from_list("div", [C["blue"], "#f0efec", C["red"]])
    fig, axes = plt.subplots(1, 2, figsize=(12, 5.2), sharey=True)
    for ax, period in zip(axes, corr["period"].unique()):
        piv = corr[corr.period == period].pivot(index="driver", columns="lag_weeks", values="spearman_r")
        piv = piv.loc[list(DRIVERS)]
        im = ax.imshow(piv.values, cmap=cmap, vmin=-0.8, vmax=0.8, aspect="auto")
        ax.set_xticks(range(0, piv.shape[1], 2), [str(c) for c in piv.columns[::2]])
        ax.set_yticks(range(len(piv)), piv.index)
        ax.set_xlabel("Driver leads flood anomaly by (weeks)")
        ax.set_title(period)
        ax.grid(False)
    cb = fig.colorbar(im, ax=axes, shrink=0.8, pad=0.02)
    cb.set_label("Spearman r with de-seasonalised unusual-flood extent")
    fig.suptitle("Which drivers lead unusual flooding, and by how long?", x=0.01, ha="left", fontweight="bold")
    fig.savefig(OUT_F / "02_lagged_driver_correlations.png", dpi=170, bbox_inches="tight")
    plt.close(fig)


def fig_onsets(comp, n_onsets):
    show = ["Lake Albert level anomaly", "Equatorial-lakes rain, 52-wk", "South Sudan rain, 26-wk",
            "South Sudan rain, 4-wk", "Sobat/Ethiopia rain, 13-wk"]
    cols = [C["violet"], C["blue"], C["aqua"], C["orange"], C["yellow"]]
    fig, ax = plt.subplots(figsize=(9, 4.6))
    for name, c in zip(show, cols):
        ax.plot(comp.index, comp[name], color=c, label=name)
        ax.text(comp.index[-1] + 0.4, comp[name].iloc[-1], name, color=C["ink2"], fontsize=8, va="center")
    ax.axvline(0, color=C["ink2"], lw=1, ls="--")
    ax.axhline(0, color="#c3c2b7", lw=1)
    ax.set_xlim(-26, 11)
    ax.set_xlabel("Weeks relative to onset of an extreme-flood episode")
    ax.set_ylabel("Mean anomaly (m for Albert; σ for rainfall)")
    ax.set_title(f"Driver anomalies before the onset of extreme-flood episodes (n = {n_onsets} onsets, 2003–2025)")
    fig.tight_layout()
    fig.savefig(OUT_F / "03_onset_composites.png", dpi=170)
    plt.close(fig)


MODEL_STYLE = {
    "Climatology": (C["muted"], "--"),
    "Persistence (logistic, flood state + season)": (C["ink2"], ":"),
    "Logistic regression – all features": (C["aqua"], "-"),
    "Gradient boosting – all features": (C["yellow"], "-"),
    "GRU – all features": (C["blue"], "-"),
    ENSEMBLE: (C["red"], "-"),
    "GRU – hydro-climate only (no flood state)": (C["aqua"], "-"),
    "GRU – without upstream": (C["orange"], "-"),
    "GRU – upstream + season only": (C["violet"], "-"),
}
MAIN_MODELS = ["Climatology", "Persistence (logistic, flood state + season)", "Logistic regression – all features",
               "Gradient boosting – all features", "GRU – all features", ENSEMBLE]
ABLATIONS = ["Persistence (logistic, flood state + season)"] + list(VARIANTS)


def fig_skill(summary, title, fname, models):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))
    for ax, metric, lab in [(axes[0], "auc", "ROC-AUC"), (axes[1], "bss", "Brier skill score vs. climatology")]:
        for mname in models:
            c, ls = MODEL_STYLE[mname]
            s = summary[summary.model == mname].sort_values("horizon")
            label = mname + (" (BSS = 0 by definition)" if mname == "Climatology" else "")
            if metric == "bss" and mname == "Climatology":
                ax.plot([], [], color=c, ls=ls, marker="o", ms=4.5, lw=1.8, label=label)
                continue
            ax.plot(s["horizon"], s[metric], color=c, ls=ls, marker="o", ms=4.5, lw=1.8, label=label)
        ax.set_xticks(HORIZONS)
        ax.set_xlabel("Lead time (weeks)")
        ax.set_title(lab)
        if metric == "bss":
            ax.axhline(0, color="#c3c2b7", lw=1)
        else:
            ax.axhline(0.5, color="#c3c2b7", lw=1)
    axes[1].legend(loc="center left", bbox_to_anchor=(1.01, 0.5), fontsize=8.5)
    fig.suptitle(title, x=0.01, ha="left", fontweight="bold")
    fig.tight_layout()
    fig.savefig(OUT_F / fname, dpi=170, bbox_inches="tight")
    plt.close(fig)


def fig_importance(imp):
    piv = imp.pivot(index="group", columns="horizon", values="auc_drop")
    piv = piv.loc[piv.mean(axis=1).sort_values().index]
    fig, ax = plt.subplots(figsize=(9.5, 4.4))
    hs = list(piv.columns)
    cols = [C["blue"], C["aqua"], C["yellow"], C["orange"], C["violet"]]
    bh = 0.8 / len(hs)
    for k, (h, c) in enumerate(zip(hs, cols)):
        ax.barh(np.arange(len(piv)) + (k - len(hs) / 2 + 0.5) * bh, piv[h], height=bh * 0.85, color=c, label=f"{h} wk")
    ax.set_yticks(range(len(piv)), piv.index)
    ax.axvline(0, color="#c3c2b7", lw=1)
    ax.set_xlabel("Drop in out-of-sample ROC-AUC when the group is permuted")
    ax.set_title("What the GRU relies on, by lead time (leave-one-year-out)")
    ax.legend(title="Lead", fontsize=8, title_fontsize=8, loc="lower right")
    ax.grid(axis="y", visible=False)
    fig.tight_layout()
    fig.savefig(OUT_F / "05_gru_feature_group_importance.png", dpi=170)
    plt.close(fig)


def fig_forward(w, test_mask, y, preds, k_show=(2, 3)):
    dates = w.index[test_mask]
    fig, axes = plt.subplots(len(k_show), 1, figsize=(11, 3.2 * len(k_show)), sharex=True)
    for ax, k in zip(np.atleast_1d(axes), k_show):
        h = HORIZONS[k]
        tgt_dates = dates + pd.Timedelta(weeks=h)
        yy = y[test_mask][:, k]
        for i, (s, e) in enumerate(_runs(np.nan_to_num(yy).astype(int))):
            ax.axvspan(tgt_dates[s], tgt_dates[e], color=C["red"], alpha=0.08, lw=0,
                       label="Observed extreme week" if i == 0 else None)
        for mname, c in [("GRU – all features", C["blue"]), ("GRU – upstream + season only", C["violet"]),
                         ("Persistence (logistic, flood state + season)", C["ink2"])]:
            ax.plot(tgt_dates, preds[mname][:, k], color=c, lw=1.4, label=mname)
        ax.set_ylim(-0.02, 1.02)
        ax.set_ylabel("P(extreme)")
        ax.set_title(f"Forward test — trained on 2003–2018 only — forecast issued {h} weeks ahead")
        ax.legend(loc="upper left", fontsize=8, ncol=2)
    fig.tight_layout()
    fig.savefig(OUT_F / "06_forward_test_2019_2025.png", dpi=170)
    plt.close(fig)


# --------------------------------------------------------------------------------------
def main():
    w, f, thr = build_weekly()
    print(f"Extreme threshold (P90 of weekly unusual extent, {REF[0]}-{REF[1]}): {thr:,.0f} km²")
    m = (w.index >= START) & (w.index <= END)
    print("Extreme-week share by year:\n", w.loc[m, "extreme"].groupby(w.index[m].year).mean().round(2).to_string())

    fig_timeline(w, f, thr)
    corr, comp, onsets, cont = association_analysis(w, f)
    fig_lagcorr(corr)
    fig_onsets(comp, len(onsets))
    print("\nDriver contrast (Jul-Dec, extreme vs normal weeks):\n", cont.round(2).to_string(index=False))
    ind, leads, regime_start = indicator_skill(w, f)
    fig_indicator(ind)
    print("\nSingle-driver early-warning AUC:\n",
          ind.pivot_table(index=["period", "driver"], columns="horizon", values="auc").round(2).to_string())
    print("\nUpstream threshold crossings vs. onset of the sustained extreme regime:\n", leads.to_string(index=False))

    y = make_targets(w)
    pos = np.arange(len(w))
    valid = m & (pos >= LOOKBACK - 1)
    valid &= ~np.all(np.isnan(y), axis=1)
    dates = w.index

    # ---- Leave-one-year-out ----
    print("\nLeave-one-year-out CV ...")
    oof, oof_perm = {}, {}
    for name, tr, te in make_folds(dates, valid):
        p, pp = run_models(w, f, tr, te, y, name, permute=True)
        for k, v in p.items():
            oof.setdefault(k, np.full((len(w), len(HORIZONS)), np.nan))[te] = v
        for k, v in pp.items():
            oof_perm.setdefault(k, np.full((len(w), len(HORIZONS)), np.nan))[te] = v

    rows = []
    for mname, P in oof.items():
        for k, h in enumerate(HORIZONS):
            s = score(y[valid, k], P[valid, k])
            rows.append(dict(model=mname, horizon=h, **s))
    loyo = pd.DataFrame(rows)
    clim_b = loyo[loyo.model == "Climatology"].set_index("horizon")["brier"]
    loyo["bss"] = 1 - loyo["brier"] / loyo["horizon"].map(clim_b)
    loyo.to_csv(OUT_T / "skill_leave_one_year_out.csv", index=False)
    print(loyo.round(3).to_string(index=False))

    base = oof["GRU – all features"]
    imp = []
    for g, P in oof_perm.items():
        for k, h in enumerate(HORIZONS):
            imp.append(dict(group=g, horizon=h, auc_drop=score(y[valid, k], base[valid, k])["auc"]
                            - score(y[valid, k], P[valid, k])["auc"]))
    imp = pd.DataFrame(imp)
    imp.to_csv(OUT_T / "gru_permutation_importance.csv", index=False)

    oof_df = pd.concat({m_: pd.DataFrame(P[valid], index=dates[valid], columns=[f"p_{h}wk" for h in HORIZONS])
                        for m_, P in oof.items()}, names=["model", "week"])
    oof_df.join(pd.DataFrame(y[valid], index=dates[valid], columns=[f"y_{h}wk" for h in HORIZONS]), on="week") \
          .to_parquet(OUT_T / "oof_predictions_leave_one_year_out.parquet")

    fig_skill(loyo, "Out-of-sample skill by lead time — leave-one-year-out, 2003–2025",
              "04a_skill_by_lead_loyo_models.png", MAIN_MODELS)
    fig_skill(loyo, "Which information carries the skill? GRU feature-group ablations — leave-one-year-out",
              "04b_skill_by_lead_loyo_ablations.png", ["Climatology"] + ABLATIONS)
    fig_importance(imp)

    # ---- Forward test: could the 2020 shift have been anticipated? ----
    print("\nForward test (train <=2018, test 2019-2025) ...")
    tgt_end = dates + pd.Timedelta(weeks=max(HORIZONS))
    tr = valid & (tgt_end <= pd.Timestamp("2018-12-31"))
    te = valid & (dates >= pd.Timestamp("2019-01-01"))
    fp, _ = run_models(w, f, tr, te, y, "forward")
    rows = []
    for mname, P in fp.items():
        for k, h in enumerate(HORIZONS):
            rows.append(dict(model=mname, horizon=h, **score(y[te, k], P[:, k])))
    fwd = pd.DataFrame(rows)
    clim_b = fwd[fwd.model == "Climatology"].set_index("horizon")["brier"]
    fwd["bss"] = 1 - fwd["brier"] / fwd["horizon"].map(clim_b)
    fwd.to_csv(OUT_T / "skill_forward_test_2019_2025.csv", index=False)
    print(fwd.round(3).to_string(index=False))
    fig_skill(fwd, "Forward test — trained on 2003–2018, evaluated on the 2019–2025 regime shift",
              "07_skill_by_lead_forward.png", MAIN_MODELS + ["GRU – upstream + season only"])
    fig_forward(w, te, y, fp)

    # ---- Final models on all data + outlook from the latest observed week ----
    print("\nFitting final models on 2003-2025 ...")
    fitted = {}
    last = np.zeros(len(w), bool)
    last[np.where(m)[0][-1]] = True
    outlook, _ = run_models(w, f, valid, last, y, "final", fitted=fitted)
    issued = w.index[last][0]
    out = pd.DataFrame({mname: P[0] for mname, P in outlook.items()}, index=[f"{h} weeks" for h in HORIZONS]).T
    out.columns.name = f"P(extreme week) at lead, issued {issued.date()}"
    out.to_csv(OUT_T / "outlook_from_latest_week.csv")
    print(out.round(2).to_string())
    mdir = HERE / "outputs" / "models"
    mdir.mkdir(parents=True, exist_ok=True)
    torch.save(fitted.pop("gru"), mdir / "gru_all_features.pt")
    import joblib
    joblib.dump(dict(fitted, horizons=HORIZONS, threshold_km2=thr, feature_groups=FEATURE_GROUPS),
                mdir / "tabular_models.joblib")

    json.dump(dict(threshold_km2=float(thr), n_onsets=len(onsets), regime_onset=str(regime_start.date()),
                   outlook_issued=str(issued.date())),
              open(OUT_T / "summary.json", "w"), indent=2)

## 2b · Data cleaning and exploratory data analysis (~1 min)

Which source covers which years, how complete each variable is, every cleaning decision with the number from the data that justifies it, and the basic patterns in the data. The EDA needs the weekly features, so the model-setup code of step 3 is defined first (cell below), without training anything.

<details><summary>About this code</summary>

```
Research question 12.1 — data cleaning and exploratory data analysis (EDA).

Runs after build_dataset.py (and download_external.py for the long lake records). Documents every source,
every cleaning decision with the number that justifies it, and the basic patterns in the data.

Outputs
  outputs/tables/eda_data_inventory.csv      one row per variable: period, observations, % missing 2003-2025
  outputs/tables/eda_cleaning_decisions.csv  what was done, why, and the evidence from the data
  outputs/figures/eda_01_data_coverage.png   which source covers which years
  outputs/figures/eda_02_flood_seasonality.png  yearly flood cycle, 2003-19 vs 2020-25
  outputs/figures/eda_03_extreme_threshold.png  distribution of weekly unusual flood area and the 408 km² threshold
  outputs/figures/eda_04_driver_correlations.png correlations between the weekly driver anomalies
  outputs/figures/eda_05_data_quality.png    flood-mask observation days, altimetry gaps, Malakal discharge artefact
```
</details>

In [ ]:
%%module eda
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import train_models as T

matplotlib.use("Agg")
C = T.C
OUT_T, OUT_F = T.OUT_T, T.OUT_F
EXT = T.HERE.parent / "external"

VARIABLES = {  # column in daily_series.parquet -> (source, description)
    "unusual_px_ssd": ("NASA MODIS/VIIRS flood masks", "unusual flood pixels, South Sudan"),
    "recurring_px_ssd": ("NASA MODIS/VIIRS flood masks", "recurring (normal) flood pixels, South Sudan"),
    "lake_albert": ("DAHITI altimetry", "Lake Albert water level"),
    "lake_victoria": ("Hydroweb altimetry (old file)", "Lake Victoria water level"),
    "lake_kyoga": ("Hydroweb altimetry (old file)", "Lake Kyoga water level"),
    "q_malakal": ("Dartmouth Flood Observatory", "White Nile discharge at Malakal (station 100205)"),
    "q_ethiopia": ("Dartmouth Flood Observatory", "SW Ethiopia discharge (station 1505, Omo-Turkana basin)"),
    "tp_ssd": ("ERA5", "rainfall, South Sudan mean"),
    "tp_lakes": ("ERA5", "rainfall, equatorial-lakes basin"),
    "tp_sobat": ("ERA5", "rainfall, Ethiopian highlands / Sobat"),
    "ro_ssd": ("ERA5", "runoff, South Sudan mean"),
    "et_ssd": ("AgERA5", "reference evapotranspiration, South Sudan mean"),
}


def load():
    d = pd.read_parquet(T.HERE / "data" / "daily_series.parquet")
    d.index = pd.to_datetime(d.index)
    raw_flood = pd.read_parquet(T.HERE / "data" / "_flood.parquet")      # days with at least one detection
    raw_flood.index = pd.to_datetime(raw_flood.index)
    grealm = None
    p = T.HERE / "data" / "external_lakes_grealm.parquet"
    if p.exists():
        grealm = pd.read_parquet(p)
        grealm.index = pd.to_datetime(grealm.index)
    return d, raw_flood, grealm


def inventory(d, raw_flood, grealm):
    win = (d.index >= T.START) & (d.index <= T.END)
    rows = []
    for col, (src, desc) in VARIABLES.items():
        s = d[col]
        if col.endswith("_px_ssd"):        # zero-filled in build_dataset; real observations = days with detections
            obs = raw_flood.index[(raw_flood.index >= "2000-01-01") & (raw_flood.index <= "2025-12-31")]
            first, last, n = obs.min(), obs.max(), len(obs)
            miss = 1 - raw_flood[(raw_flood.index >= T.START) & (raw_flood.index <= T.END)].shape[0] / win.sum()
        else:
            o = s.dropna()
            first, last, n = (o.index.min(), o.index.max(), len(o)) if len(o) else (pd.NaT, pd.NaT, 0)
            miss = 1 - s[win].notna().mean()
        rows.append(dict(variable=col, source=src, description=desc, first=first.date() if pd.notna(first) else None,
                         last=last.date() if pd.notna(last) else None, observations=n,
                         pct_days_missing_2003_2025=round(100 * miss, 1)))
    if grealm is not None:
        for c in grealm:
            o = grealm[c].dropna()
            rows.append(dict(variable=f"{c} (G-REALM)", source="NASA G-REALM altimetry (external)",
                             description=f"Lake {c.capitalize()} water level, 10-day", first=o.index.min().date(),
                             last=o.index.max().date(), observations=len(o),
                             pct_days_missing_2003_2025=round(100 * (1 - len(o[(o.index >= T.START) & (o.index <= T.END)]) * 10
                                                              / win.sum()), 1)))
    inv = pd.DataFrame(rows)
    inv.to_csv(OUT_T / "eda_data_inventory.csv", index=False)
    return inv


def cleaning(d, raw_flood, w):
    days = raw_flood.groupby(raw_flood.index.year).size()
    alb = d["lake_albert"].dropna()
    gaps = alb.index.to_series().diff().dt.days.dropna()
    q = d["q_malakal"].groupby(d.index.year).mean()
    pre = d.loc["2003":"2019"]
    flood_both = ((d["unusual_px_ssd"] > 0) & (d["recurring_px_ssd"] > 0)).mean()
    rows = [
        ("Study period starts in 2003",
         "Flood masks before 2003 have too few observation days (MODIS Aqua launched mid-2002).",
         f"days with detections: 2000 = {days.get(2000, 0)}, 2001 = {days.get(2001, 0)}, 2002 = {days.get(2002, 0)}, "
         f"2003–2025 median = {int(days.loc[2003:2025].median())}"),
        ("Daily flood counts aggregated to the weekly maximum",
         "Single days are often cloud-covered; the maximum of the 3-day composites in a week is robust to gaps.",
         f"days without any detection in 2003–2025: {100 - 100 * len(raw_flood.loc['2003':'2025']) / len(pd.date_range('2003-01-01', '2025-12-31')):.0f}%"),
        ("Pixel counts converted to km²", "0.0020833° pixels at ~7°N.", f"1 pixel = {T.PIXEL_KM2} km²"),
        ("Clipped to South Sudan", "Flood-mask tiles also cover Sudan, Ethiopia and CAR.", "point-in-polygon on the admin-0 boundary"),
        ("Old Lake Victoria / Kyoga files not used after 2002",
         "The course files stop in 2002; replaced by NASA G-REALM (1992–2026) in the extended analysis.",
         f"last Victoria observation in course file: {d['lake_victoria'].dropna().index.max().date()}"),
        ("Lake levels: weekly median, gaps filled forward only (max 26 weeks)",
         "Altimetry passes are irregular; forward filling never uses future values (no look-ahead).",
         f"Lake Albert: {len(alb)} observations, median gap {gaps.median():.0f} days, longest gap {gaps.max():.0f} days"),
        ("Malakal discharge kept only as a local feature, flagged",
         "Satellite-derived discharge drops during the biggest floods (saturation artefact).",
         f"yearly mean m³/s: 2019 = {q.get(2019):.0f}, 2022 = {q.get(2022):.0f}, 2023 = {q.get(2023):.0f} "
         f"(while flooding peaked)"),
        ("DFO station 1505 not counted as 'upstream'",
         "It lies at 4.9°N 36.0°E in the Omo–Turkana basin, which does not drain to the Nile.", "station metadata"),
        ("ERA5 hourly accumulations summed to daily totals", "ERA5 'tp' and 'ro' are hourly accumulations in metres.",
         f"South Sudan mean rainfall {pre['tp_ssd'].mean() * 365:.0f} mm/yr (2003–19), plausible"),
        ("Anomalies relative to the week-of-year climatology of 2003–2019",
         "Removes the seasonal cycle; 2003–2019 is the 'normal regime' reference.", "±2-week smoothed climatology"),
        ("Extreme week = unusual area above the 2003–2019 90th percentile", "Defines 'extreme' relative to the normal regime.",
         f"threshold = {w['unusual_km2_ssd'][(w.index.year >= 2003) & (w.index.year <= 2019)].quantile(0.9):.0f} km²"),
        ("Recurring and unusual pixels counted separately", "A pixel is classified once per day by the product.",
         f"days with both types present: {100 * flood_both:.0f}%"),
    ]
    out = pd.DataFrame(rows, columns=["decision", "why", "evidence_from_data"])
    out.to_csv(OUT_T / "eda_cleaning_decisions.csv", index=False)
    return out


def fig_coverage(d, raw_flood, grealm):
    series = [("Flood masks (MODIS/VIIRS)", raw_flood.index),
              ("ERA5 rain & runoff", d["tp_ssd"].dropna().index),
              ("AgERA5 evapotranspiration", d["et_ssd"].dropna().index),
              ("DFO discharge", d["q_malakal"].dropna().index),
              ("Lake Albert (DAHITI)", d["lake_albert"].dropna().index),
              ("Lake Victoria (course file)", d["lake_victoria"].dropna().index)]
    if grealm is not None:
        series.append(("Lake Victoria (G-REALM)", grealm["victoria"].dropna().index))
    fig, ax = plt.subplots(figsize=(11, 4.2))
    for i, (lab, idx) in enumerate(series):
        yrs = pd.Series(1, index=idx).groupby(idx.year).size()
        for y, n in yrs.items():   # shade = how complete that year is
            ax.barh(i, 1, left=y - 0.5, height=0.6, color=C["blue"], alpha=float(np.clip(n / yrs.max(), 0.25, 1)),
                    edgecolor="none")
    ax.axvspan(T.REF[0] - 0.5, 2025.5, color=C["aqua"], alpha=0.08, lw=0)
    ax.set_yticks(range(len(series)), [s[0] for s in series])
    ax.invert_yaxis()
    ax.set_xlim(1991.5, 2026.5)
    ax.set_title("Data coverage per source (shade = observations per year; band = study period 2003–2025)")
    ax.grid(axis="y", visible=False)
    fig.tight_layout()
    fig.savefig(OUT_F / "eda_01_data_coverage.png", dpi=160)
    plt.close(fig)


def fig_seasonality(w):
    woy = w.index.isocalendar().week.clip(upper=52)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.2), sharey=True)
    for ax, (lab, (a, b), c) in zip(axes, [("2003–2019 (normal regime)", (2003, 2019), C["blue"]),
                                           ("2020–2025", (2020, 2025), C["orange"])]):
        m = (w.index.year >= a) & (w.index.year <= b)
        for col, cc, name in [("recurring_km2_ssd", C["blue"], "recurring (normal)"), ("unusual_km2_ssd", C["orange"], "unusual")]:
            g = w.loc[m, col].groupby(woy[m])
            ax.plot(g.median().index, g.median(), color=cc, label=f"{name}, median")
            ax.fill_between(g.median().index, g.quantile(0.25), g.quantile(0.75), color=cc, alpha=0.15, lw=0)
        ax.set_title(lab)
        ax.set_xlabel("Week of year")
        ax.set_yscale("log")
    axes[0].set_ylabel("Weekly flood area, km² (log)")
    axes[0].legend(fontsize=9)
    fig.suptitle("Yearly flood cycle: median and interquartile range", x=0.01, ha="left", fontweight="bold")
    fig.tight_layout()
    fig.savefig(OUT_F / "eda_02_flood_seasonality.png", dpi=160)
    plt.close(fig)


def fig_threshold(w, thr):
    m = (w.index >= T.START) & (w.index <= T.END)
    v = w.loc[m, "unusual_km2_ssd"]
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
    ax = axes[0]
    bins = np.logspace(-1, np.log10(v.max() + 1), 50)
    ref = (w.index.year <= 2019)[m]
    ax.hist(v[ref] + 0.1, bins=bins, color=C["blue"], alpha=0.7, label="2003–2019")
    ax.hist(v[~ref] + 0.1, bins=bins, color=C["orange"], alpha=0.6, label="2020–2025")
    ax.axvline(thr, color=C["ink"], ls="--", lw=1.5)
    ax.text(thr * 1.1, ax.get_ylim()[1] * 0.9, f"extreme threshold\n{thr:.0f} km²", fontsize=9)
    ax.set_xscale("log")
    ax.set_xlabel("Weekly unusual flood area, km² (log)")
    ax.set_ylabel("Number of weeks")
    ax.legend(fontsize=9)
    ax.set_title("Distribution of weekly unusual flood area")
    ax = axes[1]
    share = w.loc[m, "extreme"].groupby(w.index[m].year).mean()
    ax.bar(share.index, share.values, color=[C["orange"] if y >= 2020 else C["blue"] for y in share.index])
    ax.set_ylabel("Share of extreme weeks")
    ax.set_title("Extreme weeks per year")
    fig.tight_layout()
    fig.savefig(OUT_F / "eda_03_extreme_threshold.png", dpi=160)
    plt.close(fig)


def fig_correlations(w, f):
    cols = {"albert_anom": "Lake Albert level", "albert_d26": "Lake Albert 26-wk rise", "tp_lakes_a52": "Lake-basin rain 52 wk",
            "ro_lakes_a52": "Lake-basin runoff 52 wk", "tp_sobat_a13": "Sobat rain 13 wk", "q_ethiopia_anom": "SW Ethiopia discharge",
            "tp_ssd_a4": "SSD rain 4 wk", "tp_ssd_a26": "SSD rain 26 wk", "ro_ssd_a13": "SSD runoff 13 wk",
            "wb_ssd_a26": "SSD rain − ET 26 wk", "et_ssd_a4": "SSD ET 4 wk", "q_malakal_anom": "Malakal discharge",
            "log_unusual": "log unusual flood area"}
    m = (f.index >= T.START) & (f.index <= T.END)
    cm = f.loc[m, list(cols)].rename(columns=cols).corr(method="spearman")
    cm.to_csv(OUT_T / "eda_driver_correlations.csv")
    from matplotlib.colors import LinearSegmentedColormap
    cmap = LinearSegmentedColormap.from_list("div", [C["blue"], "#f0efec", C["red"]])
    fig, ax = plt.subplots(figsize=(9.5, 8))
    im = ax.imshow(cm.values, cmap=cmap, vmin=-1, vmax=1)
    ax.set_xticks(range(len(cm)), cm.columns, rotation=55, ha="right", fontsize=9)
    ax.set_yticks(range(len(cm)), cm.index, fontsize=9)
    for i in range(len(cm)):
        for j in range(len(cm)):
            ax.text(j, i, f"{cm.values[i, j]:.1f}", ha="center", va="center", fontsize=7.5,
                    color="white" if abs(cm.values[i, j]) > 0.6 else C["ink"])
    ax.grid(False)
    fig.colorbar(im, ax=ax, shrink=0.8, label="Spearman r (weekly, 2003–2025)")
    ax.set_title("Correlations between driver anomalies (near-duplicates are not used together in Q1)")
    fig.tight_layout()
    fig.savefig(OUT_F / "eda_04_driver_correlations.png", dpi=160)
    plt.close(fig)


def fig_quality(d, raw_flood, w):
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    ax = axes[0]
    days = raw_flood.groupby(raw_flood.index.year).size().loc[2000:2025]
    ax.bar(days.index, days.values, color=[C["muted"] if y < 2003 else C["blue"] for y in days.index])
    ax.set_title("Flood masks: days with detections")
    ax.set_ylabel("Days per year")
    ax = axes[1]
    gaps = d["lake_albert"].dropna().index.to_series().diff().dt.days.dropna()
    ax.hist(gaps.clip(upper=120), bins=30, color=C["violet"])
    ax.set_title("Lake Albert altimetry: days between passes")
    ax.set_xlabel("Gap (days, capped at 120)")
    ax = axes[2]
    m = (w.index >= "2015-01-01") & (w.index <= T.END)
    ax.plot(w.index[m], w["q_malakal"][m], color=C["ink2"], lw=1, label="Malakal discharge (m³/s)")
    ax.set_ylabel("m³/s")
    hi = w["unusual_km2_sudd"][m] > w["unusual_km2_sudd"][m].quantile(0.75)
    for t in w.index[m][hi]:
        ax.axvspan(t, t + pd.Timedelta(days=7), color=C["orange"], alpha=0.15, lw=0)
    ax.set_title("Malakal discharge falls while the Sudd floods\n(orange = top-25% Sudd flood weeks)")
    fig.tight_layout()
    fig.savefig(OUT_F / "eda_05_data_quality.png", dpi=160)
    plt.close(fig)


def main():
    d, raw_flood, grealm = load()
    w, f, thr = T.build_weekly()
    pd.set_option("display.width", 220)
    inv = inventory(d, raw_flood, grealm)
    print(inv.to_string(index=False))
    cl = cleaning(d, raw_flood, w)
    print("\n" + cl.to_string(index=False))
    fig_coverage(d, raw_flood, grealm)
    fig_seasonality(w)
    fig_threshold(w, thr)
    fig_correlations(w, f)
    fig_quality(d, raw_flood, w)
    print("\nEDA figures and tables written")

In [ ]:
run_step(eda.main, ["outputs/tables/eda_data_inventory.csv", "outputs/tables/eda_cleaning_decisions.csv",
                    "outputs/figures/eda_05_data_quality.png"])
display(Markdown("**Data inventory** (observations and % of days missing in 2003–2025)"))
display(pd.read_csv(TAB / "eda_data_inventory.csv"))
display(Markdown("**Cleaning decisions and the evidence behind them**"))
with pd.option_context("display.max_colwidth", None):
    display(pd.read_csv(TAB / "eda_cleaning_decisions.csv"))

**Coverage.** Flood masks are sparse before 2003, so the study period is 2003–2025. The old Lake Victoria file stops in
2002; the NASA G-REALM record (step 1) covers 1992–2026.

In [ ]:
show(FIG / "eda_01_data_coverage.png")

**Flood cycle and target.** Before 2020 unusual flooding rises and falls with the seasons; after 2020 it stays high
all year. The extreme threshold (90th percentile of 2003–2019) marks 0–29 % of weeks per year before 2020 and most
weeks after.

In [ ]:
show(FIG / "eda_02_flood_seasonality.png", FIG / "eda_03_extreme_threshold.png")

**Drivers and data quality.** Near-duplicate drivers (e.g. rain vs rain − evaporation, r ≈ 0.99) are not used together
in the Q1 model. Lake Albert passes are irregular (gaps filled forward only). Malakal discharge falls while the Sudd
floods — a satellite artefact, so it is only used as a flagged local feature.

In [ ]:
show(FIG / "eda_04_driver_correlations.png", width=700)
show(FIG / "eda_05_data_quality.png")

## 3 · Train and evaluate the GRU and benchmarks (~45 min)
Target: an *extreme week* = unusual flood area above the 2003–2019 90th percentile (408 km²). A GRU reads 52 weeks of
history and predicts P(extreme) 2–26 weeks ahead; compared with climatology, persistence, logistic regression,
gradient boosting and an ensemble, with leave-one-year-out validation and a forward test (train ≤ 2018).

In [ ]:
run_step(train_models.main, ["outputs/tables/skill_leave_one_year_out.csv", "outputs/tables/skill_forward_test_2019_2025.csv",
                             "outputs/tables/oof_predictions_leave_one_year_out.parquet", "outputs/models/gru_all_features.pt"])
loyo = pd.read_csv(TAB / "skill_leave_one_year_out.csv")
display(loyo.pivot(index="model", columns="horizon", values="auc").round(3))
show(FIG / "01_flood_regime_and_upstream_timeline.png", FIG / "04a_skill_by_lead_loyo_models.png",
     FIG / "06_forward_test_2019_2025.png")

## 4 · Is the limit the model or the data? Physically constrained model (~1 min)



<details><summary>About this code</summary>

```
Research question 12.1 — Step 3: a simple, physically constrained logistic regression.

Same target, horizons and validation as train_models.py, but only a handful of physical features:
current flood state, seasonality, Lake Albert anomaly, Lake Albert 26-week rise and Sobat rain.
The Lake Albert coefficients are forced to be >= 0 (more water upstream can only raise flood risk).
An unconstrained version with the same features is fitted as well, to isolate the effect of the constraint.

Question: does this model anticipate the 2022-25 extremes in the forward test (train <=2018), where the GRU failed?
```
</details>

In [ ]:
%%module physical_model
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from scipy.special import expit

import train_models as T

FEATURES = ["log_unusual", "log_recurring", "unusual_share", "woy_sin", "woy_cos",
            "albert_anom", "albert_d26", "tp_sobat_a13"]
POSITIVE = ["albert_anom", "albert_d26"]
L2 = 1.0  # same strength as sklearn's C=1.0 used for the persistence baseline


class ConstrainedLogit:
    """L2-penalised logistic regression with optional lower bound 0 on selected coefficients."""

    def __init__(self, positive=()):
        self.positive = positive

    def fit(self, X, y):
        self.mu, self.sd = X.mean(0), X.std(0) + 1e-9   # standardising keeps coefficient signs
        Z = (X - self.mu) / self.sd
        n, p = Z.shape

        def loss(b):
            z = b[0] + Z @ b[1:]
            pr = expit(z)
            nll = np.sum(np.logaddexp(0, z) - y * z)
            g = pr - y
            return nll + 0.5 * L2 * b[1:] @ b[1:], np.r_[g.sum(), Z.T @ g + L2 * b[1:]]

        bounds = [(None, None)] + [(0, None) if c in self.positive else (None, None) for c in FEATURES]
        self.b = minimize(loss, np.zeros(p + 1), jac=True, method="L-BFGS-B", bounds=bounds).x
        return self

    def predict_proba(self, X):
        return expit(self.b[0] + ((X - self.mu) / self.sd) @ self.b[1:])


def fit_predict(f, y, tr, te, positive):
    X = f[FEATURES].fillna(0).values
    P = np.zeros((te.sum(), len(T.HORIZONS)))
    coefs = {}
    for k, h in enumerate(T.HORIZONS):
        ok = tr & ~np.isnan(y[:, k])
        mdl = ConstrainedLogit(positive).fit(X[ok], y[ok, k])
        P[:, k] = mdl.predict_proba(X[te])
        coefs[h] = dict(zip(FEATURES, mdl.b[1:]))
    return P, pd.DataFrame(coefs)


def skill(y, P, clim_brier):
    rows = []
    for k, h in enumerate(T.HORIZONS):
        s = T.score(y[:, k], P[:, k])
        rows.append(dict(horizon=h, auc=s["auc"], brier=s["brier"], bss=1 - s["brier"] / clim_brier[h]))
    return pd.DataFrame(rows)


def main():
    w, f, _ = T.build_weekly()
    y = T.make_targets(w)
    dates = w.index
    m = (dates >= T.START) & (dates <= T.END)
    valid = m & (np.arange(len(w)) >= T.LOOKBACK - 1) & ~np.all(np.isnan(y), axis=1)
    models = {"Physical logistic – Albert ≥ 0": POSITIVE, "Physical logistic – unconstrained": ()}

    # ---- Forward test, identical split to train_models.py ----
    tgt_end = dates + pd.Timedelta(weeks=max(T.HORIZONS))
    tr = valid & (tgt_end <= pd.Timestamp("2018-12-31"))
    te = valid & (dates >= pd.Timestamp("2019-01-01"))
    fwd_prev = pd.read_csv(T.OUT_T / "skill_forward_test_2019_2025.csv")
    clim_b = fwd_prev[fwd_prev.model == "Climatology"].set_index("horizon")["brier"]

    fwd, preds = [], {}
    for name, pos in models.items():
        P, coefs = fit_predict(f, y, tr, te, pos)
        preds[name] = P
        fwd.append(skill(y[te], P, clim_b).assign(model=name))
        coefs.to_csv(T.OUT_T / f"physical_logit_coefficients_forward_{'constrained' if pos else 'free'}.csv")
        print(f"\n{name} — standardised coefficients (forward fit, train <=2018):\n", coefs.round(2).to_string())
    fwd = pd.concat(fwd)
    keep = ["Persistence (logistic, flood state + season)", "GRU – all features", T.ENSEMBLE]
    comp = pd.concat([fwd_prev[fwd_prev.model.isin(keep)][["model", "horizon", "auc", "brier", "bss"]], fwd])
    comp.to_csv(T.OUT_T / "skill_forward_test_physical_vs_others.csv", index=False)
    print("\nForward test 2019-2025:\n", comp.pivot(index="model", columns="horizon", values="bss").round(3).to_string())
    print("\n", comp.pivot(index="model", columns="horizon", values="auc").round(3).to_string())

    # Skill on the persistent 2022-25 extremes only (the part the GRU missed)
    late = dates[te] >= pd.Timestamp("2022-01-01")
    rows = []
    for name, P in preds.items():
        for k, h in enumerate(T.HORIZONS):
            yy = y[te][late, k]
            ok = ~np.isnan(yy)
            rows.append(dict(model=name, horizon=h, hit_rate_2022_25=np.mean(P[late][ok, k][yy[ok] == 1] > 0.5)))
    print("\nShare of 2022-25 extreme weeks forecast with P > 0.5:\n",
          pd.DataFrame(rows).pivot(index="model", columns="horizon", values="hit_rate_2022_25").round(2).to_string())

    # ---- Leave-one-year-out, for comparison with the main table ----
    loyo_prev = pd.read_csv(T.OUT_T / "skill_leave_one_year_out.csv")
    clim_l = loyo_prev[loyo_prev.model == "Climatology"].set_index("horizon")["brier"]
    loyo = []
    for name, pos in models.items():
        oof = np.full((len(w), len(T.HORIZONS)), np.nan)
        for _, tr_, te_ in T.make_folds(dates, valid):
            oof[te_] = fit_predict(f, y, tr_, te_, pos)[0]
        loyo.append(skill(y[valid], oof[valid], clim_l).assign(model=name))
    loyo = pd.concat(loyo)
    loyo.to_csv(T.OUT_T / "skill_loyo_physical_logit.csv", index=False)
    print("\nLeave-one-year-out:\n", loyo.pivot(index="model", columns="horizon", values="bss").round(3).to_string())

    fig_forward(w, te, y, preds)


def fig_forward(w, te, y, preds):
    dates = w.index[te]
    fig, axes = plt.subplots(2, 1, figsize=(11, 6.4), sharex=True)
    for ax, k in zip(axes, [2, 3]):
        h = T.HORIZONS[k]
        td = dates + pd.Timedelta(weeks=h)
        for i, (s, e) in enumerate(T._runs(np.nan_to_num(y[te][:, k]).astype(int))):
            ax.axvspan(td[s], td[e], color=T.C["red"], alpha=0.08, lw=0, label="Observed extreme week" if i == 0 else None)
        ax.plot(td, preds["Physical logistic – Albert ≥ 0"][:, k], color=T.C["violet"], lw=1.6,
                label="Physical logistic – Albert ≥ 0")
        ax.plot(td, preds["Physical logistic – unconstrained"][:, k], color=T.C["aqua"], lw=1.4,
                label="Physical logistic – unconstrained")
        ax.axhline(0.5, color="#c3c2b7", lw=1, ls="--")
        ax.set_ylim(-0.02, 1.02)
        ax.set_ylabel("P(extreme)")
        ax.set_title(f"Forward test — trained on 2003–2018 only — forecast issued {h} weeks ahead")
        ax.legend(loc="upper left", fontsize=8, ncol=3)
    fig.tight_layout()
    fig.savefig(T.OUT_F / "09_forward_test_physical_logit.png", dpi=170)
    plt.close(fig)

In [ ]:
run_step(physical_model.main, ["outputs/tables/skill_forward_test_physical_vs_others.csv"])
fp = pd.read_csv(TAB / "skill_forward_test_physical_vs_others.csv")
display(fp.pivot(index="model", columns="horizon", values="bss").round(3))
show(FIG / "09_forward_test_physical_logit.png")

## 5 · Robustness: confidence intervals, surrogate test, spatial control, target sensitivity (~2 min)



<details><summary>About this code</summary>

```
Research question 12.1 — Step 4: robustness, uncertainty and causal checks.

Answers four weaknesses of the main analysis:
  A. Uncertainty      — 95 % confidence intervals from a block bootstrap that resamples whole years
                        (driver AUCs, model skill, differences between models, onset composites).
  B. n = 1 problem    — phase-randomised surrogates of Lake Albert (same autocorrelation, random timing),
                        a time-trend placebo, and the within-regime (2020-25) association.
  C. Causality        — a stated causal diagram (DAG) and three tests it implies:
                        1. spatial negative control: Bahr el Ghazal (west) is not downstream of Lake Albert
                        2. the competing path "regional rain -> local rain -> floods" is closed
                        3. mediation: lake-basin rain is associated with floods only *through* Lake Albert
  D. Target choice    — repeat the key results with targets that do not use the recurring/unusual label.
```
</details>

In [ ]:
%%module robustness
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import rankdata, spearmanr
from shapely import contains_xy
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler

import train_models as T

BASE = T.HERE.parent
B = 1000
RNG = np.random.default_rng(42)
C = T.C

# Flood regions within South Sudan, as (lat_min, lat_max, lon_min, lon_max)
FLOOD_REGIONS = {
    "west": (4.0, 10.5, 24.0, 29.5),   # Bahr el Ghazal: rain-fed, NOT downstream of Lake Albert (negative control)
    "sudd": (6.0, 10.0, 29.5, 32.5),   # Bahr el Jebel / White Nile corridor: directly downstream of Lake Albert
    "east": (5.5, 10.5, 32.5, 36.0),   # Sobat / Pibor / eastern Upper Nile: fed mainly by the Ethiopian highlands
}
REGION_LABEL = {"ssd": "South Sudan (main target)", "sudd": "Sudd / White Nile corridor",
                "east": "East: Sobat–Pibor", "west": "West: Bahr el Ghazal (negative control)"}


# --------------------------------------------------------------------------------------
# Data
# --------------------------------------------------------------------------------------
def regional_flood():
    p = T.HERE / "data" / "_flood_regions.parquet"
    if p.exists():
        return pd.read_parquet(p)
    ssd = gpd.read_file(BASE / "Administrative boundaries" / "ssd_admin0.geojson").geometry.union_all()
    out = []
    for typ in ["recurring", "unusual"]:
        for year in range(2002, 2026):
            d = pd.concat([pd.read_parquet(BASE / "flood_masks" / f"compact_{typ}" / f"flood_events_{t}_{year}.parquet",
                                           columns=["date", "lat", "lon"]) for t in ["h20v08", "h21v08"]])
            d = d[contains_xy(ssd, d["lon"].values, d["lat"].values)]
            g = {}
            for r, (a, b, c, e) in FLOOD_REGIONS.items():
                sel = d["lat"].between(a, b) & d["lon"].between(c, e)
                g[f"{typ}_px_{r}"] = d[sel].groupby("date").size()
            out.append(pd.DataFrame(g))
        print(f"  regional flood extraction: {typ} done")
    df = pd.concat(out).groupby(level=0).sum()
    df.index = pd.to_datetime(df.index)
    df.to_parquet(p)
    return df


def weekly_targets(w):
    """Main target plus alternatives that do not depend on the recurring/unusual label or on region."""
    rf = regional_flood().resample("W-SUN").max().reindex(w.index).fillna(0) * T.PIXEL_KM2
    ref = (w.index.year >= T.REF[0]) & (w.index.year <= T.REF[1])
    t = pd.DataFrame(index=w.index)
    t["main"] = w["extreme"]
    total = w["unusual_km2_ssd"] + w["recurring_km2_ssd"]
    t["total_extent"] = (total > total[ref].quantile(0.9)).astype(int)
    month_thr = total[ref].groupby(total.index[ref].month).quantile(0.9)
    t["total_extent_same_month"] = (total > total.index.month.map(month_thr).values).astype(int)
    for r in FLOOD_REGIONS:
        u = rf[f"unusual_px_{r}"]
        t[f"region_{r}"] = (u > u[ref].quantile(0.9)).astype(int)
    return t, rf


# --------------------------------------------------------------------------------------
# Helpers
# --------------------------------------------------------------------------------------
def fast_auc(y, s):
    y = np.asarray(y, bool)
    n1, n0 = y.sum(), (~y).sum()
    if n1 == 0 or n0 == 0:
        return np.nan
    r = rankdata(s)
    return (r[y].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)


def year_blocks(index):
    years = np.asarray(index.year)
    return {y: np.where(years == y)[0] for y in np.unique(years)}


def boot_ci(stat, index, b=B):
    """Point estimate and 95 % CI of stat(idx) under a bootstrap that resamples whole calendar years."""
    blocks = year_blocks(index)
    keys = list(blocks)
    est = stat(np.arange(len(index)))
    vals = []
    for _ in range(b):
        idx = np.concatenate([blocks[k] for k in RNG.choice(keys, len(keys))])
        vals.append(stat(idx))
    lo, hi = np.nanpercentile(vals, [2.5, 97.5])
    return est, lo, hi


def deseason(s):
    return T.woy_anomaly(s, np.ones(len(s), bool))


# --------------------------------------------------------------------------------------
# A. Uncertainty of driver AUCs and model skill
# --------------------------------------------------------------------------------------
KEY_DRIVERS = {"Lake Albert level anomaly": "albert_anom", "Equatorial-lakes runoff, 52-wk": "ro_lakes_a52",
               "Sobat/Ethiopia rain, 13-wk": "tp_sobat_a13", "South Sudan rain, 26-wk": "tp_ssd_a26",
               "South Sudan rain, 4-wk": "tp_ssd_a4"}


def driver_auc_ci(f, target, period, horizons=(0, 13, 26)):
    a, b = period
    rows = []
    for name, col in KEY_DRIVERS.items():
        for h in horizons:
            y = target.shift(-h)
            m = (f.index.year >= a) & (f.index.year <= b) & y.notna() & f[col].notna()
            yy, xx, idx = y[m].values, f[col][m].values, f.index[m]
            est, lo, hi = boot_ci(lambda i: fast_auc(yy[i], xx[i]), idx)
            rows.append(dict(driver=name, horizon=h, period=f"{a}–{b}", auc=est, ci_low=lo, ci_high=hi))
    return pd.DataFrame(rows)


def model_skill_ci():
    oof = pd.read_parquet(T.OUT_T / "oof_predictions_leave_one_year_out.parquet")
    models = ["Persistence (logistic, flood state + season)", "Logistic regression – all features",
              "Gradient boosting – all features", "GRU – all features", T.ENSEMBLE,
              "GRU – without upstream", "GRU – upstream + season only"]
    contrasts = {"Ensemble − Persistence": (T.ENSEMBLE, "Persistence (logistic, flood state + season)"),
                 "GRU all − GRU without upstream": ("GRU – all features", "GRU – without upstream"),
                 "GRU all − Persistence": ("GRU – all features", "Persistence (logistic, flood state + season)")}
    rows = []
    for h in T.HORIZONS:
        P = {m: oof.loc[m, f"p_{h}wk"] for m in models + ["Climatology"]}
        y = oof.loc["Climatology", f"y_{h}wk"]
        ok = y.notna().values
        y = y.values[ok].astype(int)
        P = {m: v.values[ok] for m, v in P.items()}
        idx = oof.loc["Climatology"].index[ok]

        def bss(m, i):
            return 1 - np.mean((P[m][i] - y[i]) ** 2) / np.mean((P["Climatology"][i] - y[i]) ** 2)

        for m in models:
            for metric, fn in [("BSS", lambda i, m=m: bss(m, i)), ("AUC", lambda i, m=m: fast_auc(y[i], P[m][i]))]:
                est, lo, hi = boot_ci(fn, idx)
                rows.append(dict(item=m, kind="model", metric=metric, horizon=h, est=est, ci_low=lo, ci_high=hi))
        for cname, (m1, m2) in contrasts.items():
            est, lo, hi = boot_ci(lambda i: bss(m1, i) - bss(m2, i), idx)
            rows.append(dict(item=cname, kind="difference", metric="BSS", horizon=h, est=est, ci_low=lo, ci_high=hi))
    return pd.DataFrame(rows)


# --------------------------------------------------------------------------------------
# B. The n = 1 problem
# --------------------------------------------------------------------------------------
def phase_surrogates(x, n):
    """Surrogates with the same power spectrum (autocorrelation) as x but random phases, i.e. random timing."""
    x = np.asarray(x, float)
    mu = x.mean()
    X = np.fft.rfft(x - mu)
    out = np.empty((n, len(x)))
    for k in range(n):
        ph = RNG.uniform(0, 2 * np.pi, len(X))
        ph[0] = 0
        if len(x) % 2 == 0:
            ph[-1] = 0
        out[k] = np.fft.irfft(np.abs(X) * np.exp(1j * ph), n=len(x)) + mu
    return out


def n1_tests(w, f, target):
    m = (f.index >= T.START) & (f.index <= T.END)
    alb = f["albert_anom"].ffill().bfill()[m]
    rows = []
    sur = phase_surrogates(alb.values, 2000)
    for h in [0, 13, 26]:
        y = target.shift(-h)[m]
        ok = y.notna().values
        obs = fast_auc(y.values[ok], alb.values[ok])
        s_auc = np.array([fast_auc(y.values[ok], s[ok]) for s in sur])
        trend = fast_auc(y.values[ok], np.arange(ok.sum()))
        rows.append(dict(horizon=h, albert_auc=obs, surrogate_p=(np.sum(s_auc >= obs) + 1) / (len(s_auc) + 1),
                         surrogate_auc_95pct=np.percentile(s_auc, 95), time_trend_placebo_auc=trend))
        if h == 0:
            sur_h0 = s_auc
    n1 = pd.DataFrame(rows)

    # Within-regime association (2020-25) and pre-shift (2003-19), de-seasonalised log extent
    ext = deseason(np.log1p(w["unusual_km2_ssd"] + w["recurring_km2_ssd"]))
    cum = f["albert_anom"].ffill().rolling(104, min_periods=52).mean()   # excess level over the past ~2 years
    within = []
    for label, (a, b) in {"2003–2019": (2003, 2019), "2020–2025": (2020, 2025)}.items():
        for xname, x in [("Lake Albert anomaly (now)", f["albert_anom"]),
                         ("Lake Albert anomaly, 2-yr mean", cum)]:
            mm = (w.index.year >= a) & (w.index.year <= b) & x.notna() & ext.notna()
            xx, ee, idx = x[mm].values, ext[mm].values, w.index[mm]
            est, lo, hi = boot_ci(lambda i: spearmanr(xx[i], ee[i]).statistic, idx)
            within.append(dict(period=label, driver=xname, spearman_r=est, ci_low=lo, ci_high=hi))
    return n1, pd.DataFrame(within), sur_h0


# --------------------------------------------------------------------------------------
# C. Causal checks
# --------------------------------------------------------------------------------------
def spatial_control(f, targets):
    rows = []
    for r in ["sudd", "east", "west"]:
        y = targets[f"region_{r}"]
        for name, col in [("Lake Albert level anomaly", "albert_anom"), ("Sobat/Ethiopia rain, 13-wk", "tp_sobat_a13"),
                          ("South Sudan rain, 26-wk", "tp_ssd_a26")]:
            for h in [0, 26]:
                yy = y.shift(-h)
                m = (f.index >= T.START) & (f.index <= T.END) & yy.notna() & f[col].notna()
                Y, X, idx = yy[m].values, f[col][m].values, f.index[m]
                est, lo, hi = boot_ci(lambda i: fast_auc(Y[i], X[i]), idx)
                rows.append(dict(region=REGION_LABEL[r], driver=name, horizon=h, auc=est, ci_low=lo, ci_high=hi))
    return pd.DataFrame(rows)


def spatial_magnitude(rf):
    """Growth of mean weekly flood extent, 2021-25 vs 2003-19, per region (with year-block bootstrap CI)."""
    rf = rf[(rf.index >= T.START) & (rf.index <= T.END)]
    late = rf.index.year >= 2021
    rows = []
    for r in ["sudd", "east", "west"]:
        for kind in ["unusual", "total"]:
            s = rf[f"unusual_px_{r}"] + (rf[f"recurring_px_{r}"] if kind == "total" else 0)
            v, lt = s.values, late

            def ratio(i):
                a, b = v[i][~lt[i]], v[i][lt[i]]
                return b.mean() / a.mean() if len(a) and len(b) else np.nan

            est, lo, hi = boot_ci(ratio, rf.index)
            rows.append(dict(region=REGION_LABEL[r], extent=kind, ratio_2021_25_vs_2003_19=est, ci_low=lo, ci_high=hi))
    return pd.DataFrame(rows)


def local_rain_path(w, f, target):
    """If the 2020s extremes came from local rain (the competing path), local rain should be above normal then."""
    rows = []
    for label, (a, b) in {"2003–2019": (2003, 2019), "2020–2025": (2020, 2025)}.items():
        for name, col in [("South Sudan rain, 26-wk", "tp_ssd_a26"), ("Sudd rain, 13-wk", "tp_sudd_a13"),
                          ("South Sudan P–ET, 26-wk", "wb_ssd_a26")]:
            m = (w.index.year >= a) & (w.index.year <= b) & (target == 1) & f[col].notna()
            x, idx = f[col][m].values, w.index[m]
            est, lo, hi = boot_ci(lambda i: np.mean(x[i]), idx)
            rows.append(dict(period=label, driver=name, mean_anomaly_in_extreme_weeks=est, ci_low=lo, ci_high=hi))
    return pd.DataFrame(rows)


def mediation(w, f):
    """Partial Spearman correlations on monthly de-seasonalised series (DAG: lake rain -> Albert -> floods)."""
    ext = deseason(np.log1p(w["unusual_km2_sudd"] + w["recurring_km2_sudd"]))
    df = pd.DataFrame({"flood": ext, "albert": f["albert_anom"], "rain": f["ro_lakes_a52"]})
    df = df[(df.index >= T.START) & (df.index <= T.END)].resample("MS").mean().dropna()
    R = df.rank()

    def pcorr(x, y, z, i):
        d = R.iloc[i]
        rx = d[x] - np.polyval(np.polyfit(d[z], d[x], 1), d[z])
        ry = d[y] - np.polyval(np.polyfit(d[z], d[y], 1), d[z])
        return np.corrcoef(rx, ry)[0, 1]

    rows = []
    for label, fn in [("Lake-basin runoff ↔ Sudd floods (total)", lambda i: R.iloc[i][["rain", "flood"]].corr().iloc[0, 1]),
                      ("Lake-basin runoff ↔ Sudd floods | Lake Albert", lambda i: pcorr("rain", "flood", "albert", i)),
                      ("Lake Albert ↔ Sudd floods (total)", lambda i: R.iloc[i][["albert", "flood"]].corr().iloc[0, 1]),
                      ("Lake Albert ↔ Sudd floods | lake-basin runoff", lambda i: pcorr("albert", "flood", "rain", i))]:
        est, lo, hi = boot_ci(fn, df.index)
        rows.append(dict(relation=label, partial_spearman=est, ci_low=lo, ci_high=hi))
    return pd.DataFrame(rows)


# --------------------------------------------------------------------------------------
# D. Target sensitivity (cheap models only: logistic regressions, leave-one-year-out)
# --------------------------------------------------------------------------------------
def target_sensitivity(w, f, targets):
    rows, shares = [], {}
    dates = w.index
    m = (dates >= T.START) & (dates <= T.END)
    for tname in ["main", "total_extent", "total_extent_same_month", "region_sudd", "region_west"]:
        t = targets[tname]
        shares[tname] = t[m].groupby(dates[m].year).mean()
        for h in [0, 26]:
            yy = t.shift(-h)
            mm = m & yy.notna()
            Y, X, idx = yy[mm].values, f["albert_anom"][mm].values, dates[mm]
            ok = ~np.isnan(X)
            est, lo, hi = boot_ci(lambda i: fast_auc(Y[ok][i], X[ok][i]), idx[ok])
            rows.append(dict(target=tname, metric=f"Albert AUC, lead {h} wk", est=est, ci_low=lo, ci_high=hi))
        # LOYO logistic regression (all features) vs persistence, 13-week lead
        y13 = t.shift(-13).values
        valid = m & ~np.isnan(y13)
        for mname, cols, Cc in [("Persistence", T.cols_for(["Current flood state", "Seasonality"]), 1.0),
                                ("Logistic – all features", T.cols_for(T.FEATURE_GROUPS), 0.1)]:
            Xa = f[cols].fillna(0).values
            oof = np.full(len(w), np.nan)
            for _, tr, te in T.make_folds(dates, valid):
                sc = StandardScaler().fit(Xa[tr])
                if len(np.unique(y13[tr])) < 2:
                    continue
                mdl = LogisticRegression(C=Cc, max_iter=5000).fit(sc.transform(Xa[tr]), y13[tr])
                oof[te] = mdl.predict_proba(sc.transform(Xa[te]))[:, 1]
            ok = valid & ~np.isnan(oof)
            rows.append(dict(target=tname, metric=f"{mname} AUC, lead 13 wk (LOYO)",
                             est=roc_auc_score(y13[ok], oof[ok]), ci_low=np.nan, ci_high=np.nan))
    return pd.DataFrame(rows), pd.DataFrame(shares)


# --------------------------------------------------------------------------------------
# Figures
# --------------------------------------------------------------------------------------
def fig_skill_ci(sk):
    fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.8))
    show = [("Persistence (logistic, flood state + season)", C["ink2"]), ("Logistic regression – all features", C["aqua"]),
            ("GRU – all features", C["blue"]), (T.ENSEMBLE, C["red"])]
    ax = axes[0]
    for j, (m, c) in enumerate(show):
        s = sk[(sk.item == m) & (sk.metric == "BSS")].sort_values("horizon")
        x = np.arange(len(T.HORIZONS)) + (j - 1.5) * 0.17
        ax.errorbar(x, s["est"], yerr=[s["est"] - s["ci_low"], s["ci_high"] - s["est"]], fmt="o", ms=5,
                    color=c, ecolor=c, elinewidth=1.6, capsize=0, label=m)
    ax.set_xticks(range(len(T.HORIZONS)), [f"{h} wk" for h in T.HORIZONS])
    ax.axhline(0, color="#c3c2b7", lw=1)
    ax.set_ylabel("Brier skill score vs. climatology")
    ax.set_title("Model skill with 95 % CI (year-block bootstrap)")
    ax.legend(fontsize=8, loc="lower left")
    ax = axes[1]
    show = [("Ensemble − Persistence", C["red"]), ("GRU all − GRU without upstream", C["violet"])]
    for j, (m, c) in enumerate(show):
        s = sk[(sk.item == m)].sort_values("horizon")
        x = np.arange(len(T.HORIZONS)) + (j - 0.5) * 0.2
        ax.errorbar(x, s["est"], yerr=[s["est"] - s["ci_low"], s["ci_high"] - s["est"]], fmt="o", ms=5,
                    color=c, ecolor=c, elinewidth=1.6, capsize=0, label=m)
    ax.axhline(0, color=C["ink2"], lw=1, ls="--")
    ax.set_xticks(range(len(T.HORIZONS)), [f"{h} wk" for h in T.HORIZONS])
    ax.set_ylabel("Difference in Brier skill score")
    ax.set_title("Is the difference real? (CI crossing 0 = not shown)")
    ax.legend(fontsize=8, loc="upper left")
    fig.tight_layout()
    fig.savefig(T.OUT_F / "10_model_skill_uncertainty.png", dpi=170)
    plt.close(fig)


def fig_surrogate(n1, sur_h0):
    fig, ax = plt.subplots(figsize=(8.5, 4))
    ax.hist(sur_h0, bins=40, color=C["muted"], alpha=0.6, edgecolor=C["surface"])
    obs = n1.loc[n1.horizon == 0, "albert_auc"].iloc[0]
    trend = n1.loc[n1.horizon == 0, "time_trend_placebo_auc"].iloc[0]
    p = n1.loc[n1.horizon == 0, "surrogate_p"].iloc[0]
    ax.axvline(obs, color=C["violet"], lw=2)
    ax.text(obs, ax.get_ylim()[1] * 0.92, f"  Lake Albert: AUC {obs:.2f}\n  p = {p:.3f}", color=C["ink"], fontsize=9, va="top")
    ax.axvline(trend, color=C["orange"], lw=2, ls="--")
    ax.text(trend, ax.get_ylim()[1] * 0.6, f"time-trend placebo {trend:.2f}  ", color=C["ink"], fontsize=9, ha="right")
    ax.set_xlabel("AUC for extreme weeks (lead 0)")
    ax.set_ylabel("Number of surrogate series")
    ax.set_title("Could a random series with Lake Albert's persistence line up with the floods by chance?")
    ax.grid(axis="x", visible=False)
    fig.tight_layout()
    fig.savefig(T.OUT_F / "11_surrogate_test_lake_albert.png", dpi=170)
    plt.close(fig)


def fig_spatial(targets, f, sp, mag):
    order = [REGION_LABEL["sudd"], REGION_LABEL["west"], REGION_LABEL["east"]]
    cols = {REGION_LABEL["sudd"]: C["blue"], REGION_LABEL["east"]: C["yellow"], REGION_LABEL["west"]: C["orange"]}
    ylab = ["Sudd / White Nile\n(downstream of Lake Albert)", "West: Bahr el Ghazal\n(not downstream)",
            "East: Sobat–Pibor\n(not downstream)"]
    fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.2), sharey=True)
    ax = axes[0]
    s = mag[mag.extent == "unusual"].set_index("region").loc[order]
    for j, (reg, r) in enumerate(s.iterrows()):
        ax.barh(j, r["ratio_2021_25_vs_2003_19"], color=cols[reg], height=0.55)
        ax.errorbar(r["ratio_2021_25_vs_2003_19"], j, xerr=[[r["ratio_2021_25_vs_2003_19"] - r["ci_low"]],
                    [r["ci_high"] - r["ratio_2021_25_vs_2003_19"]]], fmt="none", ecolor=C["ink2"], elinewidth=1.4)
        ax.text(r["ci_high"] * 1.15, j, f"{r['ratio_2021_25_vs_2003_19']:.1f}×", va="center", color=C["ink"], fontsize=9.5)
    ax.set_xscale("log")
    ax.axvline(1, color=C["ink2"], lw=1, ls="--")
    ax.set_yticks(range(3), ylab)
    ax.invert_yaxis()
    ax.set_xlabel("Unusual flood extent, 2021–25 ÷ 2003–19 (log, 95 % CI)")
    ax.set_title("Where did flooding grow?")
    ax.grid(axis="y", visible=False)
    ax = axes[1]
    s = sp[(sp.driver == "Lake Albert level anomaly") & (sp.horizon == 26)].set_index("region").loc[order]
    for j, (reg, r) in enumerate(s.iterrows()):
        ax.errorbar(r["auc"], j, xerr=[[r["auc"] - r["ci_low"]], [r["ci_high"] - r["auc"]]], fmt="o", ms=7,
                    color=cols[reg], elinewidth=1.8)
    ax.axvline(0.5, color=C["ink2"], lw=1, ls="--")
    ax.set_xlabel("AUC of Lake Albert anomaly for regional extremes, 26 weeks ahead (95 % CI)")
    ax.set_title("Does Lake Albert warn for each region?")
    ax.grid(axis="y", visible=False)
    fig.suptitle("Spatial control: the signal should appear only downstream of Lake Albert",
                 x=0.01, ha="left", fontweight="bold")
    fig.tight_layout()
    fig.savefig(T.OUT_F / "12_spatial_negative_control.png", dpi=170)
    plt.close(fig)


def fig_onsets_ci(w, f):
    ext = w["extreme"].values
    onsets = [i for i in range(26, len(ext) - 5) if ext[i] == 1 and ext[i - 8:i].sum() == 0
              and T.START <= str(w.index[i].date()) <= T.END]
    show = [("Lake Albert level anomaly (m)", "albert_anom", C["violet"]),
            ("Equatorial-lakes rain, 52-wk (σ)", "tp_lakes_a52", C["blue"]),
            ("Sobat/Ethiopia rain, 13-wk (σ)", "tp_sobat_a13", C["yellow"]),
            ("South Sudan rain, 4-wk (σ)", "tp_ssd_a4", C["orange"])]
    fig, axes = plt.subplots(1, 4, figsize=(14, 3.6), sharex=True)
    lags = np.arange(-26, 5)
    for ax, (name, col, c) in zip(axes, show):
        M = np.array([f[col].values[i - 26:i + 5] for i in onsets])
        mean = np.nanmean(M, 0)
        boots = np.array([np.nanmean(M[RNG.integers(0, len(M), len(M))], 0) for _ in range(B)])
        lo, hi = np.nanpercentile(boots, [2.5, 97.5], axis=0)
        ax.fill_between(lags, lo, hi, color=c, alpha=0.2, lw=0)
        ax.plot(lags, mean, color=c)
        ax.axvline(0, color=C["ink2"], lw=1, ls="--")
        ax.axhline(0, color="#c3c2b7", lw=1)
        ax.set_title(name, fontsize=9.5)
        ax.set_xlabel("Weeks to onset")
    fig.suptitle(f"Driver anomalies before extreme-flood onsets, mean and 95 % CI (n = {len(onsets)} onsets)",
                 x=0.01, ha="left", fontweight="bold")
    fig.tight_layout()
    fig.savefig(T.OUT_F / "03b_onset_composites_ci.png", dpi=170)
    plt.close(fig)


def fig_target_sensitivity(ts, shares):
    labels = {"main": "Main: unusual extent > P90", "total_extent": "Total extent > P90 (no label)",
              "total_extent_same_month": "Total extent > P90 of same month",
              "region_sudd": "Sudd only", "region_west": "West only (negative control)"}
    fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.4))
    ax = axes[0]
    cols = [C["ink2"], C["blue"], C["aqua"], C["violet"], C["orange"]]
    for (t, s), c in zip(shares.items(), cols):
        ax.plot(s.index, s.values, color=c, marker="o", ms=3, label=labels[t])
    ax.set_ylabel("Share of extreme weeks per year")
    ax.set_title("Does the regime shift depend on how 'extreme' is defined?")
    ax.legend(fontsize=8, loc="upper left")
    ax = axes[1]
    s = ts[ts.metric == "Albert AUC, lead 26 wk"].reset_index(drop=True)
    for j, r in s.iterrows():
        ax.errorbar(r["est"], j, xerr=[[r["est"] - r["ci_low"]], [r["ci_high"] - r["est"]]], fmt="o", ms=6,
                    color=cols[j], elinewidth=1.8)
    ax.set_yticks(range(len(s)), [labels[t] for t in s["target"]])
    ax.axvline(0.5, color=C["ink2"], lw=1, ls="--")
    ax.set_xlabel("AUC of Lake Albert anomaly, 26 weeks ahead (95 % CI)")
    ax.set_title("Upstream signal under each target")
    ax.grid(axis="y", visible=False)
    fig.tight_layout()
    fig.savefig(T.OUT_F / "13_target_sensitivity.png", dpi=170)
    plt.close(fig)


def main():
    w, f, _ = T.build_weekly()
    targets, rf = weekly_targets(w)
    pd.set_option("display.width", 200)

    print("\n=== A. Driver AUC with 95 % CI ===")
    auc = pd.concat([driver_auc_ci(f, targets["main"], p) for p in [(2003, 2025), (2003, 2019)]])
    auc.to_csv(T.OUT_T / "ci_driver_auc.csv", index=False)
    print(auc.round(2).to_string(index=False))

    print("\n=== A. Model skill with 95 % CI ===")
    sk = model_skill_ci()
    sk.to_csv(T.OUT_T / "ci_model_skill.csv", index=False)
    print(sk[sk.metric == "BSS"].pivot_table(index="item", columns="horizon", values=["est", "ci_low", "ci_high"])
          .round(2).to_string())
    fig_skill_ci(sk)
    fig_onsets_ci(w, f)

    print("\n=== B. n = 1: surrogate test and placebo ===")
    n1, within, sur_h0 = n1_tests(w, f, targets["main"])
    n1.to_csv(T.OUT_T / "n1_surrogate_and_placebo.csv", index=False)
    within.to_csv(T.OUT_T / "n1_within_regime_association.csv", index=False)
    print(n1.round(3).to_string(index=False))
    print(within.round(2).to_string(index=False))
    fig_surrogate(n1, sur_h0)

    print("\n=== C. Causal checks ===")
    sp = spatial_control(f, targets)
    sp.to_csv(T.OUT_T / "causal_spatial_negative_control.csv", index=False)
    print(sp.round(2).to_string(index=False))
    mag = spatial_magnitude(rf)
    mag.to_csv(T.OUT_T / "causal_spatial_magnitude.csv", index=False)
    print(mag.round(1).to_string(index=False))
    fig_spatial(targets, f, sp, mag)
    lr = local_rain_path(w, f, targets["main"])
    lr.to_csv(T.OUT_T / "causal_local_rain_path.csv", index=False)
    print(lr.round(2).to_string(index=False))
    med = mediation(w, f)
    med.to_csv(T.OUT_T / "causal_mediation.csv", index=False)
    print(med.round(2).to_string(index=False))

    print("\n=== D. Target sensitivity ===")
    ts, shares = target_sensitivity(w, f, targets)
    ts.to_csv(T.OUT_T / "target_sensitivity.csv", index=False)
    shares.to_csv(T.OUT_T / "target_sensitivity_yearly_share.csv")
    print(ts.round(2).to_string(index=False))
    print(shares.round(2).to_string())
    fig_target_sensitivity(ts, shares)

In [ ]:
run_step(robustness.main, ["outputs/tables/ci_model_skill.csv", "outputs/tables/causal_spatial_magnitude.csv",
                           "outputs/tables/n1_surrogate_and_placebo.csv", "outputs/tables/target_sensitivity.csv"])
display(pd.read_csv(TAB / "causal_spatial_magnitude.csv").round(2))
show(FIG / "10_model_skill_uncertainty.png", FIG / "12_spatial_negative_control.png", FIG / "11_surrogate_test_lake_albert.png")

## 6a · Longer record from open external data (~35 min the first time)

Landsat yearly water maps 1984–2021 (JRC Global Surface Water; only the South Sudan window is read from the server), plus the lake levels and climate indices from step 1.

<details><summary>About this code</summary>

```
Research question 12.1 — Step 5: open external data (no registration needed) that extends the record before 2003.

  1. Lake Victoria & Kyoga water levels, 1992-2026 (NASA GSFC G-REALM, 10-day altimetry)
     https://earth.gsfc.nasa.gov/gwm/lake/Index  -> lake.10.tar.gz, files lake000314 (Victoria), lake000398 (Kyoga)
     Downloaded to raw_data/external/grealm/.
  2. Surface water per year, 1984-2021, 30 m Landsat (JRC Global Surface Water, YearlyClassification v1.4)
     https://jeodpp.jrc.ec.europa.eu/ftp/jrc-opendata/GSWE/YearlyClassification/LATEST/tiles/
     Classes 2 = seasonal water, 3 = permanent water. In the downloaded tiles observed dry land is 0, so the
     share of unobserved pixels is not available; only water area (2 + 3) is used.
     Read remotely (windowed) and aggregated to 0.01° cell counts; only the South Sudan window is transferred.
  3. Climate indices (NOAA PSL): Indian Ocean Dipole (DMI, HadISST) and ENSO (Nino 3.4 anomaly).

Writes extreme_flood_early_warning/data/external_*.parquet.
```
</details>

In [ ]:
%%module external_data
import io
import urllib.request
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd
import rasterio
from rasterio.windows import Window
from shapely import contains_xy

HERE = Path(__file__).resolve().parent
BASE = HERE.parent
OUT = HERE / "data"
GREALM = BASE / "external" / "grealm"
JRC = "https://jeodpp.jrc.ec.europa.eu/ftp/jrc-opendata/GSWE/YearlyClassification/LATEST/tiles"

# Same regions as robustness.py, capped at 10 N (edge of the JRC tile row used here)
REGIONS = {"sudd": (6.0, 10.0, 29.5, 32.5), "west": (4.0, 10.0, 24.0, 29.5), "east": (5.5, 10.0, 32.5, 36.0)}
AGG = 40                                   # 40 x 0.00025° = 0.01° output cells
LON0, LON1, LAT0, LAT1 = 24.0, 36.0, 4.0, 10.0


# --------------------------------------------------------------------------------------
def grealm(lake_file):
    rows = []
    with open(lake_file, encoding="latin-1") as f:
        for line in f:
            p = line.split()
            if len(p) >= 6 and p[2].isdigit() and len(p[2]) == 8:
                h = float(p[5])
                if abs(h) < 900:                          # 999.99 = missing
                    rows.append((pd.Timestamp(p[2]), h))
    s = pd.DataFrame(rows, columns=["date", "h"]).groupby("date")["h"].median()
    return s


def lakes():
    out = pd.DataFrame({"victoria": grealm(GREALM / "lake000314.10d.2.txt"),
                        "kyoga": grealm(GREALM / "lake000398.10d.2.txt")})
    out.index.name = "date"
    return out


# --------------------------------------------------------------------------------------
def jrc_year(year):
    """Class counts per 0.01° cell over lon 24-36, lat 4-10 for one year: array (600, 1200, 4)."""
    p = OUT / "_jrc" / f"{year}.npy"
    if p.exists():
        return year, np.load(p)
    ny, nx = int((LAT1 - LAT0) * 100), int((LON1 - LON0) * 100)
    res = np.zeros((ny, nx, 4), np.uint16)
    for col, (tlon0, lo, hi) in {"0000800000": (20.0, 24.0, 30.0), "0000840000": (30.0, 30.0, 36.0)}.items():
        sep = "_" if year == 2021 else "-"           # the 2021 tiles use a different file-name separator
        url = f"/vsicurl/{JRC}/yearlyClassification{year}/yearlyClassification{year}{sep}0000280000-{col}.tif"
        c0, c1 = int((lo - tlon0) / 0.00025), int((hi - tlon0) / 0.00025)
        rows = int((10.0 - LAT0) / 0.00025)
        with rasterio.Env(GDAL_DISABLE_READDIR_ON_OPEN="EMPTY_DIR", GDAL_HTTP_MULTIRANGE="YES",
                          GDAL_HTTP_MAX_RETRY="5", GDAL_HTTP_RETRY_DELAY="3"), rasterio.open(url) as r:
            for r0 in range(0, rows, 4000):
                a = r.read(1, window=Window(c0, r0, c1 - c0, 4000))
                a = a.reshape(4000 // AGG, AGG, (c1 - c0) // AGG, AGG)
                ox = int((lo - LON0) * 100)
                # output rows are south -> north
                oy1 = ny - r0 // AGG
                for k in range(4):
                    cnt = (a == k).sum(axis=(1, 3)).astype(np.uint16)[::-1]
                    res[oy1 - cnt.shape[0]:oy1, ox:ox + cnt.shape[1], k] = cnt
    (OUT / "_jrc").mkdir(exist_ok=True)
    np.save(p, res)
    print(f"  JRC {year} done", flush=True)
    return year, res


def jrc():
    ssd = gpd.read_file(BASE / "Administrative boundaries" / "ssd_admin0.geojson").geometry.union_all()
    xc = LON0 + (np.arange(int((LON1 - LON0) * 100)) + 0.5) / 100
    yc = LAT0 + (np.arange(int((LAT1 - LAT0) * 100)) + 0.5) / 100
    X, Y = np.meshgrid(xc, yc)
    inside = contains_xy(ssd, X, Y)
    with ThreadPoolExecutor(4) as ex:
        data = dict(ex.map(jrc_year, range(1984, 2022)))
    rows = []
    px_km2 = (0.00025 * 111.32) ** 2 * np.cos(np.deg2rad(Y))       # km² per 30 m pixel, by latitude
    for year, a in sorted(data.items()):
        for reg, (la0, la1, lo0, lo1) in REGIONS.items():
            m = inside & (Y >= la0) & (Y < la1) & (X >= lo0) & (X < lo1)
            # In the downloaded yearly tiles, observed dry land is stored as 0 (checked on Lake Turkana and its
            # desert shore), so unobserved pixels cannot be separated; classes 2 and 3 (water) are unambiguous.
            area = lambda k: float((a[..., k] * px_km2)[m].sum())
            rows.append(dict(year=year, region=reg, seasonal_km2=area(2), permanent_km2=area(3),
                             water_km2=area(2) + area(3)))
    return pd.DataFrame(rows)


# --------------------------------------------------------------------------------------
def noaa(url, name):
    """Read a NOAA PSL monthly table; uses the copy in raw_data/external/noaa/ if present (the server is slow)."""
    local = BASE / "external" / "noaa" / url.rsplit("/", 1)[1]
    txt = local.read_text() if local.exists() else urllib.request.urlopen(url, timeout=120).read().decode()
    rows = []
    for line in txt.splitlines():
        p = line.split()
        if len(p) == 13 and p[0].isdigit():
            for m, v in enumerate(p[1:], 1):
                v = float(v)
                if v > -99:
                    rows.append((pd.Timestamp(int(p[0]), m, 1), v))
    return pd.Series(dict(rows), name=name)


def main():
    lk = lakes()
    lk.to_parquet(OUT / "external_lakes_grealm.parquet")
    print("Lakes:", lk.index.min().date(), "to", lk.index.max().date(), len(lk), "obs")

    idx = pd.concat([noaa("https://psl.noaa.gov/gcos_wgsp/Timeseries/Data/dmi.had.long.data", "iod_dmi"),
                     noaa("https://psl.noaa.gov/data/correlation/nina34.anom.data", "nino34")], axis=1)
    idx.index.name = "month"
    idx.to_parquet(OUT / "external_climate_indices.parquet")
    print("Climate indices:", idx.index.min().date(), "to", idx.index.max().date())

    j = jrc()
    j.to_parquet(OUT / "external_jrc_surface_water.parquet")
    print(j.pivot(index="year", columns="region", values=["water_km2", "permanent_km2"]).round(0).to_string())

In [ ]:
run_step(external_data.main, ["data/external_lakes_grealm.parquet", "data/external_climate_indices.parquet",
                              "data/external_jrc_surface_water.parquet"])

## 6b · Indian Ocean → lakes → Sudd



<details><summary>About this code</summary>

```
Research question 12.1 — Step 6: does the longer, open external record add independent events?

Chain tested: Indian Ocean Dipole (IOD) -> Lake Victoria / Kyoga level -> Sudd flooding.
Inputs: data/external_*.parquet (external_data.py) and the MODIS-based series of the main analysis.
Outputs: outputs/tables/extended_*.csv, outputs/figures/14_iod_lakes_sudd_chain.png
```
</details>

In [ ]:
%%module extended_analysis
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import train_models as T
import robustness as R

C = T.C
D = T.HERE / "data"


def load():
    lk = pd.read_parquet(D / "external_lakes_grealm.parquet").resample("MS").median().interpolate(limit=3)
    lk = lk - lk["1993":"2018"].mean()
    ci = pd.read_parquet(D / "external_climate_indices.parquet")
    son = ci["iod_dmi"][ci.index.month.isin([9, 10, 11])]
    son = son.groupby(son.index.year).mean()
    jrc = pd.read_parquet(D / "external_jrc_surface_water.parquet").pivot(index="year", columns="region",
                                                                         values="water_km2")
    jrc[jrc.sum(axis=1) < 300] = np.nan          # years without Landsat coverage over South Sudan
    w, f, _ = T.build_weekly()
    albert = f["albert_anom"].ffill().resample("MS").mean()
    rf = R.regional_flood()
    modis = (rf.resample("W-SUN").max() * T.PIXEL_KM2).resample("YE").mean()
    modis.index = modis.index.year
    modis = modis.loc[2003:2025, [f"unusual_px_{r}" for r in ["sudd", "west", "east"]]]
    modis.columns = ["sudd", "west", "east"]
    return lk, son, jrc, albert, modis


def iod_to_lakes(lk, son):
    """For each year, IOD (Sep-Nov) vs the change in annual mean lake level from that year to the next."""
    ann = lk.groupby(lk.index.year).mean()
    rows = []
    for y in range(1993, 2025):
        if y in son.index and y + 1 in ann.index and y in ann.index:
            rows.append(dict(year=y, iod_son=son[y], victoria_rise_next_year=ann.loc[y + 1, "victoria"] - ann.loc[y, "victoria"],
                             kyoga_rise_next_year=ann.loc[y + 1, "kyoga"] - ann.loc[y, "kyoga"]))
    t = pd.DataFrame(rows)
    strong = t[t.iod_son >= 0.75]
    rest = t[t.iod_son < 0.75]
    from scipy.stats import spearmanr
    summ = dict(n_years=len(t), n_strong_iod=len(strong),
                strong_iod_years=", ".join(str(y) for y in strong.year),
                mean_victoria_rise_after_strong_iod=strong.victoria_rise_next_year.mean(),
                max_victoria_rise_after_other_years=rest.victoria_rise_next_year.max(),
                spearman_iod_vs_victoria_rise=spearmanr(t.iod_son, t.victoria_rise_next_year).statistic,
                spearman_iod_vs_kyoga_rise=spearmanr(t.iod_son, t.kyoga_rise_next_year).statistic)
    return t, summ


def episodes(lk, jrc, modis, albert):
    """High-lake episodes and what the Sudd did, with both flood records where available."""
    vic = lk["victoria"].groupby(lk.index.year).mean()
    kyo = lk["kyoga"].groupby(lk.index.year).mean()
    alb = albert.groupby(albert.index.year).mean()
    jrc_base = jrc.loc[2002:2018].mean()
    mod_base = modis.loc[2003:2019].mean()
    rows = []
    for name, yrs in [("1998–2001 (after 1997 IOD)", range(1998, 2002)),
                      ("2003–2005 (Albert moderately high)", range(2003, 2006)),
                      ("2020–2021 (after 2019 IOD)", range(2020, 2022)),
                      ("2022–2025 (lakes stay high)", range(2022, 2026))]:
        yrs = list(yrs)
        g = lambda s: s.reindex(yrs).mean()
        rows.append({
            "episode": name,
            "Victoria anomaly (m)": g(vic), "Kyoga anomaly (m)": g(kyo), "Albert anomaly (m)": g(alb),
            "Sudd water, Landsat (× 2002–18)": g(jrc["sudd"]) / jrc_base["sudd"],
            "West water, Landsat (× 2002–18)": g(jrc["west"]) / jrc_base["west"],
            "East water, Landsat (× 2002–18)": g(jrc["east"]) / jrc_base["east"],
            "Sudd unusual flood, MODIS (× 2003–19)": g(modis["sudd"]) / mod_base["sudd"],
            "West unusual flood, MODIS (× 2003–19)": g(modis["west"]) / mod_base["west"],
            "East unusual flood, MODIS (× 2003–19)": g(modis["east"]) / mod_base["east"],
        })
    return pd.DataFrame(rows)


def figure(lk, son, jrc, albert, modis):
    fig, axes = plt.subplots(4, 1, figsize=(12, 10.5), sharex=True,
                             gridspec_kw=dict(height_ratios=[1, 1.4, 1.2, 1.2]))
    x0, x1 = pd.Timestamp("1984-01-01"), pd.Timestamp("2026-12-31")
    ax = axes[0]
    yrs = son.loc[1984:2025]
    ax.bar([pd.Timestamp(f"{y}-10-15") for y in yrs.index], yrs.values, width=250,
           color=[C["red"] if v >= 0.75 else "#c3c2b7" for v in yrs.values])
    for y in yrs[yrs >= 0.75].index:
        ax.text(pd.Timestamp(f"{y}-10-15"), yrs[y] + 0.08, str(y), ha="center", fontsize=9, color=C["ink"])
    ax.axhline(0, color=C["ink2"], lw=0.8)
    ax.set_ylim(-1.1, 1.35)
    ax.set_ylabel("°C")
    ax.set_title("1  Indian Ocean Dipole, Sep–Nov (red = strong positive, ≥ 0.75 °C)")
    ax = axes[1]
    for col, c, lab in [("victoria", C["blue"], "Lake Victoria"), ("kyoga", C["aqua"], "Lake Kyoga")]:
        ax.plot(lk.index, lk[col], color=c, lw=1.6, label=lab)
    ax.plot(albert.index, albert.values, color=C["violet"], lw=1.6, label="Lake Albert (from 2002)")
    ax.axhline(0, color=C["ink2"], lw=0.8)
    ax.set_ylabel("m vs. normal")
    ax.set_title("2  Upstream lakes, water-level anomaly (satellite altimetry)")
    ax.legend(loc="upper left", fontsize=8.5, ncol=3)
    ax = axes[2]
    for col, c, ls, lab in [("sudd", C["ink"], "-", "Sudd"), ("west", C["muted"], "-", "West"),
                            ("east", C["muted"], (0, (4, 2)), "East")]:
        s = jrc[col] / jrc.loc[2002:2018, col].mean()
        ax.plot([pd.Timestamp(f"{y}-07-01") for y in s.index], s.values, color=c, ls=ls, marker="o", ms=3.5,
                lw=2.2 if col == "sudd" else 1.5, label=lab)
    for a, b in [(1988, 1993), (1996, 1998)]:
        ax.axvspan(pd.Timestamp(f"{a}-01-01"), pd.Timestamp(f"{b}-12-31"), color="#ebe9e2", lw=0)
        ax.text(pd.Timestamp(f"{a}-01-01") + (pd.Timestamp(f"{b}-12-31") - pd.Timestamp(f"{a}-01-01")) / 2, 0.12,
                "no Landsat\ncoverage", ha="center", va="bottom", fontsize=8.5, color=C["ink2"])
    ax.axhline(1, color=C["ink2"], lw=0.8)
    ax.set_ylim(0, 2.8)
    ax.set_ylabel("× 2002–18 mean")
    ax.set_title("3a  Open-water area per year, Landsat (JRC Global Surface Water, 1984–2021)")
    ax.legend(loc="upper left", fontsize=8.5, ncol=3)
    ax = axes[3]
    for col, c, ls, lab in [("sudd", C["ink"], "-", "Sudd"), ("west", C["muted"], "-", "West"),
                            ("east", C["muted"], (0, (4, 2)), "East")]:
        s = modis[col] / modis.loc[2003:2019, col].mean()
        ax.plot([pd.Timestamp(f"{y}-07-01") for y in s.index], s.values, color=c, ls=ls, marker="o", ms=3.5,
                lw=2.2 if col == "sudd" else 1.5, label=lab)
    ax.axhline(1, color=C["ink2"], lw=0.8)
    ax.set_yscale("log")
    ax.set_ylabel("× 2003–19 mean (log)")
    ax.set_title("3b  Unusual flood area per year, MODIS/VIIRS flood masks (2003–2025)")
    ax.legend(loc="upper left", fontsize=8.5, ncol=3)
    ax.set_xlim(x0, x1)
    fig.tight_layout()
    fig.savefig(T.OUT_F / "14_iod_lakes_sudd_chain.png", dpi=170)
    plt.close(fig)


def main():
    lk, son, jrc, albert, modis = load()
    t, summ = iod_to_lakes(lk, son)
    t.to_csv(T.OUT_T / "extended_iod_vs_lake_rise.csv", index=False)
    print("IOD -> lake rise next year:", {k: (round(v, 2) if isinstance(v, float) else v) for k, v in summ.items()})
    ep = episodes(lk, jrc, modis, albert)
    ep.to_csv(T.OUT_T / "extended_high_lake_episodes.csv", index=False)
    print(ep.set_index("episode").T.round(2).to_string())
    figure(lk, son, jrc, albert, modis)

In [ ]:
run_step(extended_analysis.main, ["outputs/tables/extended_high_lake_episodes.csv", "outputs/figures/14_iod_lakes_sudd_chain.png"])
display(pd.read_csv(TAB / "extended_high_lake_episodes.csv").set_index("episode").T.round(2))
show(FIG / "14_iod_lakes_sudd_chain.png")

## 7 · Documented flood events (GLIDE, Dartmouth Flood Observatory)



<details><summary>About this code</summary>

```
Research question 12.1 — Step 8: documented flood events (independent of the satellite flood masks).

Sources (open, no registration, via the Humanitarian Data Exchange):
  - GLIDE disaster numbers, South Sudan 2011-2025 and Sudan (pre-2011 southern events)
    https://data.humdata.org/dataset/ssd-glide-events , https://data.humdata.org/dataset/sdn-glide-events
  - Dartmouth Flood Observatory master list (1985-2019)
    https://data.humdata.org/dataset/global-active-archive-of-large-flood-events-dfo
Files are in raw_data/external/events/.

Each event in what is now South Sudan is classified by the locations named in the source text:
  corridor = White Nile / Sudd corridor, downstream of the equatorial lakes (Jonglei, Bor, Unity, White Nile)
  west     = Bahr el Ghazal / Abyei, rain-fed rivers not fed by the lakes
  mixed    = names places in both
Then: was the upstream lake level higher in years with corridor floods than in years with western floods?
```
</details>

In [ ]:
%%module event_catalog
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import mannwhitneyu

import train_models as T

C = T.C
D = T.HERE / "data"

# year, month, source id, region class, the place names in the source text that decide the class, impact note
EVENTS = [
    (1996, 10, "DFO 2577", "mixed", "Bor (Jonglei, White Nile) and Tonj (Warrap)", "8,000 displaced, 102 dead"),
    (1998, 8, "DFO 2268 / FL-1998-000251-SDN", "corridor", "rivers listed incl. Jonglei and White Nile", "national event, 900,000 affected (all Sudan)"),
    (2008, 8, "DFO 3351 / FL-2008-000125-SDN", "west", "Aweil, Kuom river, Northern Bahr el Ghazal", "39,000 homeless"),
    (2008, 10, "DFO 3390 / FL-2008-000207-SDN", "unknown", "'Southern Sudan' only", "31 dead"),
    (2011, 9, "FL-2011-000142-SSD", "west", "Agok, Abyei", "flash floods, displaced population"),
    (2012, 6, "FL-2012-000128-SSD", "west", "Abyei", "seasonal rains, roads flooded"),
    (2014, 8, "FL-2014-000123-SSD", "corridor", "Bentiu (Unity)", "displacement sites flooded"),
    (2017, 9, "FL-2017-000137-SSD", "west", "Aweil North and Aweil West", "119,000 affected"),
    (2019, 6, "FL-2019-000062-SSD", "west", "Nyamlel, Aweil, Gokmachar (Northern Bahr el Ghazal)", "houses and roads destroyed"),
    (2020, 8, "FL-2020-000193-SSD", "corridor", "areas along the White Nile; Jonglei and Lakes worst", "600,000 affected"),
    (2021, 7, "FL-2021-000108-SSD", "corridor", "rising waters; GLIDE centroid 8.1 N 31.0 E (Sudd)", "90,000 affected (early season)"),
    (2024, 7, "FL-2024-000145-SSD", "mixed", "Aweil West, Jur River (west) and Panyijar (Unity)", "3,180 displaced"),
    (2025, 7, "FL-2025-000158-SSD", "corridor", "Mayendit (Unity) and Bor South (Jonglei)", "80,000 affected"),
]


def main():
    ev = pd.DataFrame(EVENTS, columns=["year", "month", "source", "region", "evidence", "impact"])
    lk = pd.read_parquet(D / "external_lakes_grealm.parquet").resample("MS").median().interpolate(limit=3)
    lk = lk - lk["1993":"2018"].mean()
    w, f, _ = T.build_weekly()
    albert = f["albert_anom"].ffill().resample("MS").mean()
    # lake level in the 6 months up to the event month (what an early-warning system would have seen)
    def before(s, y, m):
        end = pd.Timestamp(y, m, 1)
        v = s[(s.index > end - pd.DateOffset(months=6)) & (s.index <= end)]
        return v.mean() if v.notna().any() else np.nan
    ev["victoria_anom_6m_before"] = [before(lk["victoria"], y, m) for y, m in zip(ev.year, ev.month)]
    ev["kyoga_anom_6m_before"] = [before(lk["kyoga"], y, m) for y, m in zip(ev.year, ev.month)]
    ev["albert_anom_6m_before"] = [before(albert, y, m) for y, m in zip(ev.year, ev.month)]
    ev.to_csv(T.OUT_T / "event_catalog_south_sudan.csv", index=False)
    pd.set_option("display.width", 220)
    print(ev.drop(columns=["evidence", "impact"]).round(2).to_string(index=False))

    cor = ev[ev.region == "corridor"]
    wst = ev[ev.region == "west"]
    rows = []
    for col in ["victoria_anom_6m_before", "kyoga_anom_6m_before"]:
        a, b = cor[col].dropna(), wst[col].dropna()
        p = mannwhitneyu(a, b, alternative="greater").pvalue
        rows.append(dict(lake=col, corridor_n=len(a), corridor_mean=a.mean(), west_n=len(b), west_mean=b.mean(),
                         one_sided_p=p))
    res = pd.DataFrame(rows)
    res.to_csv(T.OUT_T / "event_catalog_lake_test.csv", index=False)
    print("\nLake level before corridor floods vs western floods (Mann-Whitney, one-sided):\n", res.round(3).to_string(index=False))

    # Without the 2020s (the single long regime), does the pattern still hold?
    pre = ev[ev.year < 2020]
    a = pre[pre.region == "corridor"]["victoria_anom_6m_before"].dropna()
    b = pre[pre.region == "west"]["victoria_anom_6m_before"].dropna()
    print(f"\nBefore 2020 only: corridor events n={len(a)} mean Victoria {a.mean():+.2f} m "
          f"({', '.join(f'{v:+.2f}' for v in a)}); west n={len(b)} mean {b.mean():+.2f} m")

    # 2020, 2021 and 2025 fall in one continuous high-lake period: count them as ONE independent event
    cor_ind = list(cor[cor.year < 2020]["victoria_anom_6m_before"]) + [cor[cor.year >= 2020]["victoria_anom_6m_before"].mean()]
    p_ind = mannwhitneyu(cor_ind, b, alternative="greater", method="exact").pvalue
    print(f"Independent events only (2020s counted once): corridor {[round(v, 2) for v in cor_ind]} vs west "
          f"{[round(v, 2) for v in b]} -> one-sided exact p = {p_ind:.3f}")
    pd.DataFrame([dict(test="2020s regime counted as one event", corridor_values=cor_ind, west_values=list(b),
                       one_sided_exact_p=p_ind)]).to_csv(T.OUT_T / "event_catalog_lake_test_independent.csv", index=False)

    # Figure: lake levels with documented events
    fig, ax = plt.subplots(figsize=(12, 4.6))
    ax.plot(lk.index, lk["victoria"], color=C["blue"], lw=1.6, label="Lake Victoria")
    ax.plot(lk.index, lk["kyoga"], color=C["aqua"], lw=1.4, label="Lake Kyoga")
    ax.axhline(0, color=C["ink2"], lw=0.8)
    style = {"corridor": (C["red"], "o", "Flood event, White Nile / Sudd corridor"),
             "west": (C["ink2"], "s", "Flood event, west (Bahr el Ghazal / Abyei)"),
             "mixed": (C["yellow"], "D", "Flood event, mixed / unclear"),
             "unknown": (C["yellow"], "D", None)}
    done, labelled = set(), set()
    for _, r in ev.iterrows():
        c, mk, lab = style[r.region]
        t = pd.Timestamp(r.year, r.month, 15)
        lab = lab if (lab and lab not in done) else None
        done.add(lab)
        row = -1.55 if r.year not in labelled else -1.25          # second event in the same year: stack above
        ax.plot(t, row, marker=mk, ms=10, color=c, mec="white", mew=1.2, ls="none", label=lab, zorder=5)
        if r.year not in labelled:
            ax.text(t, -1.75, str(r.year), ha="center", va="top", fontsize=8, color=C["ink2"], rotation=90)
            labelled.add(r.year)
    ax.set_ylim(-2.15, 2.9)
    ax.set_ylabel("m vs. 1993–2018 mean")
    ax.set_title("Documented flood events in South Sudan (GLIDE, Dartmouth Flood Observatory) and upstream lake levels",
                 fontsize=11)
    ax.legend(loc="upper left", fontsize=8.5, ncol=2)
    fig.tight_layout()
    fig.savefig(T.OUT_F / "17_documented_flood_events_and_lakes.png", dpi=170)
    plt.close(fig)

In [ ]:
run_step(event_catalog.main, ["outputs/tables/event_catalog_south_sudan.csv", "outputs/tables/event_catalog_lake_test_independent.csv"])
display(pd.read_csv(TAB / "event_catalog_south_sudan.csv").round(2))
show(FIG / "17_documented_flood_events_and_lakes.png")

## 8 · One model per question (~1 min)



<details><summary>About this code</summary>

```
Research question 12.1 — Step 7: one model per sub-question.

Q1  Which hydro-climatic factors are associated with the transition from normal to extreme flooding?
    -> Interpretable logistic regression on flood-season weeks (Jul-Dec) only, so it contrasts extreme flood
       seasons with normal ones rather than wet vs dry season. Standardised drivers -> odds ratio per +1 SD,
       95 % CI from a year-block bootstrap. Fitted on 2003-2019 (seasonal regime) and on 2003-2025.

Q2  Can upstream water-level anomalies provide an early indication of these events?
    -> Out-of-sample (leave-one-year-out) skill at 4-52 weeks ahead of
         local-only  vs  upstream-only  vs  local + upstream, and
         persistence vs  persistence + upstream (does upstream add anything beyond "it is flooded now"?)
       with year-block bootstrap CIs on the skill and on the differences.

Upstream = Lake Albert (DAHITI), Lake Victoria & Kyoga (NASA G-REALM), lake-basin runoff, Sobat rain,
           Indian Ocean Dipole (NOAA). All features use only information available at week t.
```
</details>

In [ ]:
%%module final_model
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

import train_models as T
import robustness as R

C = T.C
D = T.HERE / "data"
LEADS = [4, 13, 26, 39, 52]


# --------------------------------------------------------------------------------------
# Features
# --------------------------------------------------------------------------------------
def features():
    w, f, thr = T.build_weekly()
    ref = (w.index.year >= T.REF[0]) & (w.index.year <= T.REF[1])
    lk = pd.read_parquet(D / "external_lakes_grealm.parquet")
    lk = lk.resample("W-SUN").median().reindex(w.index).ffill(limit=4)       # causal: last observation
    for c in ["victoria", "kyoga"]:
        f[f"{c}_anom"] = lk[c] - lk[c][ref].mean()
    ci = pd.read_parquet(D / "external_climate_indices.parquet")["iod_dmi"]
    # a month's DMI is published after the month ends -> shift by one month before use
    iod3 = ci.rolling(3).mean().shift(1)
    iod_w = iod3.reindex(pd.date_range(iod3.index.min(), w.index.max(), freq="D")).ffill().reindex(w.index)
    f["iod_3m"] = iod_w
    f["iod_max12m"] = iod_w.rolling(52, min_periods=26).max()               # strong IOD in the past year
    return w, f


# One variable per physical pathway, so odds ratios are interpretable (all VIF < 3). Dropped as near-duplicates:
# rain − evaporation 26 wk (r = 0.99 with rain 26 wk) and Lake Victoria (r = 0.77 with Albert; its water reaches the
# Sudd through Lake Albert). Both stay in the Q2 early-warning models, where collinearity does not matter.
Q1_DRIVERS = {
    "Lake Albert level anomaly": "albert_anom",
    "Indian Ocean Dipole, max past 12 months": "iod_max12m",
    "Lake-basin runoff, 52 weeks": "ro_lakes_a52",
    "Sobat / Ethiopian highland rain, 13 weeks": "tp_sobat_a13",
    "South Sudan rain, 26 weeks": "tp_ssd_a26",
    "South Sudan rain, 4 weeks": "tp_ssd_a4",
}
LOCAL = ["tp_ssd_a4", "tp_ssd_a13", "tp_ssd_a26", "ro_ssd_a13", "tp_sudd_a13", "tp_ghazal_a13", "wb_ssd_a26",
         "et_ssd_a4"]
UPSTREAM = ["albert_anom", "albert_d26", "victoria_anom", "kyoga_anom", "ro_lakes_a52", "tp_lakes_a52",
            "tp_sobat_a13", "ro_sobat_a13", "iod_3m", "iod_max12m"]
STATE = ["log_unusual", "log_recurring", "unusual_share"]
SEASON = ["woy_sin", "woy_cos"]


# --------------------------------------------------------------------------------------
# Q1 — odds ratios
# --------------------------------------------------------------------------------------
def q1(w, f):
    cols = list(Q1_DRIVERS.values())
    rows = []
    for label, (a, b) in {"2003–2019 (seasonal regime)": (2003, 2019), "2003–2025 (all years)": (2003, 2025)}.items():
        m = np.asarray((w.index.year >= a) & (w.index.year <= b) & (w.index.month >= 7))
        X = f.loc[m, cols]
        ok = X.notna().all(axis=1).values
        X, y, idx = X.values[ok], w["extreme"].values[m][ok], w.index[m][ok]
        mu, sd = X.mean(0), X.std(0)
        Z = (X - mu) / sd

        def fit(i):
            return LogisticRegression(C=1.0, max_iter=5000).fit(Z[i], y[i]).coef_[0]

        est = fit(np.arange(len(y)))
        blocks = R.year_blocks(idx)
        keys = list(blocks)
        boots = []
        for _ in range(R.B):
            i = np.concatenate([blocks[k] for k in R.RNG.choice(keys, len(keys))])
            if len(np.unique(y[i])) == 2:
                boots.append(fit(i))
        boots = np.array(boots)
        lo, hi = np.percentile(boots, [2.5, 97.5], axis=0)
        for j, (name, col) in enumerate(Q1_DRIVERS.items()):
            rows.append(dict(period=label, driver=name, odds_ratio_per_sd=np.exp(est[j]),
                             ci_low=np.exp(lo[j]), ci_high=np.exp(hi[j]),
                             significant=bool(lo[j] > 0 or hi[j] < 0),
                             n_weeks=len(y), n_extreme=int(y.sum())))
    out = pd.DataFrame(rows)
    out.to_csv(T.OUT_T / "final_q1_odds_ratios.csv", index=False)
    return out


# --------------------------------------------------------------------------------------
# Q2 — early-warning skill
# --------------------------------------------------------------------------------------
Q2_MODELS = {
    "Local only": LOCAL + SEASON,
    "Upstream only": UPSTREAM + SEASON,
    "Local + upstream": LOCAL + UPSTREAM + SEASON,
    "Persistence": STATE + SEASON,
    "Persistence + upstream": STATE + UPSTREAM + SEASON,
}


def q2(w, f):
    dates = w.index
    m = (dates >= T.START) & (dates <= T.END)
    rows, diffs = [], []
    for h in LEADS:
        y = w["extreme"].shift(-h).values
        valid = m & ~np.isnan(y) & (np.arange(len(w)) >= T.LOOKBACK)
        woy = dates.isocalendar().week.clip(upper=52).values
        P = {}
        clim = np.full(len(w), np.nan)
        for mname, cols in Q2_MODELS.items():
            X = f[cols].fillna(0).values
            oof = np.full(len(w), np.nan)
            for _, tr, te in T.make_folds(dates, valid):
                sc = StandardScaler().fit(X[tr])
                mdl = LogisticRegression(C=0.1, max_iter=5000).fit(sc.transform(X[tr]), y[tr])
                oof[te] = mdl.predict_proba(sc.transform(X[te]))[:, 1]
                if mname == "Local only":   # climatology of the target week, from the same training folds
                    tab = pd.Series(y[tr]).groupby(np.roll(woy, -h)[tr]).mean()
                    clim[te] = pd.Series(np.roll(woy, -h)[te]).map(tab).fillna(y[tr].mean()).values
            P[mname] = oof
        ok = valid & ~np.isnan(clim)
        yy, idx = y[ok].astype(int), dates[ok]
        Pk = {k: v[ok] for k, v in P.items()}
        ck = clim[ok]

        def bss(p, i):
            return 1 - np.mean((p[i] - yy[i]) ** 2) / np.mean((ck[i] - yy[i]) ** 2)

        for mname, p in Pk.items():
            for metric, fn in [("AUC", lambda i, p=p: R.fast_auc(yy[i], p[i])), ("BSS", lambda i, p=p: bss(p, i))]:
                est, lo, hi = R.boot_ci(fn, idx)
                rows.append(dict(lead_weeks=h, model=mname, metric=metric, est=est, ci_low=lo, ci_high=hi))
        for dname, (a, b) in {"Upstream only − Local only": ("Upstream only", "Local only"),
                              "Local + upstream − Local only": ("Local + upstream", "Local only"),
                              "Persistence + upstream − Persistence": ("Persistence + upstream", "Persistence")}.items():
            est, lo, hi = R.boot_ci(lambda i: bss(Pk[a], i) - bss(Pk[b], i), idx)
            diffs.append(dict(lead_weeks=h, comparison=dname, bss_difference=est, ci_low=lo, ci_high=hi,
                              significant=bool(lo > 0 or hi < 0)))
        print(f"  lead {h} weeks done", flush=True)
    sk, df = pd.DataFrame(rows), pd.DataFrame(diffs)
    sk.to_csv(T.OUT_T / "final_q2_skill_by_lead.csv", index=False)
    df.to_csv(T.OUT_T / "final_q2_upstream_added_value.csv", index=False)
    return sk, df


def q2_forward(w, f):
    """Same models, trained on 2003-2018 only, evaluated on 2019-2025 (could the regime shift be anticipated?)."""
    dates = w.index
    m = (dates >= T.START) & (dates <= T.END)
    rows = []
    for h in LEADS:
        y = w["extreme"].shift(-h).values
        valid = m & ~np.isnan(y) & (np.arange(len(w)) >= T.LOOKBACK)
        tr = valid & ((dates + pd.Timedelta(weeks=h)) <= pd.Timestamp("2018-12-31"))
        te = valid & (dates >= pd.Timestamp("2019-01-01"))
        for mname, cols in Q2_MODELS.items():
            X = f[cols].fillna(0).values
            sc = StandardScaler().fit(X[tr])
            p = LogisticRegression(C=0.1, max_iter=5000).fit(sc.transform(X[tr]), y[tr]).predict_proba(
                sc.transform(X[te]))[:, 1]
            rows.append(dict(lead_weeks=h, model=mname, auc=R.fast_auc(y[te].astype(int), p),
                             hit_rate=float(np.mean(p[y[te] == 1] > 0.5))))
    out = pd.DataFrame(rows)
    out.to_csv(T.OUT_T / "final_q2_forward_test.csv", index=False)
    return out


# --------------------------------------------------------------------------------------
# Figures
# --------------------------------------------------------------------------------------
def fig_q1(od):
    fig, axes = plt.subplots(1, 2, figsize=(13, 5.2), sharey=True)
    names = list(Q1_DRIVERS)
    for ax, (period, s) in zip(axes, od.groupby("period", sort=False)):
        s = s.set_index("driver").loc[names]
        for j, (name, r) in enumerate(s.iterrows()):
            col = C["blue"] if r["significant"] and r["odds_ratio_per_sd"] > 1 else (
                C["red"] if r["significant"] else C["muted"])
            ax.errorbar(r["odds_ratio_per_sd"], j, xerr=[[r["odds_ratio_per_sd"] - r["ci_low"]],
                        [r["ci_high"] - r["odds_ratio_per_sd"]]], fmt="o", ms=7, color=col, elinewidth=2)
        ax.axvline(1, color=C["ink2"], lw=1, ls="--")
        ax.set_xscale("log")
        ax.set_yticks(range(len(names)), names)
        ax.invert_yaxis()
        ax.set_xlabel("Odds ratio per +1 SD (log scale, 95 % CI)")
        ax.set_title(f"{period}\n{int(s['n_extreme'].iloc[0])} extreme of {int(s['n_weeks'].iloc[0])} flood-season weeks",
                     fontsize=11)
        ax.grid(axis="y", visible=False)
    fig.suptitle("Q1 · Which factors distinguish an extreme flood season from a normal one?\n"
                 "blue = raises the odds, red = lowers them, grey = not significant",
                 x=0.01, ha="left", fontweight="bold", fontsize=12)
    fig.tight_layout()
    fig.savefig(T.OUT_F / "15_q1_factor_odds_ratios.png", dpi=170)
    plt.close(fig)


def fig_q2(sk, df):
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
    ax = axes[0]
    for mname, c, ls in [("Local only", C["orange"], "-"), ("Upstream only", C["violet"], "-"),
                         ("Local + upstream", C["blue"], "-"), ("Persistence", C["ink2"], ":")]:
        s = sk[(sk.model == mname) & (sk.metric == "AUC")].sort_values("lead_weeks")
        ax.fill_between(s.lead_weeks, s.ci_low, s.ci_high, color=c, alpha=0.12, lw=0)
        ax.plot(s.lead_weeks, s.est, color=c, ls=ls, marker="o", ms=5, label=mname)
    ax.axhline(0.5, color="#c3c2b7", lw=1)
    ax.set_xticks(LEADS)
    ax.set_xlabel("Warning lead time (weeks)")
    ax.set_ylabel("ROC-AUC (95 % CI band)")
    ax.set_title("How well can each source of information warn?")
    ax.legend(fontsize=9, loc="lower left")
    ax = axes[1]
    for j, (dname, c) in enumerate([("Upstream only − Local only", C["violet"]),
                                    ("Persistence + upstream − Persistence", C["ink"])]):
        s = df[df.comparison == dname].sort_values("lead_weeks")
        x = np.arange(len(LEADS)) + (j - 0.5) * 0.22
        ax.errorbar(x, s.bss_difference, yerr=[s.bss_difference - s.ci_low, s.ci_high - s.bss_difference],
                    fmt="o", ms=6, color=c, elinewidth=2, label=dname)
    ax.axhline(0, color=C["ink2"], lw=1, ls="--")
    ax.set_xticks(range(len(LEADS)), [f"{h} wk" for h in LEADS])
    ax.set_ylabel("Difference in Brier skill score (95 % CI)")
    ax.set_title("Does upstream information add skill? (CI above 0 = yes)")
    ax.legend(fontsize=9, loc="upper left")
    fig.suptitle("Q2 · Can upstream anomalies give early indication? (leave-one-year-out, 2003–2025)",
                 x=0.01, ha="left", fontweight="bold", fontsize=12)
    fig.tight_layout()
    fig.savefig(T.OUT_F / "16_q2_upstream_early_warning.png", dpi=170)
    plt.close(fig)


def main():
    w, f = features()
    pd.set_option("display.width", 200)
    print("\n=== Q1: odds ratios (flood-season weeks) ===")
    od = q1(w, f)
    print(od.round(2).to_string(index=False))
    fig_q1(od)
    print("\n=== Q2: early-warning skill ===")
    sk, df = q2(w, f)
    print(sk.pivot_table(index=["metric", "model"], columns="lead_weeks", values="est").round(2).to_string())
    print(df.round(3).to_string(index=False))
    fig_q2(sk, df)
    print("\n=== Q2: forward test (train <=2018, test 2019-2025) ===")
    print(q2_forward(w, f).pivot_table(index="model", columns="lead_weeks", values=["auc", "hit_rate"]).round(2).to_string())

In [ ]:
run_step(final_model.main, ["outputs/tables/final_q1_odds_ratios.csv", "outputs/tables/final_q2_skill_by_lead.csv",
                            "outputs/tables/final_q2_upstream_added_value.csv", "outputs/tables/final_q2_forward_test.csv"])
display(pd.read_csv(TAB / "final_q1_odds_ratios.csv").round(2))
display(pd.read_csv(TAB / "final_q2_upstream_added_value.csv").round(3))
show(FIG / "15_q1_factor_odds_ratios.png", FIG / "16_q2_upstream_early_warning.png")

## 9a · Animation (needs ffmpeg) and gridded flood maps



<details><summary>About this code</summary>

```
Research question 12.1 — animation: South Sudan's floods and the Lake Albert level, monthly 2003-2025.

  Map (left):    flood-mask detections per month on a 0.04° grid (~4.4 km): recurring (normal seasonal) in blue,
                 unusual in orange. The Sudd, West and East regions of robustness.py are outlined.
  Gauge (right): Lake Albert water-level anomaly vs. the 2003-19 mean, with the +1 m hindsight threshold.
  Strip (bottom): unusual flood extent per region (same series as robustness.py), drawn up to the current month.

Output: outputs/animation/south_sudan_floods_lake_albert_2003_2025.mp4 (1920x1080, H.264, PowerPoint-compatible)
        plus poster frames as PNG.
```
</details>

In [ ]:
%%module make_animation
from pathlib import Path

import geopandas as gpd
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.animation import FFMpegWriter
from matplotlib.patches import Patch
from shapely import contains_xy
from shapely.geometry import box

matplotlib.use("Agg")

import train_models as T
import robustness as R

BASE = T.HERE.parent
OUT = T.HERE / "outputs" / "animation"
OUT.mkdir(parents=True, exist_ok=True)

CELL = 0.04                                   # grid cell size in degrees
LON0, LON1, LAT0, LAT1 = 23.9, 36.1, 3.3, 12.4
NX, NY = int(round((LON1 - LON0) / CELL)), int(round((LAT1 - LAT0) / CELL))
MONTHS = pd.date_range("2003-01-01", "2025-12-01", freq="MS")
FPS, HOLD = 10, 40                            # 276 months at 10 fps + 4 s hold on the last frame

COL = dict(recurring="#2a78d6", unusual="#eb6834", lake="#4a3aa7", land="#f3f2ee", border="#9a988f",
           ink=T.C["ink"], ink2=T.C["ink2"], muted=T.C["muted"], grid=T.C["grid"], surface=T.C["surface"])
REGION_STYLE = {  # strip and outline encoding: emphasis on the Sudd, the region downstream of Lake Albert
    "sudd": dict(color=COL["ink"], lw=2.6, ls="-", label="Sudd / White Nile (downstream of Lake Albert)"),
    "west": dict(color="#8a8780", lw=1.8, ls="-", label="West: Bahr el Ghazal"),
    "east": dict(color="#8a8780", lw=1.8, ls=(0, (4, 2)), label="East: Sobat–Pibor"),
}
RGB = {k: matplotlib.colors.to_rgb(COL[k]) for k in ["recurring", "unusual"]}
TOWNS = {"Juba": (31.58, 4.85), "Bor": (31.56, 6.21), "Malakal": (31.66, 9.53), "Wau": (27.99, 7.70)}


def caption(t):
    if t.year <= 2019:
        return "2003–2019", "Floods pulse with the seasons.\nLake Albert stays below +1 m."
    if t.year == 2020:
        return "2020", "Lake Albert climbs to\n+3 m above normal."
    return "2021–2025", "The Sudd stays flooded, even though\nlocal rain is below normal."


# --------------------------------------------------------------------------------------
# Data
# --------------------------------------------------------------------------------------
PX_PER_CELL = (CELL / 0.0020833) ** 2      # flood-mask pixels per grid cell (~369)
FULL = 0.30                                 # cell share flooded that is drawn at full colour


def flood_grid():
    """(n_months, NY, NX) float16 arrays: share of each cell flooded, as the highest daily pixel count in the month
    (the same 'max of 3-day composites' logic as the weekly target). Area-faithful, unlike 'any pixel in cell'."""
    p = T.HERE / "data" / "_flood_grid_monthly_frac.npz"
    if p.exists():
        z = np.load(p)
        return z["recurring"].astype(np.float32), z["unusual"].astype(np.float32)
    grids = {}
    for typ in ["recurring", "unusual"]:
        g = np.zeros((len(MONTHS), NY, NX), np.float16)
        for year in range(2003, 2026):
            d = pd.concat([pd.read_parquet(BASE / "flood_masks" / f"compact_{typ}" / f"flood_events_{t}_{year}.parquet",
                                           columns=["date", "lat", "lon"]) for t in ["h20v08", "h21v08"]])
            d = d[(d.lon >= LON0) & (d.lon < LON1) & (d.lat >= LAT0) & (d.lat < LAT1)]
            dt = pd.to_datetime(d["date"])
            doy = dt.dt.dayofyear.values - 1
            ix = ((d["lon"].values - LON0) / CELL).astype(int)
            iy = ((d["lat"].values - LAT0) / CELL).astype(int)
            daily = np.zeros((366, NY, NX), np.uint16)
            np.add.at(daily, (doy, iy, ix), 1)
            days = pd.date_range(f"{year}-01-01", f"{year}-12-31", freq="D")
            for mth in range(1, 13):
                sel = np.where(days.month == mth)[0]
                g[(year - 2003) * 12 + mth - 1] = np.minimum(daily[sel].max(0) / PX_PER_CELL, 1)
            print(f"  gridding {typ} {year}")
        grids[typ] = g
    np.savez_compressed(p, **grids)
    return grids["recurring"].astype(np.float32), grids["unusual"].astype(np.float32)


def strength(frac):
    """Monotone colour strength: ~0.08 at 1 % flooded, ~0.44 at 10 %, 1 at >= 30 %."""
    return np.clip(frac / FULL, 0, 1) ** 0.75


def load():
    ssd = gpd.read_file(BASE / "Administrative boundaries" / "ssd_admin0.geojson")
    adm1 = gpd.read_file(BASE / "Administrative boundaries" / "ssd_admin1.geojson")
    geom = ssd.geometry.union_all()
    xc = LON0 + (np.arange(NX) + 0.5) * CELL
    yc = LAT0 + (np.arange(NY) + 0.5) * CELL
    inside = contains_xy(geom, *np.meshgrid(xc, yc))

    rec, unu = flood_grid()
    rec, unu = rec * inside, unu * inside

    w, f, _ = T.build_weekly()
    lake = f["albert_anom"].ffill().resample("MS").mean().reindex(MONTHS)

    rf = R.regional_flood()
    rf = (rf.resample("W-SUN").max() * T.PIXEL_KM2).resample("MS").mean().reindex(MONTHS).fillna(0)
    strip = pd.DataFrame({r: rf[f"unusual_px_{r}"] for r in REGION_STYLE})
    return ssd, adm1, geom, rec, unu, lake, strip


# --------------------------------------------------------------------------------------
# Figure
# --------------------------------------------------------------------------------------
def build_figure(ssd, adm1, geom, lake, strip):
    plt.rcParams.update({"font.family": "DejaVu Sans", "text.color": COL["ink"], "axes.labelcolor": COL["ink2"],
                         "xtick.color": COL["muted"], "ytick.color": COL["muted"]})
    fig = plt.figure(figsize=(19.2, 10.8), dpi=100, facecolor=COL["surface"])
    fig.text(0.035, 0.945, "South Sudan's floods and the Lake Albert level, 2003–2025",
             fontsize=27, fontweight="bold", color=COL["ink"])
    fig.text(0.035, 0.910, "Monthly satellite flood detections (NASA MODIS/VIIRS) · Lake Albert satellite altimetry (DAHITI)",
             fontsize=14, color=COL["ink2"])

    # ---- map ----
    axm = fig.add_axes([0.035, 0.30, 0.60, 0.58])
    axm.set_facecolor(COL["surface"])
    ssd.plot(ax=axm, color=COL["land"], edgecolor="none", zorder=0)
    adm1.boundary.plot(ax=axm, color="#d6d4cc", lw=0.6, zorder=1)
    rgba = np.zeros((NY, NX, 4))
    im = axm.imshow(rgba, extent=(LON0, LON1, LAT0, LAT1), origin="lower", interpolation="nearest", zorder=2)
    ssd.boundary.plot(ax=axm, color=COL["border"], lw=1.2, zorder=3)
    for r, (a, b, c, d) in R.FLOOD_REGIONS.items():
        shp = box(c, a, d, b).intersection(geom)
        st = REGION_STYLE[r]
        for poly in getattr(shp, "geoms", [shp]):
            x, y = poly.exterior.xy
            axm.plot(x, y, color=st["color"], lw=2.2 if r == "sudd" else 1.5, ls=st["ls"], zorder=4)
    axm.text(29.6, 10.62, "SUDD", fontsize=15, fontweight="bold", color=COL["ink"], zorder=6)
    axm.text(29.6, 10.40, "downstream of Lake Albert", fontsize=11.5, color=COL["ink"], zorder=6)
    axm.text(24.35, 10.75, "WEST", fontsize=13, fontweight="bold", color=COL["ink2"], zorder=6)
    axm.text(24.35, 10.52, "Bahr el Ghazal · not downstream", fontsize=10.5, color=COL["ink2"], zorder=6)
    axm.text(33.3, 10.95, "EAST", fontsize=13, fontweight="bold", color=COL["ink2"], zorder=6)
    axm.text(33.3, 10.72, "Sobat–Pibor · not downstream", fontsize=10.5, color=COL["ink2"], zorder=6)
    for name, (x, y) in TOWNS.items():
        left = name in ("Malakal", "Bor")       # keep labels off the region outlines
        axm.plot(x, y, "o", ms=5, mfc="white", mec=COL["ink"], mew=1.2, zorder=7)
        axm.text(x - 0.15 if left else x + 0.12, y, name, fontsize=10.5, color=COL["ink"], zorder=7,
                 va="center", ha="right" if left else "left",
                 bbox=dict(boxstyle="round,pad=0.15", fc=COL["surface"], ec="none", alpha=0.75))
    # The White Nile (Bahr el Jebel) enters South Sudan from Uganda at Nimule (3.6 N, 32.06 E)
    axm.annotate("White Nile from\nLake Albert (Uganda)", xy=(32.06, 3.75), xytext=(33.25, 3.55),
                 fontsize=11, color=COL["lake"], fontweight="bold", va="center", zorder=7,
                 arrowprops=dict(arrowstyle="-|>", color=COL["lake"], lw=2, mutation_scale=16))
    axm.set_xlim(LON0, LON1)
    axm.set_ylim(LAT0, LAT1)
    axm.set_aspect("equal")
    axm.axis("off")
    # Colour key: hue = flood type, strength = share of the 4-km cell flooded
    axk = fig.add_axes([0.105, 0.335, 0.12, 0.06])
    fr = np.linspace(0, 0.4, 200)
    rows = ["recurring", "unusual"]              # origin="lower": row 0 is drawn at the bottom
    key = np.zeros((2, len(fr), 4))
    for row, typ in enumerate(rows):
        key[row, :, :3] = RGB[typ]
        key[row, :, 3] = strength(fr)
    axk.imshow(key, aspect="auto", extent=(0, 40, 0, 2), origin="lower", interpolation="nearest")
    axk.set_yticks([0.5, 1.5], ["Normal seasonal", "Unusual"], fontsize=11.5)
    axk.set_xticks([0, 10, 20, 30, 40], ["0", "10", "20", "30", "40 %"], fontsize=10.5)
    axk.tick_params(length=0)
    for s in axk.spines.values():
        s.set_visible(False)
    axk.set_xlabel("share of each 4-km cell flooded", fontsize=10.5, color=COL["ink2"], labelpad=2)
    axk.set_title("Flooding", fontsize=12.5, fontweight="bold", loc="left", color=COL["ink"], pad=4)

    date_txt = fig.text(0.665, 0.815, "", fontsize=46, fontweight="bold", color=COL["ink"])
    phase_txt = fig.text(0.665, 0.765, "", fontsize=17, fontweight="bold", color=COL["ink2"])
    cap_txt = fig.text(0.665, 0.745, "", fontsize=17, color=COL["ink"], va="top", linespacing=1.35)

    # ---- gauge ----
    axg = fig.add_axes([0.715, 0.30, 0.055, 0.30])
    axg.set_facecolor("#ebe9e2")
    axg.set_xlim(0, 1)
    axg.set_ylim(-1, 4.0)
    bar = axg.bar(0.5, 0, width=1.0, bottom=0, color=COL["lake"], zorder=2)[0]
    axg.axhline(0, color=COL["ink2"], lw=1.2, zorder=3)
    axg.axhline(1, color=COL["ink"], lw=1.6, ls=(0, (5, 3)), zorder=3)
    axg.text(1.15, 1, "+1 m\nhindsight\nthreshold", fontsize=11, color=COL["ink"], va="center")
    axg.set_yticks([-1, 0, 1, 2, 3], ["−1 m", "0", "+1 m", "+2 m", "+3 m"], fontsize=12)
    axg.set_xticks([])
    for s in axg.spines.values():
        s.set_visible(False)
    axg.set_title("Lake Albert level\nvs. 2003–19 mean", fontsize=13.5, color=COL["ink"], loc="center", pad=12,
                  fontweight="bold")
    val_txt = axg.text(0.5, 0, "", ha="center", fontsize=15, fontweight="bold", color=COL["lake"], zorder=4)

    # ---- lake history, same y-scale as the gauge ----
    axh = fig.add_axes([0.835, 0.30, 0.135, 0.30], sharey=axg)
    axh.set_facecolor(COL["surface"])
    (lake_line,) = axh.plot([], [], color=COL["lake"], lw=2.2)
    axh.axhline(0, color=COL["ink2"], lw=1)
    axh.axhline(1, color=COL["ink"], lw=1.2, ls=(0, (5, 3)))
    axh.set_xlim(MONTHS[0], MONTHS[-1] + pd.offsets.MonthEnd(1))
    axh.xaxis.set_major_locator(matplotlib.dates.YearLocator(10, month=1, day=1))
    axh.xaxis.set_major_formatter(matplotlib.dates.DateFormatter("%Y"))
    axh.tick_params(labelleft=False, labelsize=11, length=0)
    axh.grid(axis="y", color=COL["grid"], lw=0.7)
    axh.grid(axis="x", visible=False)
    for s in ["top", "right", "left"]:
        axh.spines[s].set_visible(False)
    axh.spines["bottom"].set_color("#c3c2b7")
    axh.set_title("Lake Albert since 2003", fontsize=13.5, color=COL["ink"], loc="left", pad=12, fontweight="bold")

    # ---- strip ----
    axs = fig.add_axes([0.07, 0.065, 0.87, 0.17])
    axs.set_facecolor(COL["surface"])
    lines = {}
    for r, st in REGION_STYLE.items():
        (lines[r],) = axs.plot([], [], color=st["color"], lw=st["lw"], ls=st["ls"], label=st["label"])
    cursor = axs.axvline(MONTHS[0], color=COL["ink2"], lw=1.2)
    axs.set_xlim(MONTHS[0], MONTHS[-1] + pd.offsets.MonthEnd(1))
    axs.set_ylim(0, strip.values.max() * 1.08)
    axs.grid(axis="y", color=COL["grid"], lw=0.7)
    axs.grid(axis="x", visible=False)
    for s in ["top", "right"]:
        axs.spines[s].set_visible(False)
    axs.spines["left"].set_color("#c3c2b7")
    axs.spines["bottom"].set_color("#c3c2b7")
    axs.yaxis.set_major_formatter(matplotlib.ticker.FuncFormatter(lambda v, _: f"{v:,.0f}"))
    axs.tick_params(labelsize=11.5)
    axs.set_ylabel("km²", fontsize=12)
    axs.set_title("Unusual flood extent by region (monthly mean of weekly maxima)", fontsize=13.5, loc="left",
                  fontweight="bold", color=COL["ink"])
    axs.legend(loc="upper left", fontsize=11.5, frameon=False, ncol=3)

    return fig, dict(im=im, date=date_txt, phase=phase_txt, cap=cap_txt, bar=bar, val=val_txt,
                     lines=lines, cursor=cursor, lake=lake_line)


def update(i, A, rec, unu, lake, strip):
    k = min(i, len(MONTHS) - 1)
    t = MONTHS[k]
    # "over" compositing: unusual on top of recurring, each with area-faithful opacity
    a_r, a_u = strength(rec[k])[..., None], strength(unu[k])[..., None]
    c_r, c_u = np.array(RGB["recurring"]), np.array(RGB["unusual"])
    alpha = a_u + a_r * (1 - a_u)
    rgb = (c_u * a_u + c_r * a_r * (1 - a_u)) / np.where(alpha > 0, alpha, 1)
    A["im"].set_data(np.concatenate([rgb, alpha], axis=2))
    A["date"].set_text(t.strftime("%B %Y"))
    ph, cap = caption(t)
    A["phase"].set_text(ph)
    A["cap"].set_text(cap)
    v = lake.iloc[k]
    A["bar"].set_height(v)
    A["val"].set_text("0.0 m" if abs(v) < 0.05 else f"{v:+.1f} m".replace("-", "−"))
    A["val"].set_y(max(v, 0) + 0.08)
    A["lake"].set_data(MONTHS[:k + 1], lake.values[:k + 1])
    for r, ln in A["lines"].items():
        ln.set_data(MONTHS[:k + 1], strip[r].values[:k + 1])
    A["cursor"].set_xdata([t, t])


def main():
    ssd, adm1, geom, rec, unu, lake, strip = load()
    assert len(lake) == len(MONTHS) and lake.notna().all(), "Lake Albert series has gaps"
    fig, A = build_figure(ssd, adm1, geom, lake, strip)

    # Poster frames for slides and for checking
    for t in ["2010-09-01", "2019-09-01", "2020-12-01", "2023-09-01"]:
        update(MONTHS.get_loc(pd.Timestamp(t)), A, rec, unu, lake, strip)
        fig.savefig(OUT / f"frame_{t[:7]}.png", dpi=100, facecolor=COL["surface"])

    out = OUT / "south_sudan_floods_lake_albert_2003_2025.mp4"
    writer = FFMpegWriter(fps=FPS, codec="libx264", bitrate=-1,
                          extra_args=["-pix_fmt", "yuv420p", "-crf", "18", "-preset", "slow", "-movflags", "+faststart"])
    n = len(MONTHS) + HOLD
    with writer.saving(fig, str(out), dpi=100):
        for i in range(n):
            update(i, A, rec, unu, lake, strip)
            writer.grab_frame(facecolor=COL["surface"])
            if i % 50 == 0:
                print(f"  frame {i}/{n}")
    plt.close(fig)
    print("Saved", out)

In [ ]:
grid = "data/_flood_grid_monthly_frac.npz"
if MAKE_VIDEO and HAS_FFMPEG:
    run_step(make_animation.main, [grid, "outputs/animation/south_sudan_floods_lake_albert_2003_2025.mp4"])
else:   # the poster figures still need the gridded flood maps
    run_step(make_animation.flood_grid, [grid])

## 9b · SQ1 poster figure



<details><summary>About this code</summary>

```
Poster graphics for sub-question 1, classic science-poster style (white, Arial, boxed axes), as two separate
images with no running text (the text is written in Canva):
which factors turn South Sudan's normal yearly floods into extreme floods?

Needs the outputs of robustness.py, final_model.py and make_animation.py (flood grid cache).
```
</details>

In [ ]:
%%module poster_q1
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from shapely import contains_xy
from shapely.geometry import box

import train_models as T
import robustness as R

OUT = T.HERE / "outputs" / "poster"
OUT.mkdir(parents=True, exist_ok=True)
BASE = T.HERE.parent

# blues and greens
SUDD = "#08519c"        # Sudd / flooded land
WEST = "#31a354"
EAST = "#74c476"
LAKE = "#4292c6"
RIVER = "#2171b5"
LAND = "#f2f2f2"
GREY = "#555555"

CELL, LON0, LON1, LAT0, LAT1 = 0.04, 23.9, 36.1, 3.3, 12.4     # flood grid written by make_animation.py


def data():
    rf = R.regional_flood()
    yearly = (rf.resample("W-SUN").max() * T.PIXEL_KM2).resample("YE").mean()
    yearly.index = yearly.index.year
    yearly = yearly.loc[2003:2025]
    _, f, _ = T.build_weekly()
    albert = f["albert_anom"].ffill().resample("MS").mean()["2003":"2025"]
    od = pd.read_csv(T.OUT_T / "final_q1_odds_ratios.csv")
    od = od[od.period.str.startswith("2003–2025")].set_index("driver")
    mag = pd.read_csv(T.OUT_T / "causal_spatial_magnitude.csv")
    mag = mag[mag.extent == "unusual"].set_index("region")
    z = np.load(T.HERE / "data" / "_flood_grid_monthly_frac.npz")
    flood2023 = z["unusual"][(2023 - 2003) * 12:(2024 - 2003) * 12].astype(float).max(axis=0)
    return yearly, albert, od, mag, flood2023


def draw_map(ax, flood2023):
    ssd = gpd.read_file(BASE / "Administrative boundaries" / "ssd_admin0.geojson")
    geom = ssd.geometry.union_all()
    ssd.plot(ax=ax, color=LAND, edgecolor="black", lw=0.9, zorder=1)
    nx, ny = flood2023.shape[1], flood2023.shape[0]
    xc = LON0 + (np.arange(nx) + 0.5) * CELL
    yc = LAT0 + (np.arange(ny) + 0.5) * CELL
    inside = contains_xy(geom, *np.meshgrid(xc, yc))
    rgba = np.zeros((ny, nx, 4))
    rgba[..., :3] = plt.matplotlib.colors.to_rgb(SUDD)
    rgba[..., 3] = np.clip(flood2023 / 0.3, 0, 1) ** 0.75 * inside
    ax.imshow(rgba, extent=(LON0, LON1, LAT0, LAT1), origin="lower", interpolation="nearest", zorder=2)
    sudd = box(29.5, 6.0, 32.5, 10.0).intersection(geom)
    ax.plot(*sudd.exterior.xy, color="black", lw=1.4, ls="--", zorder=3)
    lakes = [
        [(31.8, -1.0), (31.7, -1.8), (32.0, -2.4), (32.6, -2.6), (33.2, -2.4), (33.9, -2.1), (34.5, -1.6),
         (34.8, -1.0), (34.4, -0.4), (34.0, 0.0), (33.4, 0.35), (32.8, 0.3), (32.3, 0.0), (31.8, -0.4)],
        [(30.25, 1.1), (30.6, 1.1), (31.1, 1.55), (31.55, 2.1), (31.45, 2.3), (31.0, 2.0), (30.5, 1.5)],
        [(32.4, 1.45), (32.9, 1.35), (33.4, 1.55), (33.3, 1.75), (32.8, 1.6), (32.5, 1.7)],
    ]
    for pts in lakes:
        ax.add_patch(plt.Polygon(pts, closed=True, fc=LAKE, ec="none", zorder=3))
    nile = np.array([(33.2, 0.42), (32.95, 1.3), (32.3, 2.15), (31.45, 2.25), (31.8, 3.0), (32.06, 3.6), (31.6, 4.85),
                     (31.56, 6.2), (31.0, 7.4), (30.5, 8.6), (30.45, 9.5), (31.66, 9.53), (32.0, 10.6), (32.4, 12.45)])
    ax.plot(nile[:, 0], nile[:, 1], color=RIVER, lw=2.2, zorder=4)
    for i in [4, 8]:
        ax.annotate("", xy=nile[i + 1], xytext=nile[i], zorder=5,
                    arrowprops=dict(arrowstyle="-|>", color=RIVER, lw=0, mutation_scale=18))
    for txt, (x, y), kw in [("Lake\nVictoria", (33.25, -1.25), dict(ha="center", color="white", fontweight="bold")), ("Lake Albert", (29.9, 1.6), dict(ha="right")),
                            ("Sudd", (29.55, 10.5), dict(fontweight="bold")),
                            ("SOUTH SUDAN", (25.6, 7.3), dict(color=GREY)), ("UGANDA", (30.4, 0.1), dict(color=GREY)),
                            ("Juba", (31.75, 4.85), dict(fontsize=11))]:
        ax.text(x, y, txt, **{"fontsize": 12.5, "zorder": 6, "va": "center", **kw})
    ax.plot(31.58, 4.85, "o", ms=4, color="black", zorder=6)
    ax.set_xlim(24.0, 36.6)
    ax.set_ylim(-3.1, 12.6)
    ax.set_aspect("equal")
    ax.set_xticks([25, 30, 35], ["25°E", "30°E", "35°E"])
    ax.set_yticks([0, 5, 10], ["0°", "5°N", "10°N"])
    ax.tick_params(labelsize=11)
    ax.legend(handles=[Patch(color=SUDD, label="Flooded land (normally dry), 2023"),
                       Patch(color=LAKE, label="Lakes"),
                       Line2D([], [], color=RIVER, lw=2.2, label="White Nile (flows north)"),
                       Line2D([], [], color="black", lw=1.4, ls="--", label="Sudd study area")],
              loc="lower left", fontsize=11, frameon=True, edgecolor="black", fancybox=False, framealpha=1)


def main():
    yearly, albert, od, mag, flood2023 = data()
    plt.rcParams.update(plt.rcParamsDefault)          # drop the project chart style: plain, boxed, white
    plt.rcParams.update({"font.family": "Arial", "font.size": 14, "axes.linewidth": 1.0, "axes.grid": False,
                         "xtick.direction": "out", "ytick.direction": "out", "axes.facecolor": "white"})

    # ---- part 1: map ----
    fig, axm = plt.subplots(figsize=(7.2, 8.6))
    draw_map(axm, flood2023)
    fig.tight_layout()
    save(fig, "q1_part1_map")

    # ---- part 2: flooding per region (top) and Lake Albert (bottom), shared years ----
    fig, (ax, ax2) = plt.subplots(2, 1, figsize=(9, 7.6), sharex=True, gridspec_kw=dict(height_ratios=[1.25, 1],
                                                                                        hspace=0.12))
    x = yearly.index.values
    ax.plot(x, yearly["unusual_px_sudd"], color=SUDD, lw=2.8, marker="o", ms=4.5, label="Sudd")
    ax.plot(x, yearly["unusual_px_west"], color=WEST, lw=1.8, marker="s", ms=4, label="West")
    ax.plot(x, yearly["unusual_px_east"], color=EAST, lw=1.8, marker="^", ms=4, ls="--", label="East")
    ax.set_ylabel("Flooded area (km²)")
    ax.set_ylim(0, None)
    ax.legend(loc="upper left", frameon=True, edgecolor="black", fancybox=False)
    xm = albert.index.year + (albert.index.month - 0.5) / 12
    ax2.plot(xm, albert.values, color=LAKE, lw=1.8)
    ax2.fill_between(xm, 0, albert.values, where=albert.values > 0, color=LAKE, alpha=0.25, lw=0)
    ax2.axhline(0, color="black", lw=0.8)
    ax2.set_ylabel("Lake Albert level\n(m above normal)")
    ax2.set_xlabel("Year")
    ax2.set_xlim(2002.5, 2025.5)
    for a in (ax, ax2):
        a.axvline(2019.5, color=GREY, lw=1, ls=":")
    fig.align_ylabels()
    fig.tight_layout()
    save(fig, "q1_part2_charts")
    combined(yearly, albert, flood2023)
    print("saved to", OUT)


def combined(yearly, albert, flood2023):
    """Map + both charts in one roughly square image for the poster's SQ1 column (A0), in the poster font."""
    import glob
    from matplotlib import font_manager as fm
    for f in glob.glob(str(BASE / "external" / "fonts" / "ss3" / "OTF" / "SourceSans3-*.otf")):
        fm.fontManager.addfont(f)
    plt.rcParams.update(plt.rcParamsDefault)
    plt.rcParams.update({"font.family": "Source Sans 3", "font.size": 18, "axes.linewidth": 1.1,
                         "axes.facecolor": "white", "xtick.labelsize": 16, "ytick.labelsize": 16})
    fig = plt.figure(figsize=(12, 12.4))
    axm = fig.add_axes([0.0, 0.22, 0.45, 0.78])
    draw_map(axm, flood2023)
    axm.get_legend().remove()
    for t in axm.texts:
        t.set_fontsize(t.get_fontsize() * 1.3)
    axm.tick_params(labelsize=15)
    fig.legend(handles=[Patch(color=SUDD, label="Flooded land (normally dry), 2023"),
                        Patch(color=LAKE, label="Lakes in Uganda"),
                        Line2D([], [], color=RIVER, lw=2.4, label="White Nile (flows north)"),
                        Line2D([], [], color="black", lw=1.4, ls="--", label="Sudd study area")],
               loc="lower left", bbox_to_anchor=(0.02, 0.0), fontsize=16.5, frameon=False, handlelength=1.6)
    ax = fig.add_axes([0.58, 0.56, 0.41, 0.42])
    ax2 = fig.add_axes([0.58, 0.08, 0.41, 0.40], sharex=ax)
    x = yearly.index.values
    ax.plot(x, yearly["unusual_px_sudd"], color=SUDD, lw=3.2, marker="o", ms=5.5, label="Sudd")
    ax.plot(x, yearly["unusual_px_west"], color=WEST, lw=2.2, marker="s", ms=5, label="West")
    ax.plot(x, yearly["unusual_px_east"], color=EAST, lw=2.2, marker="^", ms=5, ls="--", label="East")
    ax.set_ylabel("Flooded area (km²)")
    ax.set_ylim(0, None)
    ax.legend(loc="upper left", frameon=True, edgecolor="black", fancybox=False, fontsize=16.5)
    ax.tick_params(labelbottom=False)
    xm = albert.index.year + (albert.index.month - 0.5) / 12
    ax2.plot(xm, albert.values, color=LAKE, lw=2.2)
    ax2.fill_between(xm, 0, albert.values, where=albert.values > 0, color=LAKE, alpha=0.25, lw=0)
    ax2.axhline(0, color="black", lw=0.9)
    ax2.set_ylabel("Lake Albert level (m above normal)")
    ax2.set_xlabel("Year")
    ax2.set_xlim(2002.5, 2025.5)
    ax2.set_xticks([2005, 2010, 2015, 2020, 2025])
    for a in (ax, ax2):
        a.axvline(2019.5, color=GREY, lw=1.2, ls=":")
    save(fig, "sq1_combined")


def save(fig, name):
    fig.savefig(OUT / f"{name}.png", dpi=300, facecolor="white", bbox_inches="tight")
    fig.savefig(OUT / f"{name}_transparent.png", dpi=300, transparent=True, bbox_inches="tight")
    plt.close(fig)

In [ ]:
run_step(poster_q1.main, ["outputs/poster/sq1_combined.png"])
show(POSTER / "sq1_combined.png", width=700)

## 10 · SQ3: ADM3 priority map (~2 min)

Hazard (flooded share 2021–25) × exposure (people and cattle) × vulnerability (IPC 3+), combined with a geometric mean so an area is a top priority only when all three are high.

<details><summary>About this code</summary>

```
Sub-question 3: integrate flood hazard, exposure and vulnerability at ADM3 (payam) level to identify areas to
prioritise for anticipatory humanitarian action.

  Hazard        share of the payam flooded at the yearly peak, mean 2021-2025 (MODIS/VIIRS flood masks),
                log-scaled so that 30 % or more flooded scores 1
  Exposure      people (population_est, team's payam table) and cattle (ICPAC cattle raster)
  Vulnerability share of people in IPC Phase 3+ (county level, latest analysis)
  Priority      geometric mean of the three percentile scores (INFORM-style: high only when all three are high)

Outputs: outputs/tables/sq3_adm3_priority_index.csv, outputs/poster/sq3_priority_map.png (+ transparent)
Needs the flood grid cache from make_animation.py. Run from the raw_data folder with a Python that has rasterio:
  .venv/bin/python extreme_flood_early_warning/poster_sq3.py
```
</details>

In [ ]:
%%module poster_sq3
import glob
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
from matplotlib import font_manager as fm
from matplotlib.patches import Patch
from rasterio.mask import mask
from shapely.geometry import Point

HERE = Path(__file__).resolve().parent
BASE = HERE.parent
OUT_T = HERE / "outputs" / "tables"
OUT_P = HERE / "outputs" / "poster"
CELL, LON0, LAT0 = 0.04, 23.9, 3.3

for f in glob.glob(str(BASE / "external" / "fonts" / "ss3" / "OTF" / "SourceSans3-*.otf")):
    fm.fontManager.addfont(f)

NAVY = "#13294b"
CLASSES = [("Highest priority (top 10%)", NAVY), ("High (next 15%)", "#2a7f8f"),
           ("Moderate (next 25%)", "#9cc9cf"), ("Lower (bottom 50%)", "#eef2f4")]


def hazard(payams):
    z = np.load(HERE / "data" / "_flood_grid_monthly_frac.npz")
    tot = np.clip(z["unusual"].astype(float) + z["recurring"].astype(float), 0, 1)
    yrs = [tot[(y - 2003) * 12:(y - 2003 + 1) * 12].max(axis=0) for y in range(2021, 2026)]
    h = np.mean(yrs, axis=0)
    ny, nx = h.shape
    xc = LON0 + (np.arange(nx) + 0.5) * CELL
    yc = LAT0 + (np.arange(ny) + 0.5) * CELL
    X, Y = np.meshgrid(xc, yc)
    pts = gpd.GeoDataFrame({"h": h.ravel()}, geometry=gpd.points_from_xy(X.ravel(), Y.ravel()), crs="EPSG:4326")
    j = gpd.sjoin(pts, payams[["adm3_pcode", "geometry"]], predicate="within")
    out = j.groupby("adm3_pcode")["h"].mean()
    # payams smaller than a grid cell: nearest cell to the payam's representative point
    for _, r in payams[~payams.adm3_pcode.isin(out.index)].iterrows():
        p = r.geometry.representative_point()
        out[r.adm3_pcode] = h[int((p.y - LAT0) / CELL), int((p.x - LON0) / CELL)]
    return out


def cattle(payams):
    vals = {}
    with rasterio.open(BASE / "farmland" / "geonode__cattle_gha.tif") as src:
        for _, r in payams.iterrows():
            try:
                a, _ = mask(src, [r.geometry], crop=True, all_touched=True, filled=False)
                v = float(np.nansum(np.clip(a.compressed(), 0, None)))
            except ValueError:
                v = 0.0
            vals[r.adm3_pcode] = v
    return pd.Series(vals)


def build():
    payams = gpd.read_file(BASE / "flood_driver_analysis" / "data" / "payam_flood_predictions.geojson")
    payams = payams[["adm3_pcode", "adm3_name", "adm2_name", "adm1_name", "geometry"]]
    summ = pd.read_csv(BASE / "flood_driver_analysis" / "data" / "payam_flood_predictions_annual_summary.csv")
    ipc = pd.read_csv(BASE / "QA_results" / "county_ipc_flood_analysis.csv")[["adm2_name", "p3plus_pct_curr"]]
    df = payams.merge(summ[["adm3_pcode", "population_est"]], on="adm3_pcode").merge(ipc, on="adm2_name", how="left")
    df["hazard_flooded_share"] = df.adm3_pcode.map(hazard(payams))
    # the cattle raster is ~9 km; small payams that touch the same cell share it, so use density x area
    cat = cattle(payams)
    df["cattle"] = df.adm3_pcode.map(cat)
    pr = lambda s: s.rank(pct=True)
    # hazard on an absolute (log) scale: 0 % flooded -> 0, >= 30 % flooded -> 1. A rank would give payams that
    # barely flood a mid score, because most payams hardly flood at all.
    df["H"] = np.clip(np.log1p(df.hazard_flooded_share * 100) / np.log1p(30), 0, 1)
    df["E"] = (pr(df.population_est) + pr(df.cattle)) / 2
    df["E"] = pr(df.E)
    df["V"] = pr(df.p3plus_pct_curr)
    df["priority"] = (df.H * df.E * df.V) ** (1 / 3)
    q = df.priority.rank(pct=True, ascending=False)
    df["priority_class"] = np.select([q <= 0.10, q <= 0.25, q <= 0.50], [c[0] for c in CLASSES[:3]], CLASSES[3][0])
    df["rank"] = df.priority.rank(ascending=False).astype(int)
    df.drop(columns="geometry").sort_values("rank").to_csv(OUT_T / "sq3_adm3_priority_index.csv", index=False)
    return df


def figure(df):
    plt.rcParams.update(plt.rcParamsDefault)
    plt.rcParams.update({"font.family": "Source Sans 3", "font.size": 15})
    ssd = df.dissolve()
    adm1 = df.dissolve(by="adm1_name")
    fig = plt.figure(figsize=(14, 10))
    ax = fig.add_axes([0.0, 0.0, 0.66, 1.0])
    colors = dict(CLASSES)
    df.plot(ax=ax, color=df.priority_class.map(colors), edgecolor="white", lw=0.25)
    adm1.boundary.plot(ax=ax, color="#5b6b7d", lw=0.6)
    ssd.boundary.plot(ax=ax, color=NAVY, lw=1.2)
    # label the counties with the most highest-priority payams (one label per county)
    top = df[df.priority_class == CLASSES[0][0]]
    counties = top.dissolve(by="adm2_name")
    from matplotlib import patheffects as pe
    halo = [pe.withStroke(linewidth=3.5, foreground="white")]
    nudge = {"Rubkona": (-0.35, 0.15), "Fangak": (0.0, 0.25), "Ayod": (0.15, 0.0), "Uror": (0.25, 0.0),
             "Twic East": (0.0, -0.15), "Duk": (-0.35, 0.0)}
    for name, (dx, dy) in nudge.items():
        if name not in counties.index:
            continue
        p = counties.loc[name].geometry.representative_point()
        ax.text(p.x + dx, p.y + dy, name, fontsize=14.5, fontweight="bold", color=NAVY, ha="center", va="center",
                path_effects=halo, zorder=10)
    ax.legend(handles=[Patch(fc=c, ec="#9aa5b1", lw=0.6, label=l) for l, c in CLASSES], loc="lower left",
              fontsize=13.5, frameon=False, title="Priority for anticipatory action", title_fontsize=14.5,
              alignment="left")
    ax.set_axis_off()
    ax.set_aspect("equal")

    minis = [("H", "Hazard", "flooded share of area, 2021–25", "Blues"),
             ("E", "Exposure", "people and cattle", "Greens"),
             ("V", "Vulnerability", "people in food crisis (IPC 3+)", "Oranges")]
    for i, (col, title, sub, cmap) in enumerate(minis):
        a = fig.add_axes([0.70, 0.68 - i * 0.32, 0.28, 0.28])
        df.plot(ax=a, column=col, cmap=cmap, vmin=0, vmax=1, edgecolor="none")
        ssd.boundary.plot(ax=a, color="#5b6b7d", lw=0.6)
        a.set_axis_off()
        a.set_aspect("equal")
        a.text(0.0, 1.0, title, transform=a.transAxes, fontsize=15.5, fontweight="bold", color=NAVY, va="bottom")
        a.text(0.0, 0.93, sub, transform=a.transAxes, fontsize=12.5, color="#4a5563", va="bottom")
    for name, kw in [("sq3_priority_map", dict(facecolor="white")), ("sq3_priority_map_transparent", dict(transparent=True))]:
        fig.savefig(OUT_P / f"{name}.png", dpi=300, bbox_inches="tight", **kw)
    plt.close(fig)


def main():
    df = build()
    top10 = df[df.priority_class == CLASSES[0][0]]
    print(df.sort_values("rank")[["rank", "adm3_name", "adm2_name", "adm1_name", "hazard_flooded_share",
                                  "population_est", "cattle", "p3plus_pct_curr", "priority"]].head(15).round(3).to_string(index=False))
    print(f"\nHighest-priority payams: {len(top10)} of {len(df)}; people living there: "
          f"{top10.population_est.sum():,.0f} ({top10.population_est.sum() / df.population_est.sum():.0%} of the total)")
    print("by state:", top10.adm1_name.value_counts().to_dict())
    figure(df)
    print("saved", OUT_P / "sq3_priority_map.png")

In [ ]:
run_step(poster_sq3.main, ["outputs/tables/sq3_adm3_priority_index.csv", "outputs/poster/sq3_priority_map.png"])
display(pd.read_csv(TAB / "sq3_adm3_priority_index.csv").head(15).round(3))
show(POSTER / "sq3_priority_map.png")

## 11 · Answers (all numbers read from the result files above)

In [ ]:
od = pd.read_csv(TAB / "final_q1_odds_ratios.csv").set_index(["period", "driver"])
alb = od.loc[("2003–2025 (all years)", "Lake Albert level anomaly")]
ci = pd.read_csv(TAB / "ci_driver_auc.csv")
sob = ci[(ci.driver == "Sobat/Ethiopia rain, 13-wk") & (ci.horizon == 13) & (ci.period == "2003–2019")].iloc[0]
mag = pd.read_csv(TAB / "causal_spatial_magnitude.csv").query("extent == 'unusual'").set_index("region")
rain = pd.read_csv(TAB / "causal_local_rain_path.csv").query("period == '2020–2025' and driver == 'South Sudan rain, 26-wk'").iloc[0]
ev = pd.read_csv(TAB / "event_catalog_lake_test_independent.csv").iloc[0]
add = pd.read_csv(TAB / "final_q2_upstream_added_value.csv").query("comparison == 'Upstream only − Local only'").set_index("lead_weeks")
sk = pd.read_csv(TAB / "final_q2_skill_by_lead.csv").query("metric == 'AUC'").pivot(index="model", columns="lead_weeks", values="est")
fwd = pd.read_csv(TAB / "final_q2_forward_test.csv").pivot(index="model", columns="lead_weeks", values="auc")
ens = loyo[loyo.model.str.startswith("Ensemble")].set_index("horizon")
sq3 = pd.read_csv(TAB / "sq3_adm3_priority_index.csv")
top = sq3[sq3.priority_class.str.startswith("Highest")]
counties = top.adm2_name.value_counts().head(6).index.tolist()
f = lambda x: f"{x:.2f}"
display(Markdown(f"""
### SQ1a — Which factors?
- **Before 2020 (seasonal regime):** extremes follow Ethiopian-highland (Sobat) rain, AUC {f(sob.auc)} 13 weeks ahead
  (95% CI {f(sob.ci_low)}–{f(sob.ci_high)}).
- **Since 2020 (lake-driven regime):** each +1 SD of Lake Albert level → odds of an extreme flood season
  ×{alb.odds_ratio_per_sd:.1f} (95% CI {alb.ci_low:.1f}–{alb.ci_high:.1f}).
- **Only downstream of the lakes:** unusual flooding 2021–25 vs 2003–19 grew ×{mag.loc['Sudd / White Nile corridor','ratio_2021_25_vs_2003_19']:.1f}
  in the Sudd, ×{mag.loc['West: Bahr el Ghazal (negative control)','ratio_2021_25_vs_2003_19']:.1f} in the west and
  ×{mag.loc['East: Sobat–Pibor','ratio_2021_25_vs_2003_19']:.1f} in the east.
- **Not local rain:** local 26-week rain during 2020s extremes {rain.mean_anomaly_in_extreme_weeks:+.1f}σ (95% CI {rain.ci_low:+.1f} to {rain.ci_high:+.1f}).
- **Documented events:** corridor floods came after higher Lake Victoria levels than western floods (exact p = {ev.one_sided_exact_p:.3f}, 2020s counted once).

### SQ1b — Can upstream anomalies warn early?
- Upstream-only AUC {f(sk.loc['Upstream only', 13])} at 13 weeks and {f(sk.loc['Upstream only', 52])} at 52 weeks, vs local-only
  {f(sk.loc['Local only', 13])} / {f(sk.loc['Local only', 52])}; upstream better with CI above 0 at 13 weeks
  ({add.loc[13,'bss_difference']:+.2f}, {add.loc[13,'ci_low']:+.2f} to {add.loc[13,'ci_high']:+.2f}) and 52 weeks.
- Best forecasting model (ensemble, leave-one-year-out): AUC {f(ens.loc[2,'auc'])} at 2 weeks, {f(ens.loc[26,'auc'])} at 26 weeks.
- **Limit:** trained on 2003–2018 only, upstream-only AUC on 2019–2025 is {f(fwd.loc['Upstream only', 13])} at 13 weeks —
  the regime shift could not be learned from earlier data. Plausible early indicator, not yet a proven predictor.

### SQ3 — Where to act first?
- **{len(top)} of {len(sq3)} payams** are highest priority, home to **{top.population_est.sum()/1e6:.1f} million people**
  ({top.population_est.sum()/sq3.population_est.sum():.0%} of the population).
- Mostly in: {", ".join(counties)}.
"""))

## 12 · Check that every result was produced

In [ ]:
expected = {
    "weekly dataset": "data/daily_series.parquet",
    "EDA inventory": "outputs/tables/eda_data_inventory.csv",
    "EDA cleaning decisions": "outputs/tables/eda_cleaning_decisions.csv",
    "LOYO skill": "outputs/tables/skill_leave_one_year_out.csv",
    "forward test": "outputs/tables/skill_forward_test_2019_2025.csv",
    "physical model": "outputs/tables/skill_forward_test_physical_vs_others.csv",
    "skill CIs": "outputs/tables/ci_model_skill.csv",
    "spatial control": "outputs/tables/causal_spatial_magnitude.csv",
    "surrogate test": "outputs/tables/n1_surrogate_and_placebo.csv",
    "target sensitivity": "outputs/tables/target_sensitivity.csv",
    "IOD → lakes": "outputs/tables/extended_iod_vs_lake_rise.csv",
    "event catalogue": "outputs/tables/event_catalog_south_sudan.csv",
    "Q1 odds ratios": "outputs/tables/final_q1_odds_ratios.csv",
    "Q2 early warning": "outputs/tables/final_q2_skill_by_lead.csv",
    "SQ3 priority index": "outputs/tables/sq3_adm3_priority_index.csv",
    "GRU model": "outputs/models/gru_all_features.pt",
    "SQ1 poster figure": "outputs/poster/sq1_combined.png",
    "SQ3 poster map": "outputs/poster/sq3_priority_map.png",
}
for n in ["eda_01", "eda_02", "eda_03", "eda_04", "eda_05"]:
    expected[f"figure {n}"] = f"outputs/figures/{n}_" + next((p.name[len(n) + 1:] for p in FIG.glob(f"{n}_*.png")), "missing")
for i in [1, 2, 3, 6, 9, 10, 12, 14, 15, 16, 17]:
    expected[f"figure {i:02d}"] = next((str(p.relative_to(PROJ)) for p in FIG.glob(f"{i:02d}*.png")), f"outputs/figures/{i:02d}_missing")
if MAKE_VIDEO and HAS_FFMPEG:
    expected["animation"] = "outputs/animation/south_sudan_floods_lake_albert_2003_2025.mp4"
rows = [(k, v, (PROJ / v).exists() and (PROJ / v).stat().st_size > 0) for k, v in expected.items()]
res = pd.DataFrame(rows, columns=["result", "file", "ok"])
res["ok"] = res.ok.map({True: "✓", False: "✗"})
display(res)
# sanity checks on the headline numbers
checks = {
    "ensemble AUC at 2 weeks > 0.9": ens.loc[2, "auc"] > 0.9,
    "Sudd grew more than west and east": mag["ratio_2021_25_vs_2003_19"].idxmax() == "Sudd / White Nile corridor",
    "Lake Albert odds ratio CI above 1": alb.ci_low > 1,
    "SQ3 top class is ~10% of payams": abs(len(top) / len(sq3) - 0.10) < 0.02,
}
for k, v in checks.items():
    print("✓" if v else "✗", k)
assert (res.ok == "✓").all() and all(checks.values()), "some results are missing or out of range"
print("\nAll results produced and checked ✓")